# Object Localization

Object Localization is an important transition from **image classification** to **object detection**.

In image classification, the neural network only needs to answer:

> **What is in the image?**

In object localization, the network needs to answer:

> **What is the object, and where is it located?**

At a high level:

$$
\text{Object Localization}
=
\text{Classification}
+
\text{Bounding-Box Regression}
$$

Object Localization can also be understood from the perspective of **Multi-Task Learning (MTL)**. Several tasks share a common CNN representation and then branch into task-specific heads. During backpropagation, gradient contributions from the different tasks are accumulated when they flow back into the shared network.

---

## 1. From Image Classification to Object Localization

In image classification, the model predicts only the class:

$$
x \rightarrow \hat{y}
$$

For example:

```text
Image
  ↓
CNN
  ↓
Cat
```

The CNN learns a mapping from the input image to a class:

$$
x \rightarrow \text{class}
$$

It does not need to know where the cat is located.

In localization, the model must provide both semantic and geometric information:

```text
Image
  ↓
CNN
  ↓
Class + Bounding Box
```

Therefore:

$$
\text{Object Localization}
=
\text{What?}
+
\text{Where?}
$$

This makes localization a natural intermediate step between classification and object detection.

---

## 2. Bounding Box

To represent the location of an object, we use a **bounding box**.

```text
┌──────────────────────────────────────┐
│                                      │
│        ┌────────────────────┐        │
│        │                    │        │
│        │        CAR         │        │
│        │                    │        │
│        └────────────────────┘        │
│                                      │
└──────────────────────────────────────┘
```

Instead of predicting the four edges directly, we represent the bounding box using four parameters:

$$
b_x,\quad b_y,\quad b_h,\quad b_w
$$

where:

- $b_x$: x-coordinate of the bounding-box center;
- $b_y$: y-coordinate of the bounding-box center;
- $b_h$: bounding-box height;
- $b_w$: bounding-box width.

If the coordinates are normalized by the image dimensions, they are typically expressed relative to $[0,1]$.

The four boundaries can be recovered as:

$$
x_{\min} = b_x - \frac{b_w}{2}
$$

$$
x_{\max} = b_x + \frac{b_w}{2}
$$

$$
y_{\min} = b_y - \frac{b_h}{2}
$$

$$
y_{\max} = b_y + \frac{b_h}{2}
$$

The important idea is:

> The neural network does not need to explicitly draw the bounding box. It only needs to learn four continuous values that parameterize the box.

This is the **regression** part of the localization problem.

---

## 3. Output Representation

Suppose the problem contains three classes:

- pedestrian;
- car;
- motorcycle.

The network output can be represented as:

$$
\hat{y}
=
\begin{bmatrix}
\hat{p}_c \\
\hat{b}_x \\
\hat{b}_y \\
\hat{b}_h \\
\hat{b}_w \\
\hat{c}_1 \\
\hat{c}_2 \\
\hat{c}_3
\end{bmatrix}
$$

The ground-truth vector is:

$$
y
=
\begin{bmatrix}
p_c \\
b_x \\
b_y \\
b_h \\
b_w \\
c_1 \\
c_2 \\
c_3
\end{bmatrix}
$$

There are:

$$
1 + 4 + 3 = 8
$$

output values.

### 3.1 Objectness

$p_c$ answers:

> **Is there an object?**

### 3.2 Bounding-Box Parameters

$b_x$ and $b_y$ describe the object center:

> **Where is the object?**

$b_h$ and $b_w$ describe the object size:

> **How large is the object?**

### 3.3 Class Prediction

$c_1,c_2,c_3$ describe the object class:

> **Which class does the object belong to?**

Therefore, the output contains three types of information:

$$
\text{Object Existence}
+
\text{Object Geometry}
+
\text{Object Semantics}
$$

---

## 4. Object Localization as Multi-Task Learning

A useful way to understand the architecture is through **Multi-Task Learning**.

The CNN first learns a shared representation:

$$
h = f(x;\theta_s)
$$

where:

- $x$: input image;
- $\theta_s$: shared parameters;
- $h$: shared visual representation.

The shared representation is then sent to task-specific heads.

```mermaid
flowchart TD
    A["Input Image x"] --> B["Shared CNN θs"]
    B --> C["Shared Representation h"]

    C --> D["Classification / Objectness Head θcls"]
    C --> E["Localization Head θloc"]

    D --> F["p̂c, ĉ1, ĉ2, ..."]
    E --> G["b̂x, b̂y, b̂h, b̂w"]

    F --> H["Lcls"]
    G --> I["Lloc"]

    H --> J["Ltotal"]
    I --> J
```

Mathematically:

$$
\hat{y}_{\text{cls}}
=
g_{\text{cls}}(h;\theta_{\text{cls}})
$$

and:

$$
\hat{y}_{\text{loc}}
=
g_{\text{loc}}(h;\theta_{\text{loc}})
$$

where:

- $\theta_s$: shared parameters;
- $\theta_{\text{cls}}$: classification-specific parameters;
- $\theta_{\text{loc}}$: localization-specific parameters.

The important structure is:

$$
\text{Shared Representation}
\rightarrow
\text{Task-Specific Heads}
$$

---

## 5. Why Can the Tasks Share a Representation?

CNNs learn a hierarchy of visual features:

$$
\text{Pixels}
\rightarrow
\text{Edges}
\rightarrow
\text{Textures}
\rightarrow
\text{Shapes}
\rightarrow
\text{Parts}
\rightarrow
\text{Semantic Features}
$$

These features can support multiple tasks.

Classification asks:

> Which object does this visual representation correspond to?

Localization asks:

> Where does this object-related representation occur?

Therefore, the network does not need two completely separate CNNs.

Instead:

$$
\text{Input}
\rightarrow
\text{Shared CNN}
\rightarrow
\text{Shared Representation}
$$

Then the representation branches into different tasks:

```mermaid
flowchart LR
    A["Shared Representation h"] --> B["Classification / Objectness"]
    A --> C["Localization"]
```

This is the core idea of shared representation learning in Multi-Task Learning.

---

## 6. Loss Function

The network predicts:

$$
\hat{y}
=
\begin{bmatrix}
\hat{p}_c \\
\hat{b}_x \\
\hat{b}_y \\
\hat{b}_h \\
\hat{b}_w \\
\hat{c}_1 \\
\hat{c}_2 \\
\hat{c}_3
\end{bmatrix}
$$

while the ground truth is:

$$
y
=
\begin{bmatrix}
p_c \\
b_x \\
b_y \\
b_h \\
b_w \\
c_1 \\
c_2 \\
c_3
\end{bmatrix}
$$

We therefore need a loss:

$$
L(\hat{y},y)
$$

In the lecture, Andrew Ng uses squared error to illustrate the basic formulation.

A simple conceptual formulation is:

$$
\begin{aligned}
L
={}&
(\hat{p}_c-p_c)^2
+
(\hat{b}_x-b_x)^2
+
(\hat{b}_y-b_y)^2
\\
&+
(\hat{b}_h-b_h)^2
+
(\hat{b}_w-b_w)^2
+
\sum_i(\hat{c}_i-c_i)^2
\end{aligned}
$$

This formulation shows that different prediction components contribute to the total training objective.

In practical implementations, the classification component can use a suitable classification loss depending on the exact output formulation. The key idea here is the decomposition of the objective into different task-related components.

---

## 7. Decomposing the Loss

Instead of viewing the long equation as one expression, it is useful to decompose it conceptually:

$$
L
=
L_{\text{objectness}}
+
L_{\text{localization}}
+
L_{\text{classification}}
$$

### 7.1 Objectness Loss

Measures whether the model correctly predicts:

$$
\hat{p}_c
\quad\text{vs.}\quad
p_c
$$

### 7.2 Localization Loss

Measures the error in:

$$
(\hat{b}_x,\hat{b}_y,\hat{b}_h,\hat{b}_w)
$$

relative to:

$$
(b_x,b_y,b_h,b_w)
$$

### 7.3 Classification Loss

Measures the error in:

$$
(\hat{c}_1,\hat{c}_2,\ldots)
$$

relative to the ground-truth class.

Thus, one prediction is evaluated along three dimensions:

$$
\text{Objectness}
+
\text{Geometry}
+
\text{Semantics}
$$

---

## 8. Why Must the Loss Depend on $p_c$?

Suppose:

$$
p_c = 0
$$

This means:

> There is no object.

In that case, asking for the location of the object does not make sense.

For example, it would be meaningless to penalize:

$$
(\hat{b}_x-b_x)^2
$$

because there is no ground-truth object whose position should be predicted.

The same applies to:

$$
b_y,\quad b_h,\quad b_w
$$

The object-specific classification is also irrelevant when no object exists.

Therefore, localization and classification losses should only contribute when an object is actually present.

---

## 9. Masking the Loss with $p_c$

We can express this conceptually as:

$$
L
=
L_{\text{objectness}}
+
\mathbf{1}_{\{p_c=1\}}
L_{\text{localization}}
+
\mathbf{1}_{\{p_c=1\}}
L_{\text{classification}}
$$

where:

$$
\mathbf{1}_{\{p_c=1\}}
=
\begin{cases}
1 & \text{if an object exists} \\
0 & \text{if no object exists}
\end{cases}
$$

If an object exists:

$$
p_c = 1
$$

then:

$$
L
=
L_{\text{objectness}}
+
L_{\text{localization}}
+
L_{\text{classification}}
$$

If there is no object:

$$
p_c = 0
$$

then:

$$
L
=
L_{\text{objectness}}
$$

The key semantic idea is:

> **If there is no object, the model should learn "there is no object", not "where the object is".**

---

## 10. Forward Propagation

Let the shared CNN produce:

$$
h = f(x;\theta_s)
$$

The task-specific heads then produce:

$$
\hat{y}_{\text{cls}}
=
g_{\text{cls}}(h;\theta_{\text{cls}})
$$

and:

$$
\hat{y}_{\text{loc}}
=
g_{\text{loc}}(h;\theta_{\text{loc}})
$$

The task losses are:

$$
L_{\text{cls}}
=
L_{\text{cls}}
\left(
\hat{y}_{\text{cls}},
y_{\text{cls}}
\right)
$$

and:

$$
L_{\text{loc}}
=
L_{\text{loc}}
\left(
\hat{y}_{\text{loc}},
y_{\text{loc}}
\right)
$$

Finally:

$$
L_{\text{total}}
=
\lambda_{\text{cls}}L_{\text{cls}}
+
\lambda_{\text{loc}}L_{\text{loc}}
$$

The complete forward flow is:

```mermaid
flowchart LR
    A["Input Image x"] --> B["Shared CNN θs"]
    B --> C["Shared Representation h"]

    C --> D["Classification / Objectness"]
    C --> E["Localization"]

    D --> F["p̂c, ĉ1, ĉ2, ..."]
    E --> G["b̂x, b̂y, b̂h, b̂w"]

    F --> H["Lcls"]
    G --> I["Lloc"]

    H --> J["Ltotal"]
    I --> J
```

In compact form:

$$
x
\rightarrow
h
\rightarrow
\left\{
\hat{y}_{\text{cls}},
\hat{y}_{\text{loc}}
\right\}
\rightarrow
\left\{
L_{\text{cls}},
L_{\text{loc}}
\right\}
\rightarrow
L_{\text{total}}
$$

---

## 11. Backward Propagation

The forward direction is:

$$
x
\rightarrow
h
\rightarrow
\hat{y}
\rightarrow
L
$$

The backward direction is:

$$
L
\rightarrow
\frac{\partial L}{\partial \hat{y}}
\rightarrow
\frac{\partial L}{\partial h}
\rightarrow
\frac{\partial L}{\partial \theta}
$$

Because there are multiple tasks, there are multiple backward paths.

```mermaid
flowchart BT
    A["Ltotal"] --> B["Lcls"]
    A --> C["Lloc"]

    B --> D["Classification Head"]
    C --> E["Localization Head"]

    D --> F["Classification Gradient"]
    E --> G["Localization Gradient"]

    F --> H["Shared Representation h"]
    G --> H

    H --> I["Shared CNN Parameters θs"]
```

The critical point is:

> The gradients from different task branches are accumulated when they reach the shared representation.

---

## 12. Gradient Aggregation at the Shared Representation

We have:

$$
L_{\text{total}}
=
\lambda_{\text{cls}}L_{\text{cls}}
+
\lambda_{\text{loc}}L_{\text{loc}}
$$

We want:

$$
\frac{\partial L_{\text{total}}}{\partial h}
$$

Using the linearity of differentiation:

$$
\frac{\partial L_{\text{total}}}{\partial h}
=
\lambda_{\text{cls}}
\frac{\partial L_{\text{cls}}}{\partial h}
+
\lambda_{\text{loc}}
\frac{\partial L_{\text{loc}}}{\partial h}
$$

For $T$ tasks:

$$
\frac{\partial L_{\text{total}}}{\partial h}
=
\sum_{t=1}^{T}
\lambda_t
\frac{\partial L_t}{\partial h}
$$

This is the gradient aggregation that occurs in the shared part of the network.

---

## 13. Why Do the Gradients Add?

This is not a special rule created specifically for Multi-Task Learning.

It is a direct consequence of the **chain rule** and the structure of the computational graph.

If:

$$
z = a + b
$$

then gradients propagate through both branches. When multiple computational paths lead back to the same upstream variable or parameter, their gradient contributions are accumulated.

Conceptually:

```mermaid
flowchart TD
    A["Shared Representation h"] --> B["Task 1"]
    A --> C["Task 2"]

    B --> D["L1"]
    C --> E["L2"]

    D --> F["Gradient Contribution"]
    E --> G["Gradient Contribution"]

    F --> H["Shared Gradient"]
    G --> H
```

Thus:

$$
g_h
=
\lambda_1 g_1
+
\lambda_2 g_2
$$

---

## 14. Numerical Example: Forward to Backward

Consider a very small network.

Let:

$$
x = 1
$$

and let the shared parameter be:

$$
w_s = 1
$$

The shared representation is:

$$
h = w_sx = 1
$$

### 14.1 Classification Branch

Let:

$$
w_c = 0.8
$$

Then:

$$
\hat{p} = w_ch = 0.8
$$

Suppose the ground truth is:

$$
p = 1
$$

Using a simple squared error:

$$
L_{\text{cls}}
=
\frac{1}{2}(\hat{p}-p)^2
$$

Therefore:

$$
L_{\text{cls}}
=
\frac{1}{2}(0.8-1)^2
=
0.02
$$

### 14.2 Localization Branch

Let:

$$
w_b = 1.2
$$

Then:

$$
\hat{b} = w_bh = 1.2
$$

Suppose the ground truth is:

$$
b = 0.5
$$

The localization loss is:

$$
L_{\text{loc}}
=
\frac{1}{2}(\hat{b}-b)^2
$$

Therefore:

$$
L_{\text{loc}}
=
\frac{1}{2}(1.2-0.5)^2
=
0.245
$$

### 14.3 Total Loss

Assume:

$$
\lambda_{\text{cls}}
=
\lambda_{\text{loc}}
=
1
$$

Then:

$$
L_{\text{total}}
=
L_{\text{cls}}
+
L_{\text{loc}}
$$

$$
L_{\text{total}}
=
0.02+0.245
=
0.265
$$

---

## 15. Classification Backward Path

We want:

$$
\frac{\partial L_{\text{cls}}}{\partial h}
$$

Using the chain rule:

$$
\frac{\partial L_{\text{cls}}}{\partial h}
=
\frac{\partial L_{\text{cls}}}{\partial \hat{p}}
\frac{\partial \hat{p}}{\partial h}
$$

Since:

$$
L_{\text{cls}}
=
\frac{1}{2}(\hat{p}-p)^2
$$

we have:

$$
\frac{\partial L_{\text{cls}}}{\partial \hat{p}}
=
\hat{p}-p
=
0.8-1
=
-0.2
$$

Also:

$$
\hat{p}
=
w_ch
$$

so:

$$
\frac{\partial \hat{p}}{\partial h}
=
w_c
=
0.8
$$

Therefore:

$$
\frac{\partial L_{\text{cls}}}{\partial h}
=
(-0.2)(0.8)
=
-0.16
$$

The classification branch sends a gradient of:

$$
-0.16
$$

toward the shared representation.

---

## 16. Localization Backward Path

Similarly:

$$
\frac{\partial L_{\text{loc}}}{\partial h}
=
\frac{\partial L_{\text{loc}}}{\partial \hat{b}}
\frac{\partial \hat{b}}{\partial h}
$$

We have:

$$
\frac{\partial L_{\text{loc}}}{\partial \hat{b}}
=
\hat{b}-b
=
1.2-0.5
=
0.7
$$

and:

$$
\frac{\partial \hat{b}}{\partial h}
=
w_b
=
1.2
$$

Therefore:

$$
\frac{\partial L_{\text{loc}}}{\partial h}
=
0.7(1.2)
=
0.84
$$

The localization branch sends a gradient of:

$$
0.84
$$

toward the shared representation.

---

## 17. The Two Gradients Meet

We now have:

$$
\frac{\partial L_{\text{cls}}}{\partial h}
=
-0.16
$$

and:

$$
\frac{\partial L_{\text{loc}}}{\partial h}
=
0.84
$$

Since:

$$
L_{\text{total}}
=
L_{\text{cls}}
+
L_{\text{loc}}
$$

we obtain:

$$
\frac{\partial L_{\text{total}}}{\partial h}
=
-0.16 + 0.84
$$

Therefore:

$$
\frac{\partial L_{\text{total}}}{\partial h}
=
0.68
$$

This is the key step:

$$
\text{Task Gradients}
\rightarrow
\text{Gradient Aggregation}
\rightarrow
\text{Shared Representation}
$$

---

## 18. Backward into the Shared Parameter

Recall:

$$
h = w_sx
$$

and:

$$
x = 1
$$

Therefore:

$$
\frac{\partial h}{\partial w_s}
=
x
=
1
$$

Using the chain rule:

$$
\frac{\partial L_{\text{total}}}{\partial w_s}
=
\frac{\partial L_{\text{total}}}{\partial h}
\frac{\partial h}{\partial w_s}
$$

Thus:

$$
\frac{\partial L_{\text{total}}}{\partial w_s}
=
0.68 \times 1
=
0.68
$$

The shared parameter therefore receives:

$$
\frac{\partial L_{\text{total}}}{\partial w_s}
=
0.68
$$

This gradient already contains contributions from both tasks.

---

## 19. Task-Specific Parameters vs. Shared Parameters

The classification parameter $w_c$ belongs only to the classification branch:

$$
\frac{\partial L_{\text{total}}}{\partial w_c}
=
\lambda_{\text{cls}}
\frac{\partial L_{\text{cls}}}{\partial w_c}
$$

The localization parameter $w_b$ belongs only to the localization branch:

$$
\frac{\partial L_{\text{total}}}{\partial w_b}
=
\lambda_{\text{loc}}
\frac{\partial L_{\text{loc}}}{\partial w_b}
$$

But the shared parameter $w_s$ belongs to both paths:

$$
\frac{\partial L_{\text{total}}}{\partial w_s}
=
\lambda_{\text{cls}}
\frac{\partial L_{\text{cls}}}{\partial w_s}
+
\lambda_{\text{loc}}
\frac{\partial L_{\text{loc}}}{\partial w_s}
$$

This is the fundamental distinction between:

- **shared parameters**;
- **task-specific parameters**.

---

## 20. Parameter Update

After backpropagation, the optimizer performs one update using the aggregated gradient.

For the shared parameter:

$$
w_s^{\text{new}}
=
w_s^{\text{old}}
-
\eta
\frac{\partial L_{\text{total}}}{\partial w_s}
$$

In our example:

$$
w_s^{\text{new}}
=
1-\eta(0.68)
$$

Similarly:

$$
w_c^{\text{new}}
=
w_c^{\text{old}}
-
\eta
\frac{\partial L_{\text{total}}}{\partial w_c}
$$

and:

$$
w_b^{\text{new}}
=
w_b^{\text{old}}
-
\eta
\frac{\partial L_{\text{total}}}{\partial w_b}
$$

The shared parameter is updated **once** using the aggregated gradient.

We do not perform one optimizer update for classification, another update for localization, and then combine the updated parameters.

---

## 21. The Role of $\lambda$

Suppose:

$$
\lambda_{\text{cls}} = 1
$$

and:

$$
\lambda_{\text{loc}} = 1
$$

Then:

$$
g_h
=
-0.16 + 0.84
=
0.68
$$

Now suppose:

$$
\lambda_{\text{loc}} = 0.1
$$

Then:

$$
g_h
=
-0.16 + 0.1(0.84)
=
-0.076
$$

The direction of the shared gradient has changed.

Therefore, the task weights $\lambda_t$ do not only affect the magnitude of the objective. They can also affect the **optimization direction** of the shared representation.

---

## 22. Gradient Conflict

This leads to a deeper Multi-Task Learning insight.

Suppose:

$$
g_1 > 0
$$

while:

$$
g_2 < 0
$$

Then the two tasks want the shared parameters to move in different directions.

For example:

$$
g_1 = 0.8
$$

and:

$$
g_2 = -0.7
$$

Then:

$$
g_{\text{shared}}
=
0.8 - 0.7
=
0.1
$$

The two gradients almost cancel each other.

This is an example of **gradient conflict** in Multi-Task Learning.

The important conceptual point is that shared learning is not always automatically beneficial. Different tasks can sometimes provide competing optimization signals.

---

## 23. What Happens When There Is No Object?

Now consider:

$$
p_c = 0
$$

The object does not exist, so the bounding-box parameters are not meaningful.

Suppose:

$$
\hat{p}_c = 0.2
$$

Then:

$$
L_{\text{objectness}}
=
(\hat{p}_c-p_c)^2
$$

Therefore:

$$
L_{\text{objectness}}
=
(0.2-0)^2
=
0.04
$$

However:

$$
\mathbf{1}_{\{p_c=1\}}=0
$$

so the localization and classification terms are masked.

Therefore:

$$
L_{\text{total}}
=
L_{\text{objectness}}
$$

Conceptually:

```mermaid
flowchart LR
    A["No object: pc = 0"] --> B["Objectness Loss"]
    A --> C["Localization Loss"]
    A --> D["Classification Loss"]

    C --> E["Mask = 0"]
    D --> F["Mask = 0"]

    B --> G["Ltotal"]
    E --> G
    F --> G
```

The meaning is:

> **If no object exists, the model only needs to learn that no object exists.**

It does not need to learn the location or class of an object that is not present.

---

## 24. Complete Forward-to-Backward Flow

The entire Object Localization training process can be summarized as:

```mermaid
flowchart TD
    A["Input Image x"] --> B["Shared CNN θs"]
    B --> C["Shared Representation h"]

    C --> D["Classification / Objectness Head θcls"]
    C --> E["Localization Head θloc"]

    D --> F["p̂c, ĉ1, ĉ2, ..."]
    E --> G["b̂x, b̂y, b̂h, b̂w"]

    F --> H["Lcls / Objectness Loss"]
    G --> I["Lloc / Bounding-Box Loss"]

    H --> J["Ltotal"]
    I --> J

    J --> K["Classification Backward Path"]
    J --> L["Localization Backward Path"]

    K --> M["Gradient Contribution"]
    L --> N["Gradient Contribution"]

    M --> O["Gradient Accumulation"]
    N --> O

    O --> P["Shared CNN Gradient"]
    P --> Q["Optimizer Update"]
```

The forward process is:

$$
h = f(x;\theta_s)
$$

$$
\hat{y}_{\text{cls}}
=
g_{\text{cls}}(h;\theta_{\text{cls}})
$$

$$
\hat{y}_{\text{loc}}
=
g_{\text{loc}}(h;\theta_{\text{loc}})
$$

$$
L_{\text{total}}
=
\lambda_{\text{cls}}L_{\text{cls}}
+
\lambda_{\text{loc}}L_{\text{loc}}
$$

The backward process is:

$$
\frac{\partial L_{\text{total}}}{\partial h}
=
\lambda_{\text{cls}}
\frac{\partial L_{\text{cls}}}{\partial h}
+
\lambda_{\text{loc}}
\frac{\partial L_{\text{loc}}}{\partial h}
$$

Then:

$$
\frac{\partial L_{\text{total}}}{\partial \theta_s}
=
\frac{\partial L_{\text{total}}}{\partial h}
\frac{\partial h}{\partial \theta_s}
$$

Finally:

$$
\theta_s^{\text{new}}
=
\theta_s^{\text{old}}
-
\eta
\frac{\partial L_{\text{total}}}{\partial \theta_s}
$$

---

## 25. Core Mental Model

Object Localization can be understood at three levels.

### 25.1 Problem Level

$$
\text{What?} + \text{Where?}
$$

### 25.2 Architecture Level

$$
\text{Shared CNN}
\rightarrow
\text{Task-Specific Heads}
$$

### 25.3 Optimization Level

$$
L_{\text{total}}
=
\sum_t \lambda_t L_t
$$

For shared parameters:

$$
\nabla_{\theta_s}L_{\text{total}}
=
\sum_t
\lambda_t
\nabla_{\theta_s}L_t
$$

The most important idea is:

> **Classification and localization have separate task-specific computations, but their gradients are accumulated when they flow back into the shared representation and shared parameters.**

---

## 26. From Object Localization to Object Detection

Object Localization typically assumes one main object and therefore predicts one bounding box:

$$
[p_c,b_x,b_y,b_h,b_w,c_1,\ldots,c_n]
$$

But a real image may contain multiple objects:

```text
┌────────────────────────────────┐
│                                │
│     Car             Person     │
│                                │
│ Motorcycle                     │
│                                │
└────────────────────────────────┘
```

One bounding box is no longer sufficient.

We now need:

$$
\text{Image}
\rightarrow
\{
\text{object}_1,
\text{object}_2,
\ldots
\}
$$

This leads to:

$$
\text{Object Detection}
$$

The logical progression of Week 3 is:

$$
\text{Object Localization}
\rightarrow
\text{Sliding Window}
\rightarrow
\text{Convolutional Sliding Window}
\rightarrow
\text{YOLO}
$$

followed by:

$$
\text{YOLO}
\rightarrow
\text{IoU}
\rightarrow
\text{Non-Max Suppression}
\rightarrow
\text{Anchor Boxes}
$$

The important idea is that Object Detection is not an unrelated new problem. It extends localization from:

$$
1\text{ image} \rightarrow 1\text{ object}
$$

to:

$$
1\text{ image} \rightarrow \text{multiple objects}
$$

# Landmark Detection

Landmark Detection is the next concept after **Object Localization** in Week 3.

If Object Localization asks:

> **What is the object, and where is it located?**

Landmark Detection asks:

> **Where are the important points or keypoints inside the object?**

This makes Landmark Detection useful for tasks such as:

- face alignment;
- facial landmark detection;
- human pose estimation;
- hand keypoint detection;
- object pose estimation.

---

## 1. From Object Localization to Landmark Detection

In Object Localization, the model predicts a bounding box:

$$
(b_x,b_y,b_h,b_w)
$$

The bounding box tells us:

> **Where is the object?**

For example:

    ┌──────────────────────────────┐
    │                              │
    │       ┌──────────────┐       │
    │       │    FACE      │       │
    │       │              │       │
    │       └──────────────┘       │
    │                              │
    └──────────────────────────────┘

However, the bounding box does not tell us:

- where the left eye is;
- where the right eye is;
- where the nose is;
- where the mouth is.

Landmark Detection goes one step further by predicting important points inside the object.

The conceptual transition is:

$$
\text{Object Region}
\rightarrow
\text{Important Points}
$$

---

## 2. What Is a Landmark?

A **landmark** is a point with a meaningful geometric or semantic location on an object.

For example, for a face:

    ●         ●
  left eye  right eye

       ●
      nose

    ●     ●
  mouth  mouth

Each landmark is represented by a 2D coordinate:

$$
(x_i,y_i)
$$

If there are $N$ landmarks, the network predicts:

$$
2N
$$

coordinate values.

The output can be written as:

$$
\hat{y}
=
\begin{bmatrix}
\hat{x}_1 \\
\hat{y}_1 \\
\hat{x}_2 \\
\hat{y}_2 \\
\vdots \\
\hat{x}_N \\
\hat{y}_N
\end{bmatrix}
$$

Therefore:

$$
\text{Number of output values}
=
2N
$$

---

## 3. Example: Face Landmark Detection

Suppose we detect five facial landmarks:

1. left eye;
2. right eye;
3. nose;
4. left mouth corner;
5. right mouth corner.

Then:

$$
N=5
$$

so the network must predict:

$$
2N=10
$$

values.

The prediction vector is:

$$
\hat{y}
=
\begin{bmatrix}
\hat{x}_1,\hat{y}_1,
\hat{x}_2,\hat{y}_2,
\hat{x}_3,\hat{y}_3,
\hat{x}_4,\hat{y}_4,
\hat{x}_5,\hat{y}_5
\end{bmatrix}^{T}
$$

The ground truth is:

$$
y
=
\begin{bmatrix}
x_1,y_1,
x_2,y_2,
x_3,y_3,
x_4,y_4,
x_5,y_5
\end{bmatrix}^{T}
$$

---

## 4. Landmark Detection as Coordinate Regression

Classification produces a categorical output.

For example:

$$
\hat{y}\in\mathbb{R}^{C}
$$

The model answers:

> Which class does the object belong to?

Landmark Detection instead predicts continuous coordinates:

$$
\hat{y}\in\mathbb{R}^{2N}
$$

The model answers:

> Where are the important points?

Therefore, Landmark Detection is fundamentally a **coordinate regression problem**.

The distinction is:

    Classification
        ↓
    Discrete semantic output

    Landmark Detection
        ↓
    Continuous geometric output

The network learns a mapping:

$$
\text{Visual Features}
\rightarrow
\text{Geometric Coordinates}
$$

---

## 5. Why Can a CNN Predict Landmarks?

The basic architecture is still a CNN:

$$
x
\rightarrow
\text{CNN}
\rightarrow
\text{Landmark Coordinates}
$$

The CNN first learns a visual representation:

$$
h=f(x;\theta)
$$

This representation can contain information about:

- edges;
- textures;
- shapes;
- object parts;
- spatial relationships.

A landmark head then maps this representation to coordinates:

$$
\hat{y}=g(h;W,b)
$$

The complete flow is:

```mermaid
flowchart LR
    A["Input Image x"] --> B["CNN"]
    B --> C["Visual Representation h"]
    C --> D["Landmark Regression Head"]
    D --> E["Landmark Coordinates"]
```

The important idea is:

> The CNN does not directly observe coordinate labels as visual objects. It learns a mapping from visual patterns to geometric positions.

---

## 6. Coordinate Normalization

Landmark coordinates are usually normalized relative to the image dimensions.

Suppose an image has:

$$
W=1000
$$

pixels in width and:

$$
H=800
$$

pixels in height.

Suppose a landmark is located at:

$$
(x,y)=(400,200)
$$

The normalized coordinates are:

$$
x_{\text{norm}}
=
\frac{400}{1000}
=
0.4
$$

and:

$$
y_{\text{norm}}
=
\frac{200}{800}
=
0.25
$$

Therefore:

$$
(x,y)
\rightarrow
(x_{\text{norm}},y_{\text{norm}})
$$

with:

$$
x_{\text{norm}},y_{\text{norm}}\in[0,1]
$$

Normalization makes the regression target less dependent on the absolute pixel dimensions of the image.

---

## 7. Forward Propagation

The forward computation is:

$$
h=f(x;\theta)
$$

followed by:

$$
\hat{y}=g(h;W,b)
$$

where:

$$
\hat{y}
=
\begin{bmatrix}
\hat{x}_1 \\
\hat{y}_1 \\
\vdots \\
\hat{x}_N \\
\hat{y}_N
\end{bmatrix}
$$

The loss is then:

$$
L=L(\hat{y},y)
$$

The overall flow is:

```mermaid
flowchart LR
    A["Input Image x"] --> B["CNN"]
    B --> C["Feature Representation h"]
    C --> D["Landmark Head"]
    D --> E["x̂1, ŷ1, ..., x̂N, ŷN"]
    E --> F["Landmark Loss"]
```

So the forward process is:

$$
x
\rightarrow
h
\rightarrow
\hat{y}
\rightarrow
L
$$

---

## 8. Landmark Loss

Because the output consists of continuous coordinates, a simple loss is squared error.

For $N$ landmarks:

$$
L
=
\sum_{i=1}^{N}
\left[
(\hat{x}_i-x_i)^2
+
(\hat{y}_i-y_i)^2
\right]
$$

Another way to interpret this is to treat each landmark as a 2D point.

Let:

$$
p_i=(x_i,y_i)
$$

and:

$$
\hat{p}_i=(\hat{x}_i,\hat{y}_i)
$$

Then:

$$
L
=
\sum_{i=1}^{N}
\left\|
\hat{p}_i-p_i
\right\|^2
$$

This gives a useful geometric interpretation:

> The model is trained to move each predicted landmark toward its corresponding ground-truth point.

---

## 9. Intuition Behind the Landmark Loss

Suppose one landmark has ground truth:

$$
p=(0.4,0.6)
$$

but the model predicts:

$$
\hat{p}=(0.5,0.55)
$$

Then the coordinate errors are:

$$
\Delta x
=
0.5-0.4
=
0.1
$$

and:

$$
\Delta y
=
0.55-0.6
=
-0.05
$$

The squared loss is:

$$
L
=
0.1^2+(-0.05)^2
$$

$$
L
=
0.0125
$$

Geometrically:

    Ground Truth
          ●
          │
          │
          │
          ● Prediction

The loss measures how far the predicted point is from the target point.

---

## 10. Backward Propagation

Suppose a single landmark has loss:

$$
L
=
\frac{1}{2}
\left[
(\hat{x}-x)^2
+
(\hat{y}-y)^2
\right]
$$

The gradients with respect to the predicted coordinates are:

$$
\frac{\partial L}{\partial\hat{x}}
=
\hat{x}-x
$$

and:

$$
\frac{\partial L}{\partial\hat{y}}
=
\hat{y}-y
$$

These gradients tell the model how the predicted landmark should move.

For example:

- if $\hat{x}>x$, the gradient pushes the prediction toward smaller $x$;
- if $\hat{x}<x$, it pushes the prediction toward larger $x$;
- the same logic applies to the $y$ coordinate.

This is the core mechanism of coordinate regression.

---

## 11. Backward Through the CNN

Suppose:

$$
h=f(x;\theta)
$$

and:

$$
\hat{y}=g(h;W,b)
$$

The loss is:

$$
L=L(\hat{y},y)
$$

The backward flow is:

$$
L
\rightarrow
\frac{\partial L}{\partial\hat{y}}
\rightarrow
\frac{\partial L}{\partial h}
\rightarrow
\frac{\partial L}{\partial\theta}
$$

Conceptually:

```mermaid
flowchart RL
    A["Landmark Loss"] --> B["Landmark Head"]
    B --> C["CNN Representation"]
    C --> D["CNN Parameters"]
```

If the predicted landmarks are inaccurate, the gradient flows backward through the landmark head and into the CNN, modifying the visual features that are used for landmark localization.

---

## 12. Numerical Example: Forward to Backward

Consider a very small network with one landmark coordinate.

Let:

$$
x=1
$$

The shared parameter is:

$$
w_s=1
$$

Therefore:

$$
h=w_sx=1
$$

The landmark head is:

$$
\hat{x}=w_xh
$$

with:

$$
w_x=0.8
$$

Therefore:

$$
\hat{x}=0.8
$$

Suppose the ground-truth coordinate is:

$$
x=0.5
$$

The loss is:

$$
L
=
\frac{1}{2}(\hat{x}-x)^2
$$

Therefore:

$$
L
=
\frac{1}{2}(0.8-0.5)^2
=
0.045
$$

### 12.1 Gradient at the Landmark Output

We first compute:

$$
\frac{\partial L}{\partial\hat{x}}
=
\hat{x}-x
$$

Therefore:

$$
\frac{\partial L}{\partial\hat{x}}
=
0.8-0.5
=
0.3
$$

### 12.2 Gradient Back to the Shared Representation

Since:

$$
\hat{x}=w_xh
$$

we have:

$$
\frac{\partial\hat{x}}{\partial h}
=
w_x
=
0.8
$$

Therefore:

$$
\frac{\partial L}{\partial h}
=
\frac{\partial L}{\partial\hat{x}}
\frac{\partial\hat{x}}{\partial h}
$$

$$
\frac{\partial L}{\partial h}
=
0.3\times0.8
=
0.24
$$

### 12.3 Gradient Back to the Shared Parameter

Recall:

$$
h=w_sx
$$

Therefore:

$$
\frac{\partial h}{\partial w_s}
=
x
=
1
$$

So:

$$
\frac{\partial L}{\partial w_s}
=
\frac{\partial L}{\partial h}
\frac{\partial h}{\partial w_s}
$$

$$
\frac{\partial L}{\partial w_s}
=
0.24\times1
=
0.24
$$

### 12.4 Parameter Update

With learning rate $\eta$:

$$
w_s^{\text{new}}
=
w_s^{\text{old}}
-
\eta
\frac{\partial L}{\partial w_s}
$$

Therefore:

$$
w_s^{\text{new}}
=
1-\eta(0.24)
$$

The complete chain is:

$$
\text{Input}
\rightarrow
\text{Feature}
\rightarrow
\text{Landmark}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradient}
\rightarrow
\text{CNN Update}
$$

---

## 13. Is Landmark Detection Multi-Task Learning?

**Landmark Detection itself does not have to be Multi-Task Learning.**

A simple landmark detector can have one task:

$$
\text{Image}
\rightarrow
\text{CNN}
\rightarrow
\text{Landmark Coordinates}
$$

with one loss:

$$
L_{\text{landmark}}
$$

This is a **single-task learning** problem.

However, Landmark Detection can become one task inside a Multi-Task Learning architecture.

For example:

```mermaid
flowchart TD
    A["Input Image"] --> B["Shared CNN"]
    B --> C["Shared Representation"]

    C --> D["Face Classification"]
    C --> E["Landmark Detection"]
    C --> F["Head Pose Estimation"]

    D --> G["Lclass"]
    E --> H["Llandmark"]
    F --> I["Lpose"]

    G --> J["Ltotal"]
    H --> J
    I --> J
```

The total loss becomes:

$$
L_{\text{total}}
=
\lambda_1L_{\text{class}}
+
\lambda_2L_{\text{landmark}}
+
\lambda_3L_{\text{pose}}
$$

This gives an important distinction:

> **Landmark Detection is a task. Multi-Task Learning is a framework for jointly learning multiple tasks.**

These two concepts are not the same thing.

---

## 14. Object Localization vs. Landmark Detection

This distinction is important.

### 14.1 Object Localization

The model predicts:

$$
(b_x,b_y,b_h,b_w)
$$

It answers:

> **Where is the object?**

### 14.2 Landmark Detection

The model predicts:

$$
(x_1,y_1),\ldots,(x_N,y_N)
$$

It answers:

> **Where are the important points inside the object?**

Object Localization describes a region:

    ┌──────────────┐
    │              │
    │     FACE     │
    │              │
    └──────────────┘

Landmark Detection describes internal structure:

    ┌──────────────┐
    │   ●      ●   │
    │              │
    │      ●       │
    │              │
    │   ●      ●   │
    └──────────────┘

Therefore:

$$
\text{Localization}
\rightarrow
\text{Region}
$$

while:

$$
\text{Landmark Detection}
\rightarrow
\text{Keypoints}
$$

---

## 15. Bounding Box vs. Landmarks

These two representations describe different levels of geometry.

A bounding box provides:

$$
\text{Coarse Geometry}
$$

Landmarks provide:

$$
\text{Fine-Grained Geometry}
$$

A bounding box tells us:

> The object occupies this region.

Landmarks tell us:

> Important semantic parts of the object are located at these positions.

Therefore:

$$
\text{Localization}
\rightarrow
\text{Coarse Geometry}
$$

while:

$$
\text{Landmarks}
\rightarrow
\text{Fine-Grained Geometry}
$$

---

## 16. Why Is Landmark Detection Useful?

Landmarks convert an image into a structured geometric representation.

For example:

$$
\text{Face}
\rightarrow
\{
\text{Eye},
\text{Nose},
\text{Mouth},
\ldots
\}
$$

This representation can then support other tasks.

For example:

$$
\text{Face}
\rightarrow
\text{Landmarks}
\rightarrow
\text{Face Alignment}
$$

or:

$$
\text{Face}
\rightarrow
\text{Landmarks}
\rightarrow
\text{Pose Estimation}
$$

or:

$$
\text{Face}
\rightarrow
\text{Landmarks}
\rightarrow
\text{Expression Analysis}
$$

Thus, landmark detection can act as an **intermediate geometric representation**.

---

## 17. Landmark Detection as a Geometric Representation

A CNN normally learns:

$$
\text{Image}
\rightarrow
\text{Feature Representation}
$$

Landmark Detection extends this idea:

$$
\text{Feature Representation}
\rightarrow
\text{Geometric Representation}
$$

where:

$$
G
=
\{
(x_1,y_1),
(x_2,y_2),
\ldots,
(x_N,y_N)
\}
$$

Instead of only saying:

> This is a face.

the model provides a structured representation:

> This is a face, and its important parts are located at these coordinates.

This is one of the most useful ways to understand the purpose of landmark detection.

---

## 18. Connection to Object Detection

The concepts in Week 3 can be viewed as progressively richer forms of spatial understanding.

```mermaid
flowchart LR
    A["Classification"] --> B["Object Localization"]
    B --> C["Landmark Detection"]
    B --> D["Object Detection"]
```

They answer different questions.

### 18.1 Classification

> **What is in the image?**

### 18.2 Object Localization

> **Where is the object?**

### 18.3 Landmark Detection

> **Where are important points inside the object?**

### 18.4 Object Detection

> **What objects are present, and where is each one?**

Landmark Detection is therefore not simply "multiple bounding boxes."

It is a **keypoint or coordinate regression problem**.

---

## 19. Core Mental Model

The simplest mental model for Landmark Detection is:

$$
\text{Image}
\rightarrow
\text{CNN}
\rightarrow
\text{Visual Representation}
\rightarrow
\text{Landmark Coordinates}
$$

For $N$ landmarks:

$$
\hat{y}
=
[
\hat{x}_1,\hat{y}_1,
\ldots,
\hat{x}_N,\hat{y}_N
]^T
$$

A simple landmark regression loss is:

$$
L
=
\sum_{i=1}^{N}
\left[
(\hat{x}_i-x_i)^2
+
(\hat{y}_i-y_i)^2
\right]
$$

The backward flow is:

$$
L
\rightarrow
\text{Landmark Head}
\rightarrow
\text{CNN}
\rightarrow
\text{Parameter Update}
$$

The most important distinction from Object Localization is:

> **Landmark Detection is a task. It becomes Multi-Task Learning only when it is jointly trained with other tasks using a shared representation.**

---

## 20. Connection to the Next Concepts

The next major question in Week 3 is:

> How can we detect **multiple objects** in a single image?

This leads to:

$$
\text{Object Detection}
$$

and then to:

$$
\text{Sliding Window}
\rightarrow
\text{Convolutional Sliding Window}
\rightarrow
\text{YOLO}
\rightarrow
\text{IoU}
\rightarrow
\text{Non-Max Suppression}
\rightarrow
\text{Anchor Boxes}
$$

The overall progression can be viewed as:

$$
\text{Classification}
\rightarrow
\text{Localization}
\rightarrow
\text{Landmarks / Detection}
$$

The key conceptual transition is from:

> **Where is the object?**

to:

> **Where are the important geometric structures inside the object?**

and then to:

> **Which objects are present, and where is each one?**

# Object Detection

Object Detection is the next major step after **Object Localization**.

In Object Localization, we usually assume that the image contains one main object, and the network predicts:

> **What is the object, and where is it located?**

In a real image, however, there may be multiple objects.

    ┌──────────────────────────────────────┐
    │                                      │
    │      CAR                 PERSON      │
    │                                      │
    │   MOTORCYCLE                         │
    │                                      │
    └──────────────────────────────────────┘

Now a single bounding box is no longer sufficient.

The model must answer:

> **Which objects are present?**

and:

> **Where is each object located?**

This is the core problem of **Object Detection**.

At a high level:

$$
\text{Object Detection}
=
\text{Classification}
+
\text{Localization}
+
\text{Multiplicity}
$$

Here, multiplicity means that the model must be able to detect multiple objects in the same image.

---

## 1. From Classification to Localization to Detection

The progression of Week 3 can be understood as:

~~~mermaid
flowchart LR
    A["Image Classification"] --> B["Object Localization"]
    B --> C["Object Detection"]
~~~

### 1.1 Image Classification

The model predicts only the class:

$$
\text{Image}
\rightarrow
\text{Class}
$$

For example:

    Image
      ↓
    CNN
      ↓
    Cat

The model knows **what** is present, but not **where** it is.

### 1.2 Object Localization

The model predicts:

$$
\text{Image}
\rightarrow
\text{Class}
+
\text{One Bounding Box}
$$

It answers:

> This is a car, and it is located in this region.

### 1.3 Object Detection

The model predicts:

$$
\text{Image}
\rightarrow
\text{Multiple Objects}
+
\text{Multiple Bounding Boxes}
$$

For example:

    Image
      │
      ├── Car
      │    └── Bounding Box
      │
      ├── Person
      │    └── Bounding Box
      │
      └── Motorcycle
           └── Bounding Box

Therefore:

$$
\text{Localization}
=
1\text{ main object}
$$

while:

$$
\text{Detection}
=
\text{multiple objects}
$$

The key difficulty is that the number of objects is not known in advance.

---

## 2. Why Is Object Detection Harder?

Classification requires one main prediction:

$$
\hat{y}
$$

Localization requires one bounding box:

$$
(b_x,b_y,b_h,b_w)
$$

Object Detection is different because the number of objects can vary.

For one image:

$$
N=0
$$

For another:

$$
N=1
$$

For another:

$$
N=10
$$

The model therefore cannot simply use a fixed output such as:

$$
[
\text{class},
b_x,b_y,b_h,b_w
]
$$

because this represents only one object.

The central challenge becomes:

> **How can a neural network generate multiple object predictions from a single image?**

This question motivates the major detection approaches introduced in Week 3.

---

## 3. What Does an Object Detector Need to Predict?

Suppose we have three classes:

- pedestrian;
- car;
- motorcycle.

For one object, the prediction can contain:

$$
p_c
$$

for objectness,

$$
b_x,b_y,b_h,b_w
$$

for the bounding box,

and:

$$
c_1,c_2,c_3
$$

for the class.

Thus, one object prediction can be written as:

$$
y
=
[
p_c,
b_x,
b_y,
b_h,
b_w,
c_1,
c_2,
c_3
]
$$

This is the same basic idea introduced in Object Localization.

The difference is that Object Detection must produce **many such predictions**.

Therefore:

$$
\text{Localization}
\rightarrow
1\text{ prediction}
$$

while:

$$
\text{Detection}
\rightarrow
\text{many predictions}
$$

---

## 4. The Core Question: Where Do Multiple Predictions Come From?

Consider an image containing two objects:

    ┌──────────────────────────────┐
    │                              │
    │        CAR                   │
    │                              │
    │                   PERSON     │
    │                              │
    └──────────────────────────────┘

The CNN needs to determine:

- which regions contain objects;
- what those objects are;
- where their bounding boxes are.

A direct idea is:

> Divide the image into local regions and inspect them one by one.

This leads to **Sliding Window**.

---

## 5. Sliding Window

The basic idea is to scan the image using a fixed-size window.

For example:

    ┌──────────────────────────────┐
    │                              │
    │    ┌───────┐                 │
    │    │       │                 │
    │    │   ?   │                 │
    │    │       │                 │
    │    └───────┘                 │
    │                              │
    └──────────────────────────────┘

The selected region is passed to a CNN.

The CNN answers:

> Is there an object in this region?

Then the window is shifted:

    ┌──────────────────────────────┐
    │                              │
    │        ┌───────┐             │
    │        │       │             │
    │        │   ?   │             │
    │        │       │             │
    │        └───────┘             │
    │                              │
    └──────────────────────────────┘

The process continues across the image.

~~~mermaid
flowchart LR
    A["Image"] --> B["Window 1"]
    B --> C["CNN"]
    C --> D["Prediction"]

    A --> E["Window 2"]
    E --> F["CNN"]
    F --> G["Prediction"]

    A --> H["Window 3"]
    H --> I["CNN"]
    I --> J["Prediction"]
~~~

This converts Object Detection into a sequence of local classification and localization problems.

---

## 6. Why Does Sliding Window Work?

The key idea is:

$$
\text{Large Image}
\rightarrow
\text{Many Local Regions}
\rightarrow
\text{Prediction for Each Region}
$$

Each local region is treated as a candidate place where an object might exist.

If the window overlaps an object sufficiently, the CNN may predict:

- objectness;
- class;
- bounding box.

So conceptually:

$$
\text{Object Detection}
\approx
\text{Repeated Localization}
$$

This is a useful starting point, but it introduces a major computational problem.

---

## 7. The Computational Problem of Sliding Window

Suppose the image requires:

$$
N_{\text{windows}}=1000
$$

windows.

If each window is processed independently, the CNN must perform approximately:

$$
1000
$$

separate forward passes.

In general:

$$
\text{Computation}
\propto
N_{\text{windows}}
\times
\text{CNN Cost}
$$

As the image becomes larger or the stride becomes smaller, the number of windows can become very large.

Therefore, although Sliding Window is conceptually simple, it is computationally inefficient.

This leads to the next question:

> **Can we share the CNN computation between different windows?**

---

## 8. Convolutional Implementation of Sliding Window

CNNs have a very important property:

> Convolution applies the same learned filters at many spatial locations.

Therefore, instead of running:

    Window 1 → CNN
    Window 2 → CNN
    Window 3 → CNN
    Window 4 → CNN

we want to process the whole image once:

~~~mermaid
flowchart LR
    A["Full Image"] --> B["Convolutional CNN"]
    B --> C["Spatial Feature Map"]
    C --> D["Predictions at Many Locations"]
~~~

The same CNN computation is reused across different spatial positions.

This gives:

$$
\text{Shared Convolution}
\rightarrow
\text{Shared Features}
\rightarrow
\text{Many Predictions}
$$

The crucial insight is:

> **The network can perform many local predictions while sharing the expensive feature-extraction computation.**

This is an important bridge from classical Sliding Window to convolutional object detectors.

---

## 9. Why Spatial Structure Matters

A classification network can eventually collapse its representation into a single vector:

$$
h\in\mathbb{R}^{d}
$$

But Object Detection needs to preserve information about **where** things occur.

Therefore, instead of immediately collapsing the spatial dimensions, the network maintains a feature map:

$$
H\times W\times C
$$

where:

- $H$: spatial height;
- $W$: spatial width;
- $C$: number of feature channels.

Each spatial location corresponds to a region of the input image.

For example:

    Feature Map

    ┌────┬────┬────┬────┐
    │    │    │    │    │
    ├────┼────┼────┼────┤
    │    │ car│    │    │
    ├────┼────┼────┼────┤
    │    │    │person   │
    ├────┼────┼────┼────┤
    │    │    │    │    │
    └────┴────┴────┴────┘

A detector can use these spatial locations to produce local predictions.

---

## 10. Object Detection as Dense Prediction

A useful way to think about Object Detection is as **dense prediction**.

Classification:

$$
\text{Image}
\rightarrow
\text{One Prediction}
$$

Detection:

$$
\text{Image}
\rightarrow
\text{Spatial Feature Map}
\rightarrow
\text{Many Predictions}
$$

Therefore:

$$
\text{Detection}
=
\text{Prediction Distributed Across Space}
$$

Conceptually:

~~~mermaid
flowchart TD
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Spatial Feature Map"]

    C --> D["Prediction at Location 1"]
    C --> E["Prediction at Location 2"]
    C --> F["Prediction at Location 3"]
    C --> G["..."]
~~~

Each spatial location can provide information about:

- objectness;
- bounding box;
- class.

This idea is fundamental for understanding YOLO.

---

## 11. Grid-Based View

To make the spatial prediction idea easier to understand, imagine dividing the image into an $S\times S$ grid.

For example:

    ┌────┬────┬────┬────┐
    │    │    │    │    │
    ├────┼────┼────┼────┤
    │    │    │    │    │
    ├────┼────┼────┼────┤
    │    │    │    │    │
    ├────┼────┼────┼────┤
    │    │    │    │    │
    └────┴────┴────┴────┘

Each grid cell is responsible for a region of the image.

If the center of an object lies inside a particular cell, that cell can be responsible for predicting that object.

This gives a simple organizational principle:

$$
\text{Image}
\rightarrow
\text{Grid Cells}
\rightarrow
\text{Local Object Predictions}
$$

This is the basic spatial idea behind the YOLO family introduced in the course.

---

## 12. Output Tensor of a Grid-Based Detector

Suppose we have:

- grid size: $S\times S$;
- $C$ classes.

For one cell, a prediction can contain:

$$
[
p_c,
b_x,
b_y,
b_h,
b_w,
c_1,
\ldots,
c_C
]
$$

The number of values for one prediction is:

$$
5+C
$$

Therefore, a simple output tensor can have the shape:

$$
S\times S\times(5+C)
$$

For example, if:

$$
S=3
$$

and:

$$
C=3
$$

then:

$$
3\times3\times8
$$

The important idea is not the specific numbers, but the structure:

> **A detector produces a prediction tensor spread across spatial locations.**

This is fundamentally different from classification, where the output is usually one vector:

$$
\hat{y}\in\mathbb{R}^{C}
$$

Detection instead produces:

$$
\hat{Y}\in\mathbb{R}^{S\times S\times(5+C)}
$$

The network is therefore producing predictions **across space**.

---

## 13. Interpretation of One Grid Cell

Consider one grid cell with prediction:

$$
[
p_c,
b_x,
b_y,
b_h,
b_w,
c_1,
c_2,
c_3
]
$$

We can interpret the components as:

### 13.1 Objectness

$$
p_c
$$

answers:

> Is there an object associated with this prediction?

### 13.2 Bounding Box

$$
b_x,b_y,b_h,b_w
$$

answer:

> Where is the predicted object?

### 13.3 Class

$$
c_1,c_2,c_3
$$

answer:

> What class is the object?

Thus one local prediction contains:

$$
\text{Objectness}
+
\text{Geometry}
+
\text{Semantics}
$$

This is the same conceptual decomposition from Object Localization, now distributed over many spatial locations.

---

## 14. Forward Propagation of an Object Detector

A simplified detector can be represented as:

~~~mermaid
flowchart LR
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Spatial Feature Map"]
    C --> D["Detection Head"]
    D --> E["Prediction Tensor"]
~~~

Mathematically:

$$
h=f(x;\theta)
$$

followed by:

$$
\hat{Y}=g(h;\phi)
$$

where $\hat{Y}$ is no longer a simple vector.

Instead:

$$
\hat{Y}
\in
\mathbb{R}^{S\times S\times(5+C)}
$$

This is a major change from classification.

Classification:

$$
\hat{y}\in\mathbb{R}^{C}
$$

Detection:

$$
\hat{Y}\in\mathbb{R}^{S\times S\times(5+C)}
$$

---

## 15. Object Detection Loss

For each grid cell, we can define a local prediction:

$$
\hat{y}_{ij}
$$

and a corresponding target:

$$
y_{ij}
$$

The total loss can be understood as the sum over all spatial locations:

$$
L
=
\sum_i\sum_j L_{ij}
$$

Each local loss can conceptually contain:

$$
L_{ij}
=
L_{\text{objectness},ij}
+
L_{\text{localization},ij}
+
L_{\text{classification},ij}
$$

Therefore:

$$
L
=
\sum_{i,j}
L_{ij}
$$

This is an important difference from Object Localization.

In Object Localization, one object contributes one main localization problem.

In Detection, the training objective is distributed across many spatial locations.

---

## 16. Not Every Cell Contains an Object

Most cells will not contain the center of an object.

Therefore, the detector must distinguish between:

- cells responsible for objects;
- cells containing no relevant object.

Conceptually:

~~~mermaid
flowchart LR
    A["Grid Cell"] --> B{"Object Present?"}

    B -->|Yes| C["Objectness + Box + Class"]
    B -->|No| D["Objectness Only"]
~~~

This extends the idea from Object Localization:

> If there is no object, the model should not be penalized as though a meaningful bounding box existed there.

This becomes especially important in detection because many spatial locations may correspond to background.

---

## 17. Backward Propagation

The forward flow is:

$$
x
\rightarrow
h
\rightarrow
\hat{Y}
\rightarrow
L
$$

The backward flow is:

$$
L
\rightarrow
\frac{\partial L}{\partial\hat{Y}}
\rightarrow
\frac{\partial L}{\partial h}
\rightarrow
\frac{\partial L}{\partial\theta}
$$

Because the prediction tensor contains many spatial locations, the shared CNN can receive gradient contributions from many cells.

~~~mermaid
flowchart BT
    A["Total Detection Loss"] --> B["Cell 1 Loss"]
    A --> C["Cell 2 Loss"]
    A --> D["Cell 3 Loss"]
    A --> E["..."]

    B --> F["Shared CNN"]
    C --> F
    D --> F
    E --> F

    F --> G["CNN Parameters"]
~~~

If:

$$
L
=
\sum_{i,j}L_{ij}
$$

then:

$$
\frac{\partial L}{\partial\theta}
=
\sum_{i,j}
\frac{\partial L_{ij}}{\partial\theta}
$$

This means the shared CNN receives gradient contributions from many spatial locations.

---

## 18. Object Detection and Multi-Task Learning

Object Detection naturally combines several objectives.

For one prediction, we may have:

$$
L_{\text{objectness}}
$$

$$
L_{\text{localization}}
$$

and:

$$
L_{\text{classification}}
$$

These can be combined as:

$$
L_{\text{prediction}}
=
\lambda_{\text{obj}}L_{\text{objectness}}
+
\lambda_{\text{loc}}L_{\text{localization}}
+
\lambda_{\text{cls}}L_{\text{classification}}
$$

Then the image-level loss aggregates over spatial predictions:

$$
L
=
\sum_{i,j}
L_{ij}
$$

Thus Object Detection has two interacting structures.

### 18.1 Task Dimension

$$
\text{Objectness}
+
\text{Localization}
+
\text{Classification}
$$

### 18.2 Spatial Dimension

$$
\text{Cell}_{1,1},
\text{Cell}_{1,2},
\ldots,
\text{Cell}_{S,S}
$$

A useful general expression is:

$$
\frac{\partial L}{\partial\theta}
=
\sum_{i,j}
\sum_t
\lambda_t
\frac{\partial L_{ij,t}}{\partial\theta}
$$

This is not necessary to memorize. It is useful because it shows that a shared detector can receive gradient contributions from both:

- different tasks;
- different spatial locations.

---

## 19. Why Can One CNN Produce Many Predictions?

This is one of the key strengths of convolution.

The same convolutional filters are applied at many spatial locations.

Suppose a filter learns a useful visual pattern.

The same filter can detect that pattern at:

- the left side of the image;
- the center;
- the right side;
- different rows.

Therefore:

$$
\text{Shared Convolution}
\rightarrow
\text{Shared Feature Detector}
\rightarrow
\text{Many Spatial Responses}
$$

These responses form a spatial feature map.

The detection head can then turn these spatial responses into multiple predictions.

Therefore:

$$
\text{One CNN}
\rightarrow
\text{Many Spatial Predictions}
$$

This is one of the fundamental reasons convolutional networks are effective for Object Detection.

---

## 20. The Problem of Duplicate Predictions

Now we encounter a new problem.

Suppose there is only one car in the image.

Multiple spatial locations may predict the same car:

    Prediction 1
    ┌──────────────┐
    │     CAR      │
    └──────────────┘

    Prediction 2
    ┌────────────────┐
    │      CAR       │
    └────────────────┘

    Prediction 3
      ┌─────────────┐
      │     CAR     │
      └─────────────┘

The model may therefore produce multiple candidate boxes for the same object.

We do not want the final output to interpret them as multiple cars.

This leads to two important concepts:

$$
\text{IoU}
$$

and:

$$
\text{Non-Max Suppression}
$$

---

## 21. Intersection over Union

**Intersection over Union (IoU)** measures how much two bounding boxes overlap.

Given two boxes:

- predicted box;
- ground-truth box;

IoU is:

$$
IoU
=
\frac{
\text{Area of Intersection}
}{
\text{Area of Union}
}
$$

Conceptually:

    Ground Truth
    ┌─────────────────┐
    │                 │
    │    ┌───────────────┐
    │    │   Prediction  │
    │    │               │
    └────┴───────────────┘

If the boxes overlap strongly:

$$
IoU\rightarrow1
$$

If they do not overlap:

$$
IoU=0
$$

IoU is therefore a measure of **geometric agreement between bounding boxes**.

It is useful for:

- evaluating localization quality;
- identifying duplicate predictions;
- supporting Non-Max Suppression.

---

## 22. IoU as a Measure of Box Quality

Suppose the ground-truth box is:

$$
B_{\text{gt}}
$$

and the predicted box is:

$$
B_{\text{pred}}
$$

Then:

$$
IoU(B_{\text{pred}},B_{\text{gt}})
$$

measures how well the predicted region overlaps the true object region.

A larger IoU generally means that the predicted box is better aligned with the ground truth.

Thus:

$$
\text{IoU}
=
\text{Spatial Agreement Between Two Boxes}
$$

---

## 23. Non-Max Suppression

Suppose the detector produces:

    Car, confidence = 0.92
    Car, confidence = 0.88
    Car, confidence = 0.76

and their boxes overlap strongly.

We want to keep the strongest prediction and suppress redundant ones.

The basic Non-Max Suppression process is:

~~~mermaid
flowchart TD
    A["Candidate Predictions"] --> B["Select Highest-Confidence Box"]
    B --> C["Compute IoU with Other Boxes"]
    C --> D["Suppress Highly Overlapping Boxes"]
    D --> E["Repeat"]
    E --> F["Final Detections"]
~~~

Conceptually:

1. Choose the box with the highest confidence.
2. Keep it.
3. Compute IoU between it and the remaining boxes.
4. Remove boxes whose overlap is above the chosen threshold.
5. Repeat.

Thus:

$$
\text{Many Candidate Boxes}
\rightarrow
\text{Few Final Boxes}
$$

---

## 24. IoU vs. Non-Max Suppression

These concepts are related but not the same.

### 24.1 IoU

IoU is a **measurement**:

$$
IoU(A,B)
$$

It answers:

> How much do these two boxes overlap?

### 24.2 Non-Max Suppression

NMS is an **algorithmic decision procedure**.

It uses:

- confidence scores;
- IoU;

to answer:

> Which boxes should remain?

Therefore:

$$
\text{IoU}
=
\text{Overlap Measurement}
$$

while:

$$
\text{NMS}
=
\text{Box Selection and Suppression}
$$

---

## 25. Confidence Score

An object detector also needs to estimate how trustworthy each prediction is.

Conceptually, a good detection should have:

- high objectness;
- a plausible class prediction;
- a well-localized bounding box.

For example:

    Box A
    Objectness = 0.95
    Class = Car
    Confidence = High

    Box B
    Objectness = 0.20
    Class = Car
    Confidence = Low

Low-confidence predictions can be removed before or during post-processing.

The exact confidence formulation depends on the detector design, but the core idea is:

> **Confidence helps rank candidate detections by how strongly the model believes they represent real objects.**

---

## 26. Anchor Boxes

Another difficulty appears when different objects have very different shapes.

Consider:

    Tall Object

        ┌───┐
        │   │
        │   │
        │   │
        └───┘

    Wide Object

    ┌──────────────┐
    │              │
    └──────────────┘

A detector must handle many possible aspect ratios.

Anchor boxes introduce a set of predefined reference box shapes.

For example:

    Anchor 1
    ┌──────────┐
    │          │
    └──────────┘

    Anchor 2
    ┌────┐
    │    │
    │    │
    │    │
    └────┘

    Anchor 3
    ┌────────────────┐
    │                │
    └────────────────┘

The model can then learn how to adjust these reference boxes to match real objects.

---

## 27. Why Do We Need Anchor Boxes?

Without anchors, the model must directly learn:

$$
(b_x,b_y,b_h,b_w)
$$

for every possible object shape.

Anchor boxes provide useful geometric templates:

$$
\text{Anchor}
\rightarrow
\text{Box Refinement}
$$

Conceptually:

> Which anchor shape is appropriate for this object, and how should it be adjusted?

Therefore:

$$
\text{Anchor Boxes}
=
\text{Prior Geometric Templates}
$$

This reduces the burden of learning arbitrary box geometries from scratch.

---

## 28. The Complete Object Detection Pipeline

We can now connect the major components into one pipeline:

~~~mermaid
flowchart LR
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Spatial Feature Map"]
    C --> D["Detection Head"]
    D --> E["Candidate Predictions"]
    E --> F["Confidence Filtering"]
    F --> G["IoU"]
    G --> H["Non-Max Suppression"]
    H --> I["Final Detections"]
~~~

The final output may conceptually look like:

    Car
    Bounding Box
    Confidence

    Person
    Bounding Box
    Confidence

    Motorcycle
    Bounding Box
    Confidence

The important distinction is:

> **The neural network produces candidate predictions. The complete detection system may then apply post-processing to convert those candidates into final detections.**

---

## 29. Object Detection Is More Than Classification

A useful way to remember the increasing complexity is:

### 29.1 Classification

$$
\text{What?}
$$

### 29.2 Localization

$$
\text{What?}
+
\text{Where?}
$$

### 29.3 Detection

$$
\text{What?}
+
\text{Where?}
+
\text{How Many?}
$$

Therefore:

$$
\text{Object Detection}
=
\text{Semantic}
+
\text{Geometric}
+
\text{Multiplicity}
$$

where:

- semantic information gives the class;
- geometric information gives the bounding box;
- multiplicity allows multiple objects.

---

## 30. Forward and Backward View

A detector can be viewed as a computational graph:

~~~mermaid
flowchart TD
    A["Image x"] --> B["CNN Backbone"]
    B --> C["Feature Map h"]
    C --> D["Detection Head"]
    D --> E["Predictions"]

    E --> F["Objectness Loss"]
    E --> G["Localization Loss"]
    E --> H["Classification Loss"]

    F --> I["Total Loss"]
    G --> I
    H --> I

    I --> J["Backward"]
    J --> K["CNN Gradients"]
    K --> L["Parameter Update"]
~~~

The forward direction is:

$$
x
\rightarrow
h
\rightarrow
\hat{Y}
\rightarrow
L
$$

The backward direction is:

$$
L
\rightarrow
\frac{\partial L}{\partial\hat{Y}}
\rightarrow
\frac{\partial L}{\partial h}
\rightarrow
\frac{\partial L}{\partial\theta}
$$

If the prediction tensor contains many spatial locations:

$$
L
=
\sum_{i,j}
L_{ij}
$$

then the shared CNN receives:

$$
\frac{\partial L}{\partial\theta}
=
\sum_{i,j}
\frac{\partial L_{ij}}{\partial\theta}
$$

Thus, a shared convolutional parameter may receive gradient contributions from many spatial positions.

---

## 31. Object Detection vs. Object Localization

| Aspect | Object Localization | Object Detection |
| --- | --- | --- |
| Number of objects | Usually one main object | Multiple objects |
| Bounding boxes | One | Multiple |
| Class predictions | One | Multiple |
| Spatial predictions | One region | Many spatial locations |
| Main challenge | What + Where | What + Where + How Many |

The core distinction is:

$$
\text{Localization}
=
1\text{ object}
$$

while:

$$
\text{Detection}
=
N\text{ objects}
$$

where $N$ is not known in advance.

---

## 32. Where Does YOLO Fit?

YOLO, meaning **You Only Look Once**, is one of the key ideas introduced in this part of the course.

At a high level, YOLO processes the full image with a convolutional network and produces detection predictions in a single forward pass.

The mental model is:

~~~mermaid
flowchart LR
    A["Full Image"] --> B["CNN"]
    B --> C["Spatial Predictions"]
    C --> D["Bounding Boxes + Classes + Confidence"]
    D --> E["Post-Processing"]
    E --> F["Final Detections"]
~~~

This combines several concepts we have already studied:

$$
\text{CNN}
+
\text{Spatial Prediction}
+
\text{Bounding-Box Regression}
+
\text{Objectness}
+
\text{Class Prediction}
+
\text{Post-Processing}
$$

YOLO is therefore not an isolated idea.

It is the result of combining solutions to the major problems encountered in Object Detection.

---

## 33. The Logical Progression Toward YOLO

The progression can be understood causally:

$$
\text{Object Localization}
$$

$$
↓
$$

> How do we find one object?

$$
↓
$$

$$
\text{Sliding Window}
$$

$$
↓
$$

> How do we search across the image?

$$
↓
$$

$$
\text{Convolutional Sliding Window}
$$

$$
↓
$$

> How do we avoid recomputing CNN features for every window?

$$
↓
$$

$$
\text{YOLO}
$$

$$
↓
$$

> How do we efficiently predict many objects?

$$
↓
$$

$$
\text{IoU + NMS + Anchor Boxes}
$$

$$
↓
$$

> How do we turn many candidate predictions into useful final detections?

This is much more useful than memorizing these techniques as unrelated algorithms.

---

## 34. Core Mental Model

Object Detection can be understood at four levels.

### 34.1 Problem Level

$$
\text{What?}
+
\text{Where?}
+
\text{How Many?}
$$

### 34.2 Neural Network Level

$$
\text{Image}
\rightarrow
\text{CNN}
\rightarrow
\text{Spatial Prediction Tensor}
$$

### 34.3 Prediction Level

A local prediction can contain:

$$
p_c,
b_x,
b_y,
b_h,
b_w,
c_1,
\ldots,
c_C
$$

### 34.4 Post-Processing Level

$$
\text{Candidate Boxes}
\rightarrow
\text{Confidence Filtering}
\rightarrow
\text{IoU}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}
$$

Therefore, an object detector is not merely:

$$
\text{CNN}
$$

A more complete view is:

$$
\text{Object Detector}
=
\text{CNN}
+
\text{Prediction Mechanism}
+
\text{Post-Processing}
$$

---

## 35. Connection to the Rest of Week 3

The major concepts form a causal chain:

~~~mermaid
flowchart LR
    A["Object Localization"] --> B["Sliding Window"]
    B --> C["Convolutional Sliding Window"]
    C --> D["YOLO"]
    D --> E["IoU"]
    E --> F["Non-Max Suppression"]
    F --> G["Anchor Boxes"]
~~~

The underlying reasoning is:

$$
\text{Multiple Objects}
$$

$$
↓
$$

$$
\text{Need Multiple Predictions}
$$

$$
↓
$$

$$
\text{Sliding Window}
$$

$$
↓
$$

$$
\text{Too Much Computation}
$$

$$
↓
$$

$$
\text{Convolutional Sliding Window}
$$

$$
↓
$$

$$
\text{Need Efficient Dense Detection}
$$

$$
↓
$$

$$
\text{YOLO}
$$

$$
↓
$$

$$
\text{Duplicate Predictions}
$$

$$
↓
$$

$$
\text{IoU + NMS}
$$

$$
↓
$$

$$
\text{Different Object Shapes}
$$

$$
↓
$$

$$
\text{Anchor Boxes}
$$

This causal chain is the key mental model for the Object Detection section of Week 3.

# Convolutional Implementation of Sliding Windows

Convolutional Implementation of Sliding Windows is an important step in Week 3 because it addresses the main computational problem of **naive sliding-window detection**.

The core idea is:

$$
\text{Sliding Window}
\rightarrow
\text{Shared Convolutional Computation}
$$

Instead of running a CNN independently for every window, we transform the network so that it can process many spatial locations in **one forward pass**.

The key insight is:

> **Compute shared visual features once, then reuse them to make predictions at many locations.**

---

## 1. The Problem with Naive Sliding Windows

Suppose we want to detect objects in a large image.

We can select a window:

    ┌──────────────────────────────┐
    │                              │
    │    ┌───────┐                 │
    │    │       │                 │
    │    │   ?   │                 │
    │    │       │                 │
    │    └───────┘                 │
    │                              │
    └──────────────────────────────┘

Then:

$$
\text{Window}
\rightarrow
\text{CNN}
\rightarrow
\text{Prediction}
$$

Move the window:

    ┌──────────────────────────────┐
    │                              │
    │        ┌───────┐             │
    │        │       │             │
    │        │   ?   │             │
    │        │       │             │
    │        └───────┘             │
    │                              │
    └──────────────────────────────┘

and run the CNN again.

If the image requires $N$ windows:

$$
N
$$

separate CNN forward passes may be required.

Therefore:

$$
\text{Computation}
\propto
N
\times
\text{CNN Cost}
$$

This becomes very expensive when:

- the image is large;
- the stride is small;
- multiple scales are used;
- many windows overlap.

---

## 2. Why Is Naive Sliding Window Wasteful?

Consider two neighboring windows:

    Window 1

    ┌──────────────┐
    │              │
    │   Region A   │
    │              │
    └──────────────┘


    Window 2

          ┌──────────────┐
          │              │
          │   Region B   │
          │              │
          └──────────────┘

These windows overlap substantially.

However, naive sliding-window detection recomputes the CNN features for the overlapping regions.

Conceptually:

$$
\text{Window 1}
\rightarrow
\text{Feature Computation}
$$

and:

$$
\text{Window 2}
\rightarrow
\text{Same Feature Computation Again}
$$

This is the central inefficiency.

The important question becomes:

> **Can the computation for overlapping regions be shared?**

---

## 3. The Key Insight

CNNs already have a mechanism for sharing computation across spatial locations:

> **Convolution applies the same learned filters at many spatial positions.**

Therefore, instead of:

    Window 1 → CNN
    Window 2 → CNN
    Window 3 → CNN
    Window 4 → CNN

we want:

    Full Image
         ↓
    Shared CNN
         ↓
    Spatial Feature Map
         ↓
    Multiple Predictions

The goal is:

$$
\boxed{
\text{Compute Features Once}
+
\text{Reuse Them Across Locations}
}
$$

This is the fundamental idea of the **Convolutional Implementation of Sliding Windows**.

---

## 4. Fully Connected Layers and the Key Transformation

Suppose the original classifier is:

~~~mermaid
flowchart LR
    A["Input Window"] --> B["Convolution Layers"]
    B --> C["Pooling"]
    C --> D["Fully Connected"]
    D --> E["Output"]
~~~

The problem is that a fully connected layer is typically connected to a fixed-size feature vector.

For example, suppose the feature map before the FC layer has shape:

$$
H\times W\times C
$$

A fully connected neuron receives the entire feature map.

Its weight vector has:

$$
HWC
$$

parameters, and its computation is:

$$
z=w^Ta+b
$$

The important observation is:

> A fully connected neuron can be interpreted as a filter whose spatial size covers the entire input feature map.

This allows an FC layer to be converted into a convolutional layer.

---

## 5. Fully Connected Layer as a Convolution

Suppose the feature map before the FC layer has shape:

$$
5\times5\times64
$$

and the FC layer contains:

$$
400
$$

neurons.

We can reinterpret this as:

- $400$ convolutional filters;
- each filter has shape:

$$
5\times5\times64
$$

Conceptually:

~~~mermaid
flowchart LR
    A["5 × 5 × 64 Feature Map"] --> B["400 FC Neurons"]
    C["400 Filters of Size 5 × 5 × 64"] --> D["Convolutional Representation"]
~~~

Each FC neuron corresponds to one convolutional filter.

Therefore:

$$
\boxed{
\text{Fully Connected Layer}
\leftrightarrow
\text{Convolutional Layer}
}
$$

when the convolution filters cover the entire input feature map.

This is the key mathematical trick behind the convolutional implementation.

---

## 6. Why Does This Help Sliding Windows?

Suppose the original network was designed to classify a fixed-size window:

$$
14\times14\times3
$$

The original architecture might be:

$$
14\times14\times3
\rightarrow
\text{Conv}
\rightarrow
\text{Pool}
\rightarrow
\text{FC}
\rightarrow
\text{Output}
$$

After converting the FC layer into a convolutional layer, the network becomes fully convolutional.

Now instead of repeatedly cropping:

$$
\text{Window}_1
\rightarrow
\text{CNN}
$$

$$
\text{Window}_2
\rightarrow
\text{CNN}
$$

$$
\text{Window}_3
\rightarrow
\text{CNN}
$$

we can process the whole image:

$$
\text{Full Image}
\rightarrow
\text{Convolutional Network}
$$

and produce predictions at multiple spatial locations.

---

## 7. The Computational Difference

Naive sliding window:

~~~mermaid
flowchart TD
    A["Image"] --> B["Window 1"]
    A --> C["Window 2"]
    A --> D["Window 3"]

    B --> E["CNN"]
    C --> F["CNN"]
    D --> G["CNN"]

    E --> H["Prediction 1"]
    F --> I["Prediction 2"]
    G --> J["Prediction 3"]
~~~

Convolutional implementation:

~~~mermaid
flowchart TD
    A["Full Image"] --> B["Shared CNN"]
    B --> C["Shared Feature Map"]

    C --> D["Prediction 1"]
    C --> E["Prediction 2"]
    C --> F["Prediction 3"]
~~~

The difference is therefore:

$$
\boxed{
\text{Repeated Feature Extraction}
\rightarrow
\text{Shared Feature Extraction}
}
$$

The conceptual operation is still "make predictions at many locations", but the computation is reorganized so that expensive feature extraction is shared.

---

## 8. Why Does Convolution Allow This?

The reason is **weight sharing**.

Suppose a convolution filter is:

$$
W
$$

The same $W$ is applied at every spatial location:

$$
W_1=W_2=W_3=\cdots=W
$$

The network does not learn a different edge detector for every location.

Instead, one learned filter is reused across the image.

For example:

    Location 1 → Filter W
    Location 2 → Filter W
    Location 3 → Filter W
    Location 4 → Filter W

This gives:

$$
\boxed{
\text{One Feature Extractor}
\rightarrow
\text{Many Spatial Locations}
}
$$

This property is exactly what makes convolutional implementation of sliding windows possible.

---

## 9. Convolutional Sliding Window Does Not Change the Basic Idea

A common misconception is:

> "Convolutional implementation removes the sliding-window idea."

The better interpretation is:

> **The spatial scanning idea remains, but the computation is reorganized.**

Naive method:

$$
\text{Crop}
\rightarrow
\text{CNN}
\rightarrow
\text{Prediction}
$$

repeated many times.

Convolutional implementation:

$$
\text{Full Image}
\rightarrow
\text{Shared Convolution}
\rightarrow
\text{Predictions at Many Locations}
$$

Thus both methods are trying to answer:

> What happens if we apply the detector at different locations?

The difference is **how the computation is performed**.

---

## 10. Spatial Structure of the Feature Map

This leads to another important idea.

A standard classifier may eventually collapse its representation into a single vector:

$$
h\in\mathbb{R}^{d}
$$

But object detection requires information about **where** the features occur.

Therefore, we preserve the spatial dimensions:

$$
H\times W\times C
$$

where:

- $H$: feature-map height;
- $W$: feature-map width;
- $C$: number of channels.

Each spatial position corresponds to a region of the original image.

For example:

    Feature Map

    ┌────┬────┬────┬────┐
    │    │    │    │    │
    ├────┼────┼────┼────┤
    │    │ car│    │    │
    ├────┼────┼────┼────┤
    │    │    │person   │
    ├────┼────┼────┼────┤
    │    │    │    │    │
    └────┴────┴────┴────┘

Therefore:

$$
\text{Feature Map Location}
\leftrightarrow
\text{Input Image Region}
$$

This spatial correspondence is essential for detection.

---

## 11. Receptive Field and Spatial Correspondence

A neuron at a particular spatial location in a convolutional feature map corresponds to a **receptive field** in the input.

Conceptually:

    Input Image

    ┌──────────────────────────────┐
    │                              │
    │      ┌──────────────┐        │
    │      │ Receptive    │        │
    │      │    Field     │        │
    │      └──────────────┘        │
    │                              │
    └──────────────────────────────┘

Therefore:

$$
\text{Spatial Feature}
\leftrightarrow
\text{Spatial Region in Input}
$$

When the detection head produces an output at a certain feature-map location, that output can be associated with a particular region of the original image.

This is the geometric reason a convolutional feature map can support object localization.

---

## 12. Forward Propagation

Let the convolutional feature extractor be:

$$
h=f(x;\theta)
$$

The detection head produces:

$$
\hat{Y}=g(h;\phi)
$$

Unlike image classification, $\hat{Y}$ is not necessarily a single vector.

Instead:

$$
\hat{Y}
\in
\mathbb{R}^{H'\times W'\times K}
$$

where:

- $H'$: output spatial height;
- $W'$: output spatial width;
- $K$: number of output channels.

Each spatial position contains a prediction vector.

The forward flow is:

~~~mermaid
flowchart LR
    A["Full Image"] --> B["Convolutional Backbone"]
    B --> C["Spatial Feature Map"]
    C --> D["Convolutional Detection Head"]
    D --> E["H′ × W′ × K Output"]
~~~

---

## 13. Numerical Shape Example

Suppose the convolutional detector produces:

$$
4\times4\times10
$$

outputs.

There are:

$$
4\times4=16
$$

spatial positions.

Each position has:

$$
10
$$

values.

Therefore, the output contains:

$$
16
$$

prediction vectors of dimension:

$$
10
$$

Conceptually:

    Position 1 → Prediction
    Position 2 → Prediction
    Position 3 → Prediction
    ...
    Position 16 → Prediction

The important point is:

$$
\boxed{
\text{One Forward Pass}
\rightarrow
\text{Many Predictions}
}
$$

---

## 14. Why Is This More Efficient?

Suppose there are $N$ sliding windows.

Naive method:

$$
N
\times
\text{CNN computation}
$$

Convolutional implementation instead performs:

$$
\text{Shared Feature Extraction}
+
\text{Spatial Prediction}
$$

The expensive convolutional features are computed once and reused.

The exact computational gain depends on architecture, image size, stride, and implementation, but the central reason for the improvement is always the same:

> **Overlapping windows reuse the same intermediate convolutional features instead of recomputing them independently.**

---

## 15. Backward Propagation

The forward process is:

$$
x
\rightarrow
h
\rightarrow
\hat{Y}
\rightarrow
L
$$

The backward process is:

$$
L
\rightarrow
\frac{\partial L}{\partial\hat{Y}}
\rightarrow
\frac{\partial L}{\partial h}
\rightarrow
\frac{\partial L}{\partial\theta}
$$

Suppose the output has many spatial positions.

The total loss can be decomposed as:

$$
L
=
\sum_{i,j}
L_{ij}
$$

Therefore:

$$
\frac{\partial L}{\partial\theta}
=
\sum_{i,j}
\frac{\partial L_{ij}}{\partial\theta}
$$

This means one shared convolutional parameter can receive gradient contributions from many spatial predictions.

~~~mermaid
flowchart BT
    A["Prediction 1 Loss"] --> E["Shared Convolution"]
    B["Prediction 2 Loss"] --> E
    C["Prediction 3 Loss"] --> E
    D["..."] --> E

    E --> F["Shared Parameters"]
~~~

This is closely related to the gradient aggregation idea from Multi-Task Learning.

The difference is:

- in MTL, different paths may correspond to different tasks;
- here, different paths may correspond to different **spatial locations**.

---

## 16. Numerical Example of Gradient Aggregation

Suppose one shared parameter is:

$$
w
$$

and it affects two spatial predictions.

The first prediction gives:

$$
\frac{\partial L_1}{\partial w}
=
0.3
$$

The second gives:

$$
\frac{\partial L_2}{\partial w}
=
0.5
$$

The total loss is:

$$
L=L_1+L_2
$$

Therefore:

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L_1}{\partial w}
+
\frac{\partial L_2}{\partial w}
$$

so:

$$
\frac{\partial L}{\partial w}
=
0.3+0.5
=
0.8
$$

The parameter update is:

$$
w^{\text{new}}
=
w^{\text{old}}
-
\eta(0.8)
$$

The important idea is:

> Multiple spatial predictions jointly train the same convolutional parameters.

---

## 17. A Deeper Tensor-Level View

Suppose a convolutional feature map is:

$$
A\in\mathbb{R}^{H\times W\times C}
$$

A convolution filter is:

$$
K\in\mathbb{R}^{k\times k\times C}
$$

The filter is applied at many spatial positions $(i,j)$.

Conceptually:

$$
Z_{i,j}
=
K\star A_{i,j}
$$

for many values of $i$ and $j$.

This means we do not need a separate feature tensor for every sliding window.

Instead, we have one feature map:

$$
A
$$

and convolution generates outputs for all relevant locations.

This is the tensor-level expression of **computation sharing**.

---

## 18. Fully Connected vs. Convolutional Form

The original classifier may look like:

~~~mermaid
flowchart LR
    A["Input"] --> B["Convolution"]
    B --> C["Pooling"]
    C --> D["Fully Connected"]
    D --> E["Prediction"]
~~~

After conversion:

~~~mermaid
flowchart LR
    A["Input"] --> B["Convolution"]
    B --> C["Pooling"]
    C --> D["Convolutional Head"]
    D --> E["Spatial Predictions"]
~~~

The important architectural transformation is:

$$
\text{Fully Connected Layer}
\rightarrow
\text{Convolutional Layer}
$$

This makes the network able to process inputs with larger spatial dimensions while preserving spatial output.

---

## 19. What the Converted Network Is Really Doing

The converted network is no longer producing only:

$$
\text{one prediction}
$$

It produces:

$$
\text{many spatial predictions}
$$

Conceptually:

    Full Image
         ↓
    Shared Features
         ↓
    ┌─────┬─────┬─────┐
    ↓     ↓     ↓     ↓
    P1    P2    P3    ...

Each $P_i$ corresponds to a different spatial region.

Thus:

$$
\text{Fully Convolutional Network}
=
\text{Spatially Distributed Predictor}
$$

This is a fundamental idea that appears repeatedly in modern Computer Vision.

---

## 20. Convolutional Implementation vs. YOLO

These two concepts should not be treated as identical.

### Convolutional Implementation of Sliding Windows

This is primarily a **computational technique**:

> Convert fully connected layers to convolutional layers so that predictions over many spatial locations can be computed together.

### YOLO

YOLO is an **object detection architecture**:

> It predicts object locations, objectness, and classes across the image using a convolutional detection framework.

Therefore:

$$
\text{Convolutional Implementation}
=
\text{Efficient Computation Idea}
$$

while:

$$
\text{YOLO}
=
\text{Detection Architecture}
$$

The former is one of the ideas that leads toward the latter.

---

## 21. What Problem Does Convolutional Implementation Solve?

It is important not to overstate what this technique solves.

It primarily addresses:

$$
\text{Repeated Feature Computation}
$$

It provides:

- shared convolutional computation;
- spatially distributed predictions;
- more efficient processing than naive repeated CNN inference.

It does **not** by itself solve all object-detection problems.

We still need to deal with:

- multiple objects;
- duplicate predictions;
- bounding-box overlap;
- different object shapes;
- final prediction selection.

These lead to concepts such as:

$$
\text{YOLO}
\rightarrow
\text{IoU}
\rightarrow
\text{Non-Max Suppression}
\rightarrow
\text{Anchor Boxes}
$$

---

## 22. Connection to YOLO

The conceptual progression is:

~~~mermaid
flowchart LR
    A["Sliding Window"] --> B["Repeated CNN Computation"]
    B --> C["Convolutional Implementation"]
    C --> D["Shared Spatial Computation"]
    D --> E["YOLO"]
~~~

The reasoning is:

### Sliding Window

> Search many locations.

### Computational Problem

> The same visual features are recomputed repeatedly.

### Convolutional Implementation

> Share feature extraction across locations.

### YOLO

> Build an efficient detector that directly predicts multiple objects across the image.

Therefore, the transition is:

$$
\boxed{
\text{Search Many Locations}
\rightarrow
\text{Share Computation Across Locations}
}
$$

---

## 23. Forward and Backward Mental Model

A complete computational view is:

~~~mermaid
flowchart TD
    A["Full Image"] --> B["Convolutional Feature Extractor"]
    B --> C["Spatial Feature Map"]
    C --> D["Convolutional Prediction Head"]
    D --> E["Many Spatial Predictions"]
    E --> F["Loss"]

    F --> G["Backward Through Prediction Head"]
    G --> H["Gradient Contributions from Many Locations"]
    H --> I["Shared Convolutional Parameters"]
~~~

Forward:

$$
x
\rightarrow
h
\rightarrow
\hat{Y}
\rightarrow
L
$$

Backward:

$$
L
\rightarrow
\frac{\partial L}{\partial\hat{Y}}
\rightarrow
\frac{\partial L}{\partial h}
\rightarrow
\frac{\partial L}{\partial\theta}
$$

The key property is:

$$
\frac{\partial L}{\partial\theta}
=
\sum_{i,j}
\frac{\partial L_{ij}}{\partial\theta}
$$

Thus, the same convolutional parameters are trained by many spatial prediction locations.

---

## 24. Core Mental Model

The entire concept can be summarized as a four-step chain.

### 24.1 Naive Sliding Window

$$
\text{Many Windows}
\rightarrow
\text{Many CNN Runs}
$$

### 24.2 Problem

$$
\text{Repeated Feature Computation}
$$

### 24.3 Convolutional Implementation

$$
\text{Full Image}
\rightarrow
\text{Shared CNN}
\rightarrow
\text{Many Spatial Predictions}
$$

### 24.4 Key Insight

$$
\boxed{
\text{Compute Features Once}
+
\text{Reuse Them Across Locations}
}
$$

The most important conceptual statement is:

> **Convolutional Implementation of Sliding Windows does not fundamentally change the idea of scanning the image. It changes the computation so that overlapping regions share convolutional features instead of recomputing them independently.**

This is the bridge from **naive sliding-window detection** to modern convolutional object detectors such as **YOLO**.

# Receptive Field

The **receptive field** of an output unit is the region of the original input image that can influence that output.

For example, if one final output unit has a receptive field of:

$$
14\times14
$$

then that output unit can be influenced by a $14\times14$ region of the input image.

This concept is especially important in **Convolutional Implementation of Sliding Windows**, because it explains why each spatial prediction can be interpreted as corresponding to a particular region of the original image.

## 1. Kernel Size vs. Receptive Field

These two concepts should be distinguished clearly.

A convolution may use a:

$$
5\times5
$$

kernel.

This means that one output unit of **that convolution layer** directly depends on a $5\times5$ region of its input feature map.

However, after several convolution and pooling layers, the final output can have a much larger receptive field.

For example:

$$
5\times5
\rightarrow
2\times2\text{ Pool}
\rightarrow
5\times5
$$

can produce a receptive field of:

$$
14\times14
$$

Therefore:

$$
\boxed{
\text{Kernel Size}
\neq
\text{Receptive Field}
}
$$

The kernel describes a local operation at one layer, while the receptive field describes the accumulated region of the **original input** seen by a deeper output unit.

## 2. Computing the Receptive Field

To compute the receptive field through a CNN, we track two quantities:

$$
r_l
$$

the receptive field size at layer $l$, and:

$$
j_l
$$

the **jump**, which is the distance between neighboring units at layer $l$ measured in terms of the original input.

Initially:

$$
r_0=1
$$

$$
j_0=1
$$

For a layer with kernel size $k_l$ and stride $s_l$:

$$
j_l
=
j_{l-1}s_l
$$

and:

$$
r_l
=
r_{l-1}
+
(k_l-1)j_{l-1}
$$

The receptive field grows because a unit at the current layer depends on multiple units from the previous layer, and each of those previous units already corresponds to a region of the original input.

## 3. Example from Convolutional Sliding Windows

Consider:

$$
16\times16\times3
$$

followed by:

$$
5\times5\text{ Conv},\quad s=1
$$

then:

$$
2\times2\text{ Pool},\quad s=2
$$

then:

$$
5\times5\text{ Conv},\quad s=1
$$

The receptive field evolves as:

$$
r_0=1
$$

After the first $5\times5$ convolution:

$$
r_1
=
1+(5-1)(1)
=
5
$$

After the $2\times2$ pooling:

$$
r_2
=
5+(2-1)(1)
=
6
$$

The jump becomes:

$$
j_2=2
$$

After the second $5\times5$ convolution:

$$
r_3
=
6+(5-1)(2)
=
14
$$

Therefore:

$$
\boxed{
r_3=14
}
$$

So each spatial output unit has a:

$$
14\times14
$$

receptive field on the original input.

## 4. Receptive Field, Output Size, and Jump

These three quantities answer different questions.

### Output Size

> How many output units are there?

For example:

$$
2\times2\times400
$$

contains:

$$
2\times2=4
$$

spatial output locations.

### Receptive Field

> How much of the original input can influence one output location?

For the example above:

$$
14\times14
$$

### Jump

> How far apart are neighboring output locations when mapped back to the original input?

For the example:

$$
j=2
$$

So neighboring predictions are shifted by 2 pixels in the original image.

This means:

$$
\text{Receptive Field}=14
$$

while:

$$
\text{Jump}=2
$$

Thus neighboring $14\times14$ receptive fields overlap heavily.

## 5. Can We Recover the Receptive Field from the Output Shape?

Not from the output shape alone.

For example:

$$
2\times2\times400
$$

only tells us:

- there are $2\times2$ spatial locations;
- there are 400 channels.

It does **not** tell us:

- kernel sizes;
- strides;
- pooling operations;
- padding;
- number of layers.

Different CNN architectures can produce the same output shape while having different receptive fields.

Therefore:

$$
\boxed{
\text{Output Shape}
\neq
\text{Receptive Field}
}
$$

However, if the architecture and its layer hyperparameters are known, we can calculate the receptive field from the output back to the input using:

$$
j_l
=
j_{l-1}s_l
$$

and:

$$
r_l
=
r_{l-1}
+
(k_l-1)j_{l-1}
$$

## 6. Connection to Convolutional Sliding Windows

In Andrew Ng's example, the final output has:

$$
2\times2
$$

spatial locations, and each location has a:

$$
14\times14
$$

receptive field.

Therefore, we can interpret the network as producing:

$$
2\times2=4
$$

predictions corresponding to four $14\times14$ regions of the input.

The important distinction is:

$$
\boxed{
14\times14
=
\text{Receptive Field}
}
$$

while:

$$
\boxed{
5\times5
=
\text{Convolution Kernel}
}
$$

The network does not explicitly crop four $14\times14$ windows and process them independently. The convolutional network computes the outputs simultaneously, and the receptive-field calculation tells us which region of the original image each output corresponds to.

The core mental model is:

$$
\boxed{
\text{Output Shape}
\rightarrow
\text{How Many Predictions}
}
$$

$$
\boxed{
\text{Receptive Field}
\rightarrow
\text{How Much Input Each Prediction Sees}
}
$$

$$
\boxed{
\text{Jump}
\rightarrow
\text{How Far Apart Neighboring Predictions Are}
}
$$

# Bounding Box Predictions

Bounding Box Prediction is the part of Object Detection that learns the **geometry of an object**: where the object is located and how large it is.

In Object Localization, we already introduced:

$$
[b_x,b_y,b_w,b_h]
$$

In Object Detection, the same idea is extended to **multiple spatial prediction locations**.

The key distinction is:

$$
\text{Objectness}
\rightarrow
\text{Does an object exist?}
$$

while:

$$
\text{Bounding Box}
\rightarrow
\text{Where is the object and how large is it?}
$$

---

## 1. What Is a Bounding Box Prediction?

A bounding box is represented using four parameters:

$$
b_x,\quad b_y,\quad b_w,\quad b_h
$$

where:

- $b_x$: x-coordinate of the box center;
- $b_y$: y-coordinate of the box center;
- $b_w$: box width;
- $b_h$: box height.

The network predicts:

$$
\hat b
=
[\hat b_x,\hat b_y,\hat b_w,\hat b_h]
$$

while the ground truth is:

$$
b
=
[b_x,b_y,b_w,b_h]
$$

Because these are continuous values, Bounding Box Prediction is fundamentally a **regression problem**.

---

## 2. Why Predict the Center and Size?

Instead of predicting the four box boundaries directly:

$$
x_{\min},x_{\max},y_{\min},y_{\max}
$$

we predict:

$$
b_x,b_y,b_w,b_h
$$

This gives a natural representation:

$$
\text{Bounding Box}
=
\text{Center}
+
\text{Size}
$$

The four boundaries can then be recovered as:

$$
x_{\min}
=
b_x-\frac{b_w}{2}
$$

$$
x_{\max}
=
b_x+\frac{b_w}{2}
$$

$$
y_{\min}
=
b_y-\frac{b_h}{2}
$$

$$
y_{\max}
=
b_y+\frac{b_h}{2}
$$

Thus the network only needs to learn four continuous values.

---

## 3. Bounding Box Prediction in Object Detection

In Object Localization, one object may be represented as:

$$
[
p_c,
b_x,
b_y,
b_w,
b_h,
c_1,\ldots,c_C
]
$$

In Object Detection, the same kind of prediction can appear at many spatial locations.

A simplified architecture is:

~~~mermaid
flowchart LR
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Spatial Feature Map"]
    C --> D["Detection Head"]

    D --> E["Location 1"]
    D --> F["Location 2"]
    D --> G["Location 3"]
    D --> H["..."]
~~~

At each location, the model can predict:

$$
[
p_c,
b_x,
b_y,
b_w,
b_h,
c_1,\ldots,c_C
]
$$

The bounding-box component is:

$$
[b_x,b_y,b_w,b_h]
$$

---

## 4. Grid Cells and Bounding Boxes

In a grid-based detector, the image is divided into spatial cells.

For example:

    ┌────┬────┬────┬────┐
    │    │    │    │    │
    ├────┼────┼────┼────┤
    │    │ ●  │    │    │
    ├────┼────┼────┼────┤
    │    │    │    │    │
    ├────┼────┼────┼────┤
    └────┴────┴────┴────┘

Suppose the center of an object lies inside one particular cell.

That cell can be responsible for predicting the object.

This gives two different roles:

$$
\text{Grid Cell}
\rightarrow
\text{Coarse Spatial Assignment}
$$

and:

$$
(b_x,b_y,b_w,b_h)
\rightarrow
\text{Fine Box Geometry}
$$

In other words:

> The cell determines **which spatial prediction is responsible**, while the bounding-box parameters determine **the exact geometry of the object**.

---

## 5. Bounding-Box Coordinates

Bounding-box parameters are commonly normalized relative to the image or to the relevant grid-cell formulation.

For example:

$$
0\le b_x\le1
$$

and:

$$
0\le b_y\le1
$$

This makes the representation less dependent on the absolute image resolution.

In a grid-based detector, the exact coordinate parameterization is especially important because the prediction may be expressed **relative to the responsible cell** rather than only in global image coordinates.

This becomes more important when we study YOLO.

---

## 6. Bounding Box Prediction as Regression

Suppose the ground truth is:

$$
b
=
[0.5,0.6,0.4,0.3]
$$

and the prediction is:

$$
\hat b
=
[0.55,0.58,0.45,0.28]
$$

The coordinate errors are:

$$
\Delta_x
=
0.55-0.5
=
0.05
$$

$$
\Delta_y
=
0.58-0.6
=
-0.02
$$

$$
\Delta_w
=
0.45-0.4
=
0.05
$$

$$
\Delta_h
=
0.28-0.3
=
-0.02
$$

A simple conceptual regression loss is:

$$
L_{\text{box}}
=
(\hat b_x-b_x)^2
+
(\hat b_y-b_y)^2
+
(\hat b_w-b_w)^2
+
(\hat b_h-b_h)^2
$$

The network is therefore trained to reduce the difference between the predicted box and the ground-truth box.

---

## 7. Interpreting the Four Parameters

Each parameter has a specific geometric role.

### 7.1 Center X

$$
b_x
$$

controls the horizontal position of the box center.

### 7.2 Center Y

$$
b_y
$$

controls the vertical position of the box center.

### 7.3 Width

$$
b_w
$$

controls the horizontal size of the box.

### 7.4 Height

$$
b_h
$$

controls the vertical size of the box.

Therefore:

$$
[b_x,b_y,b_w,b_h]
$$

can be understood as:

$$
\text{Center Position}
+
\text{Box Size}
$$

---

## 8. Forward Propagation

A simplified detector can be written as:

$$
h=f(x;\theta)
$$

followed by a detection head:

$$
\hat Y=g(h;\phi)
$$

The output is typically a spatial tensor:

$$
\hat Y
\in
\mathbb{R}^{H\times W\times K}
$$

At one spatial location, part of the output corresponds to:

$$
[\hat b_x,\hat b_y,\hat b_w,\hat b_h]
$$

and the complete prediction may contain:

$$
[
\hat p_c,
\hat b_x,
\hat b_y,
\hat b_w,
\hat b_h,
\hat c_1,\ldots,\hat c_C
]
$$

The overall computational flow is:

~~~mermaid
flowchart TD
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Spatial Feature Map"]
    C --> D["Detection Head"]

    D --> E["Objectness"]
    D --> F["Bounding Box"]
    D --> G["Class"]

    F --> H["bx, by, bw, bh"]
~~~

---

## 9. Bounding Box and Objectness

These two concepts must be clearly separated.

### 9.1 Objectness

$$
p_c
$$

answers:

> **Is there an object?**

### 9.2 Bounding Box

$$
b_x,b_y,b_w,b_h
$$

answers:

> **Where is the object and how large is it?**

Therefore:

$$
\text{Objectness}
\rightarrow
\text{Existence}
$$

while:

$$
\text{Bounding Box}
\rightarrow
\text{Geometry}
$$

A box prediction is meaningful only when there is actually an object associated with that prediction.

---

## 10. Backward Propagation

Suppose the bounding-box loss is:

$$
L_{\text{box}}
=
\sum_k(\hat b_k-b_k)^2
$$

For one coordinate:

$$
\frac{\partial L_{\text{box}}}{\partial\hat b_k}
=
2(\hat b_k-b_k)
$$

For example:

$$
\frac{\partial L}{\partial\hat b_x}
=
2(\hat b_x-b_x)
$$

If:

$$
\hat b_x>b_x
$$

the prediction is too far to the right.

If:

$$
\hat b_x<b_x
$$

the prediction is too far to the left.

The same reasoning applies to:

$$
b_y,\quad b_w,\quad b_h
$$

These gradients propagate backward through the detection head and into the CNN backbone.

~~~mermaid
flowchart BT
    A["Bounding Box Loss"] --> B["Bounding Box Prediction"]
    B --> C["Detection Head"]
    C --> D["CNN Feature Map"]
    D --> E["CNN Parameters"]
~~~

Thus, the CNN learns visual features that help determine the object's geometry.

---

## 11. Bounding Box and Receptive Field

This connects directly to the **Receptive Field** concept.

Suppose a spatial prediction location has a receptive field of:

$$
14\times14
$$

Then the features at that location are produced from a $14\times14$ region of the original image.

The detection head uses those features to predict:

$$
b_x,b_y,b_w,b_h
$$

The conceptual flow is:

$$
\text{Input Region}
\rightarrow
\text{Receptive Field}
\rightarrow
\text{Visual Features}
\rightarrow
\text{Bounding Box Prediction}
$$

Therefore:

> **Receptive Field tells us which region the network can see.**

while:

> **Bounding Box Prediction tells us where the object is and how large it is.**

These are different concepts, but they are directly connected.

---

## 12. Bounding Box Prediction and Sliding Windows

In naive sliding-window detection, we can think of the process as:

$$
14\times14\text{ Window}
\rightarrow
\text{CNN}
\rightarrow
\text{Bounding Box}
$$

After converting the network into a convolutional implementation:

$$
\text{Full Image}
\rightarrow
\text{Shared CNN}
\rightarrow
\text{Multiple Spatial Locations}
\rightarrow
\text{Multiple Bounding Boxes}
$$

Conceptually:

    Location 1 → Bounding Box 1
    Location 2 → Bounding Box 2
    Location 3 → Bounding Box 3
    ...

This is why the detection output becomes a spatial tensor instead of a single prediction vector.

---

## 13. Why Can Multiple Boxes Represent the Same Object?

Suppose there is only one car in the image.

Different spatial predictions may produce:

    Prediction 1
    ┌──────────────┐
    │     CAR      │
    └──────────────┘

    Prediction 2
    ┌────────────────┐
    │      CAR       │
    └────────────────┘

    Prediction 3
      ┌─────────────┐
      │     CAR     │
      └─────────────┘

These predictions may all refer to the same physical object.

Therefore, after producing bounding-box candidates, the detector needs a way to determine:

> **Which boxes represent the same object?**

This leads to:

$$
\text{IoU}
$$

and:

$$
\text{Non-Max Suppression}
$$

---

## 14. Bounding Boxes and Anchor Boxes

Another problem is that objects can have very different shapes.

For example:

    Tall Object

        ┌───┐
        │   │
        │   │
        │   │
        └───┘


    Wide Object

    ┌──────────────┐
    │              │
    └──────────────┘

A detector has to handle different sizes and aspect ratios.

Anchor Boxes introduce reference box shapes:

$$
\text{Anchor}
\rightarrow
\text{Box Refinement}
$$

The model then learns how to adjust the reference box to better match the actual object.

This is why understanding bounding-box geometry is essential before learning anchor boxes.

---

## 15. Bounding Box Prediction in the Overall Detection Pipeline

The complete conceptual pipeline is:

$$
\text{Image}
\rightarrow
\text{CNN}
\rightarrow
\text{Spatial Features}
\rightarrow
\text{Objectness}
+
\text{Bounding Box}
+
\text{Class}
$$

Then the candidate boxes are processed further:

$$
\text{Candidate Boxes}
\rightarrow
\text{IoU}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Boxes}
$$

Thus Bounding Box Prediction is the bridge between:

$$
\text{CNN Features}
$$

and:

$$
\text{Geometric Object Representation}
$$

---

## 16. Core Mental Model

Bounding Box Prediction is fundamentally:

$$
\text{Visual Features}
\rightarrow
\text{Object Geometry}
$$

The geometry is represented by:

$$
\boxed{
[b_x,b_y,b_w,b_h]
}
$$

which describes:

$$
\text{Center Position}
+
\text{Box Size}
$$

In Object Detection, these predictions are produced at multiple spatial locations.

The complete reasoning is:

$$
\text{Spatial Location}
\rightarrow
\text{Candidate Object}
\rightarrow
\text{Bounding Box}
$$

and then:

$$
\text{Candidate Boxes}
\rightarrow
\text{IoU}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}
$$

The key idea to remember is:

> **A spatial location determines where a prediction is made, while the bounding-box parameters determine the exact geometry of the object.**

# Intersection over Union (IoU)

**Intersection over Union (IoU)** is an important metric in Object Detection for measuring the **geometric overlap between two bounding boxes**.

It answers:

> **How well does one bounding box overlap with another bounding box?**

IoU is commonly used for:

- evaluating bounding-box localization;
- determining whether a detection sufficiently matches a ground-truth box;
- comparing candidate boxes;
- supporting Non-Max Suppression (NMS).

---

## 1. Intuition

Suppose we have two bounding boxes:

    ┌──────────────────────┐
    │                      │
    │   Ground Truth       │
    │                      │
    │        ┌─────────────┤
    │        │ Prediction  │
    │        │             │
    └────────┴─────────────┘

The two boxes overlap.

We want to measure:

> How large is the overlapping region compared with the total region covered by the two boxes?

This is exactly what IoU measures.

---

## 2. IoU Definition

Let:

$$
B_{\text{pred}}
$$

be the predicted bounding box and:

$$
B_{\text{gt}}
$$

be the ground-truth bounding box.

IoU is defined as:

$$
IoU
=
\frac{
\text{Area of Intersection}
}{
\text{Area of Union}
}
$$

In set notation:

$$
IoU
=
\frac{
|B_{\text{pred}}\cap B_{\text{gt}}|
}{
|B_{\text{pred}}\cup B_{\text{gt}}|
}
$$

The two key quantities are:

### 2.1 Intersection

The area shared by both boxes:

$$
B_{\text{pred}}\cap B_{\text{gt}}
$$

### 2.2 Union

The total area covered by at least one of the two boxes:

$$
B_{\text{pred}}\cup B_{\text{gt}}
$$

---

## 3. Why Divide by the Union?

Using only the intersection would not be enough.

Imagine the ground-truth box is small, while the predicted box is extremely large and completely contains it.

The intersection could still be large relative to the ground-truth box, but the prediction is clearly not a good localization.

By dividing by the union, IoU also penalizes boxes that are too large or poorly aligned.

Therefore:

$$
\text{IoU}
=
\frac{\text{Agreement Region}}{\text{Total Covered Region}}
$$

This gives a normalized measure of geometric agreement.

---

## 4. Range of IoU

IoU always satisfies:

$$
0\le IoU\le1
$$

### 4.1 Perfect overlap

If the two boxes are identical:

$$
IoU=1
$$

### 4.2 No overlap

If the boxes are completely separated:

$$
IoU=0
$$

### 4.3 Partial overlap

If the boxes overlap partially:

$$
0<IoU<1
$$

Therefore:

> **The larger the IoU, the more similar the two bounding boxes are geometrically.**

---

## 5. Simple Numerical Example

Suppose:

$$
\text{Intersection Area}=40
$$

and:

$$
\text{Union Area}=100
$$

Then:

$$
IoU
=
\frac{40}{100}
=
0.4
$$

So the two boxes have:

$$
IoU=0.4
$$

This means their intersection occupies $40\%$ of the union area.

It does **not** mean that the model is "40\% accurate."

IoU is specifically a **geometric overlap measure**.

---

## 6. Computing IoU from Bounding-Box Coordinates

Suppose a box is represented by:

$$
(x_1,y_1,x_2,y_2)
$$

where:

- $(x_1,y_1)$ is the top-left corner;
- $(x_2,y_2)$ is the bottom-right corner.

Let:

$$
B_1=
(x_1^{(1)},y_1^{(1)},x_2^{(1)},y_2^{(1)})
$$

and:

$$
B_2=
(x_1^{(2)},y_1^{(2)},x_2^{(2)},y_2^{(2)})
$$

We first compute the intersection.

The top-left corner of the intersection is:

$$
x_{\text{left}}
=
\max(x_1^{(1)},x_1^{(2)})
$$

$$
y_{\text{top}}
=
\max(y_1^{(1)},y_1^{(2)})
$$

The bottom-right corner is:

$$
x_{\text{right}}
=
\min(x_2^{(1)},x_2^{(2)})
$$

$$
y_{\text{bottom}}
=
\min(y_2^{(1)},y_2^{(2)})
$$

Therefore:

$$
w_I
=
\max(0,x_{\text{right}}-x_{\text{left}})
$$

and:

$$
h_I
=
\max(0,y_{\text{bottom}}-y_{\text{top}})
$$

The intersection area is:

$$
A_I=w_Ih_I
$$

---

## 7. Union Area

Suppose:

$$
A_1
$$

and:

$$
A_2
$$

are the areas of the two boxes.

Then:

$$
A_U
=
A_1+A_2-A_I
$$

Why do we subtract the intersection?

Because when we calculate:

$$
A_1+A_2
$$

the overlapping region is counted twice.

So:

$$
\text{Union}
=
\text{Area}_1
+
\text{Area}_2
-
\text{Intersection}
$$

Finally:

$$
IoU
=
\frac{A_I}{A_U}
$$

---

## 8. Worked Example

Consider:

$$
B_1=(0,0,4,4)
$$

and:

$$
B_2=(2,2,6,6)
$$

Each box has:

$$
A_1=4\times4=16
$$

and:

$$
A_2=4\times4=16
$$

The intersection has width:

$$
w_I=4-2=2
$$

and height:

$$
h_I=4-2=2
$$

Therefore:

$$
A_I=2\times2=4
$$

The union area is:

$$
A_U
=
16+16-4
=
28
$$

Thus:

$$
IoU
=
\frac{4}{28}
\approx0.143
$$

The overlap is therefore relatively small.

---

## 9. IoU as an Evaluation Metric

Suppose the model predicts:

$$
B_{\text{pred}}
$$

and the ground truth is:

$$
B_{\text{gt}}
$$

We calculate:

$$
IoU
=
IoU(B_{\text{pred}},B_{\text{gt}})
$$

A high IoU indicates that the predicted box is well aligned with the ground truth.

A low IoU indicates poor localization.

For example:

$$
IoU=0.9
$$

indicates very strong overlap, while:

$$
IoU=0.2
$$

indicates weak overlap.

---

## 10. IoU Threshold

In Object Detection, we often need to decide whether a prediction is sufficiently close to the ground truth.

We can introduce a threshold:

$$
IoU\ge\tau
$$

where $\tau$ is a chosen threshold.

For example:

$$
\tau=0.5
$$

gives the rule:

$$
IoU\ge0.5
$$

The prediction satisfies the chosen localization criterion.

An important distinction is:

> **IoU is the metric. The threshold is a decision rule applied to that metric.**

The threshold itself is not part of the definition of IoU.

---

## 11. IoU Does Not Measure Classification

Suppose the ground-truth box contains a car, but the model predicts the class as person.

If the two boxes are identical:

$$
IoU=1
$$

The localization is perfect, but the class prediction is wrong.

Therefore, IoU only evaluates the **geometry of the boxes**.

It does not evaluate:

- object class;
- semantic correctness;
- confidence.

This connects to the decomposition:

$$
\text{Detection}
=
\text{Objectness}
+
\text{Geometry}
+
\text{Semantics}
$$

where:

$$
IoU
\rightarrow
\text{Geometry}
$$

---

## 12. IoU and Non-Max Suppression

IoU is also crucial for **Non-Max Suppression (NMS)**.

Suppose the detector produces several predictions:

    Car — confidence 0.92
    Car — confidence 0.87
    Car — confidence 0.74

and the corresponding boxes overlap heavily.

These may all represent the same physical car.

We can compare their boxes using IoU.

If:

$$
IoU(B_1,B_2)
$$

is very high, the two predictions are likely describing the same object.

NMS then uses confidence and IoU to decide which prediction should be kept.

---

## 13. IoU vs. Non-Max Suppression

These concepts are related but fundamentally different.

### 13.1 IoU

IoU is a **measurement**:

$$
IoU(A,B)
$$

It answers:

> How much do these two boxes overlap?

### 13.2 Non-Max Suppression

NMS is an **algorithm** that uses confidence scores and IoU to decide:

> Which predictions should remain?

Therefore:

$$
\text{IoU}
=
\text{Overlap Measurement}
$$

while:

$$
\text{NMS}
=
\text{Selection and Suppression}
$$

IoU is one of the tools used by NMS.

---

## 14. IoU in the Detection Pipeline

The overall process can be viewed as:

~~~mermaid
flowchart LR
    A["CNN"] --> B["Candidate Bounding Boxes"]
    B --> C["Confidence Filtering"]
    C --> D["IoU"]
    D --> E["Non-Max Suppression"]
    E --> F["Final Detections"]
~~~

IoU therefore appears after the network has generated candidate boxes and helps the system reason about their geometric overlap.

---

## 15. IoU in Evaluation vs. Post-Processing

IoU can be used in two different contexts.

### 15.1 Evaluation

Compare:

$$
B_{\text{pred}}
$$

with:

$$
B_{\text{gt}}
$$

to measure localization quality.

### 15.2 Post-Processing

Compare candidate predictions:

$$
B_1,B_2,\ldots,B_N
$$

to identify highly overlapping predictions during NMS.

The same metric is used for two different purposes:

$$
\text{IoU}
\rightarrow
\begin{cases}
\text{Localization Evaluation} \\
\text{Prediction Overlap for NMS}
\end{cases}
$$

---

## 16. Core Mental Model

The most important equation is:

$$
IoU
=
\frac{
\text{Intersection Area}
}{
\text{Union Area}
}
$$

with:

$$
0\le IoU\le1
$$

Interpretation:

- $0$ → no overlap;
- $1$ → perfect overlap;
- larger IoU → better geometric agreement.

The key mental model is:

$$
\boxed{
\text{IoU}
=
\text{Geometric Overlap Measure}
}
$$

not classification accuracy.

In Object Detection:

$$
\text{Bounding Box Prediction}
\rightarrow
\text{IoU}
\rightarrow
\text{Evaluate or Compare Boxes}
$$

and for duplicate predictions:

$$
\text{Candidate Boxes}
\rightarrow
\text{IoU}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}
$$

# Non-Max Suppression

**Non-Max Suppression (NMS)** is a post-processing algorithm used in Object Detection to remove redundant bounding-box predictions.

The core problem is:

> **One physical object can generate multiple candidate detections.**

For example, one car may produce several highly overlapping boxes:

    Box A — confidence = 0.95
    Box B — confidence = 0.88
    Box C — confidence = 0.72

These boxes may all correspond to the same car.

Therefore, the goal of NMS is:

$$
\boxed{
\text{Many Candidate Boxes}
\rightarrow
\text{Few Final Detections}
}
$$

More specifically:

$$
\boxed{
\text{Keep the strongest prediction}
+
\text{Suppress redundant predictions}
}
$$

NMS does not improve the predictions produced by the neural network. It decides **which candidate predictions should remain in the final output**.

---

## 1. Why Does NMS Need to Exist?

A convolutional detector produces predictions at many spatial locations:

$$
\text{Image}
\rightarrow
\text{CNN}
\rightarrow
\text{Candidate Predictions}
$$

Because multiple spatial locations can detect the same object, several predictions may overlap strongly.

This creates an important distinction:

$$
\boxed{
\text{Candidate Generation}
\neq
\text{Final Detection}
}
$$

The CNN generates candidate detections, while NMS converts these candidates into a cleaner set of final detections.

### 1.1 Confidence Score

Each candidate prediction has a confidence score, such as:

$$
p_c
$$

A higher confidence means the detector has stronger evidence that the candidate corresponds to a real object.

### 1.2 IoU

IoU measures the geometric overlap between two bounding boxes:

$$
IoU(A,B)
=
\frac{
\text{Area of Intersection}
}{
\text{Area of Union}
}
$$

A high IoU means the two boxes overlap strongly.

NMS therefore uses two different pieces of information:

$$
\boxed{
\text{Confidence}
\rightarrow
\text{Which box should be kept?}
}
$$

and:

$$
\boxed{
\text{IoU}
\rightarrow
\text{Which other boxes are redundant?}
}
$$

---

## 2. The NMS Algorithm

After generating candidate predictions, we usually perform a confidence filtering step first.

If:

$$
p_c\le\tau_{\text{conf}}
$$

the prediction can be discarded.

For example:

$$
\tau_{\text{conf}}=0.6
$$

means very weak predictions are removed before NMS.

This gives:

$$
\text{Many Candidates}
\rightarrow
\text{Confidence Filtering}
\rightarrow
\text{Fewer Candidates}
$$

The confidence threshold and NMS threshold have different purposes:

$$
\boxed{
\tau_{\text{conf}}
\rightarrow
\text{Is this prediction trustworthy enough?}
}
$$

while:

$$
\boxed{
\tau_{\text{NMS}}
\rightarrow
\text{Is this prediction redundant with a selected box?}
}
$$

### 2.1 Core Procedure

Suppose the remaining candidates are:

$$
B=\{B_1,B_2,\ldots,B_N\}
$$

We maintain two sets:

$$
\text{Candidates}
$$

and:

$$
\text{Final}
$$

Initially:

$$
\text{Final}=\varnothing
$$

NMS repeatedly performs:

1. Select the remaining box with the highest confidence.
2. Move that box to `Final`.
3. Compute its IoU with all remaining candidate boxes.
4. Suppress candidates whose IoU satisfies:

$$
IoU\ge\tau_{\text{NMS}}
$$

5. Repeat while candidates remain.

The crucial point is:

> **Once a box is selected, it is removed from the candidate set.**

Therefore, the next iteration selects the highest-confidence box among the **remaining candidates**, not the same box again.

Conceptually:

$$
\boxed{
\text{Pick}
\rightarrow
\text{Keep}
\rightarrow
\text{Suppress Redundancy}
\rightarrow
\text{Repeat}
}
$$

### 2.2 Example

Suppose after confidence filtering we have:

| Box | Confidence |
| --- | ---: |
| $A$ | 0.95 |
| $B$ | 0.90 |
| $C$ | 0.85 |
| $D$ | 0.80 |

Set:

$$
\tau_{\text{NMS}}=0.5
$$

and suppose:

$$
IoU(A,B)=0.80
$$

$$
IoU(A,C)=0.20
$$

$$
IoU(A,D)=0.10
$$

$$
IoU(C,D)=0.70
$$

#### First iteration

$A$ has the highest confidence:

$$
A=0.95
$$

So:

$$
\text{Final}=\{A\}
$$

$B$ is suppressed because:

$$
IoU(A,B)=0.80>0.5
$$

while $C$ and $D$ remain candidates because:

$$
IoU(A,C)=0.20<0.5
$$

and:

$$
IoU(A,D)=0.10<0.5
$$

Therefore:

$$
\text{Candidates}=\{C,D\}
$$

and:

$$
\text{Final}=\{A\}
$$

#### Second iteration

The algorithm now chooses the highest-confidence **remaining** candidate:

$$
C=0.85
$$

It moves $C$ into Final:

$$
\text{Final}=\{A,C\}
$$

Then:

$$
IoU(C,D)=0.70>0.5
$$

so $D$ is suppressed.

The final result is:

$$
\boxed{
\text{Final}=\{A,C\}
}
$$

The important idea is that the second iteration does not revisit $A`. It processes the remaining candidate set.

Even if an iteration suppresses nothing, that iteration is still necessary because the selected box has been confirmed as a final detection and removed from the candidate set.

---

## 3. Why Is NMS Applied Per Class?

In a multi-class detector, NMS is typically applied **separately for each class**.

Suppose the detector produces:

    Car:
        Box A — 0.95
        Box B — 0.88
        Box C — 0.75

    Person:
        Box D — 0.92
        Box E — 0.81

We conceptually split the predictions:

~~~mermaid
flowchart TD
    A["All Predictions"] --> B["Separate by Class"]

    B --> C["Car Predictions"]
    B --> D["Person Predictions"]
    B --> E["Motorcycle Predictions"]

    C --> F["NMS for Car"]
    D --> G["NMS for Person"]
    E --> H["NMS for Motorcycle"]

    F --> I["Final Detections"]
    G --> I
    H --> I
~~~

Then we perform:

$$
\text{NMS(Car boxes)}
$$

separately from:

$$
\text{NMS(Person boxes)}
$$

and:

$$
\text{NMS(Motorcycle boxes)}
$$

We should not blindly compare a Car box with a Person box and suppress one simply because their IoU is high.

Two different classes can overlap spatially while still representing two different physical objects.

Therefore:

> **Class-wise NMS removes redundant predictions within the same class while preserving potentially different objects from different classes.**

---

## 4. Why Does the Iterative Process Matter?

The purpose of the loop is not simply to suppress boxes.

Its deeper purpose is to **construct the final set of detections one object at a time**.

At each iteration:

$$
\text{Highest-confidence remaining box}
\rightarrow
\text{Final Detection}
$$

and then:

$$
\text{Remove its redundant neighbors}
$$

After that, the algorithm asks:

> **What is the strongest object prediction that has not already been processed or suppressed?**

This continues until:

$$
\text{Candidates}=\varnothing
$$

Therefore:

$$
\boxed{
\text{NMS}
=
\text{Greedy Selection}
+
\text{Overlap-Based Suppression}
}
$$

The word **greedy** means that the algorithm always selects the currently highest-confidence remaining candidate.

---

## 5. Role of NMS in the Object Detection Pipeline

The complete inference pipeline can be viewed as:

~~~mermaid
flowchart LR
    A["Input Image"] --> B["CNN"]
    B --> C["Candidate Predictions"]
    C --> D["Confidence Filtering"]
    D --> E["Separate by Class"]
    E --> F["Non-Max Suppression"]
    F --> G["Final Detections"]
~~~

Each stage has a different responsibility:

$$
\text{CNN}
\rightarrow
\text{Generate Candidates}
$$

$$
\text{Confidence Filtering}
\rightarrow
\text{Remove Very Weak Predictions}
$$

$$
\text{Class-wise NMS}
\rightarrow
\text{Remove Redundant Predictions}
$$

Therefore, NMS is a **post-processing step**, not another convolutional layer.

The neural network answers:

> **What could be an object, where is it, and what class might it belong to?**

NMS answers:

> **Which of these candidate predictions should appear in the final detection result?**

---

## 6. Core Mental Model

The entire idea can be summarized as:

$$
\boxed{
\text{Candidate Boxes}
\rightarrow
\text{Confidence Filtering}
\rightarrow
\text{Class-wise NMS}
\rightarrow
\text{Final Detections}
}
$$

The core logic of NMS is:

$$
\boxed{
\text{Pick Highest Confidence}
\rightarrow
\text{Keep}
\rightarrow
\text{Suppress High-IoU Duplicates}
\rightarrow
\text{Repeat}
}
$$

And the most important distinction is:

$$
\boxed{
\text{IoU}
=
\text{Overlap Measurement}
}
$$

while:

$$
\boxed{
\text{NMS}
=
\text{Greedy Selection + Redundancy Removal}
}
$$

The key mental model to keep is:

> **The detector generates many candidate boxes. NMS does not detect new objects; it selects strong representatives from those candidates and removes redundant predictions, typically within each class.**

# Anchor Boxes

**Anchor Boxes** are a technique used in Object Detection to address an important limitation of simple grid-based detection:

> **A single grid cell may need to represent multiple objects with different shapes.**

Without anchor boxes, we can think of:

$$
\text{Grid Cell}
\rightarrow
\text{One Prediction}
$$

If two objects have their centers inside the same cell, one prediction slot may not be enough.

Anchor Boxes introduce multiple **prediction slots** for each grid cell, together with different reference box shapes.

The key idea is:

$$
\boxed{
\text{Grid Cell}
+
\text{Anchor}
\rightarrow
\text{Prediction Slot}
}
$$

---

## 1. Why Do We Need Anchor Boxes?

In a simple grid-based detector, the center of an object determines which cell is responsible for predicting it.

For example:

    ┌────┬────┬────┐
    │    │    │    │
    ├────┼────┼────┤
    │    │ ●  │    │
    ├────┼────┼────┤
    │    │    │    │
    └────┴────┴────┘

If one object has its center in the middle cell, that cell can produce a prediction such as:

$$
[p_c,b_x,b_y,b_w,b_h,c_1,\ldots,c_C]
$$

The problem appears when two objects have centers inside the same cell:

    ┌──────────────────────┐
    │      ●               │
    │   Object 1           │
    │                      │
    │             ●        │
    │          Object 2    │
    └──────────────────────┘

A single prediction vector is not enough to represent both objects.

Therefore, we want:

$$
1\text{ cell}
\rightarrow
\text{multiple prediction slots}
$$

Anchor boxes provide these additional slots.

---

## 2. What Is an Anchor Box?

An **anchor box** is a predefined reference box shape that acts as a template for predicting a bounding box.

For example:

    Anchor 1: Wide

    ┌────────────────────┐
    │                    │
    └────────────────────┘


    Anchor 2: Tall

        ┌──────┐
        │      │
        │      │
        │      │
        └──────┘

The network does not simply output these anchor boxes as the final detections.

Instead:

$$
\text{Anchor}
\rightarrow
\text{Box Refinement}
\rightarrow
\text{Predicted Bounding Box}
$$

Therefore:

$$
\boxed{
\text{Anchor Box}
=
\text{Reference Geometry}
}
$$

while:

$$
\boxed{
\text{Bounding Box Prediction}
=
\text{Refined Geometry}
}
$$

---

## 3. Multiple Prediction Slots per Cell

Suppose each cell has two anchors.

Then:

$$
\text{Cell}
\rightarrow
\begin{cases}
\text{Anchor 1}\\
\text{Anchor 2}
\end{cases}
$$

Instead of:

$$
1\text{ cell}
\rightarrow
1\text{ prediction}
$$

we now have:

$$
1\text{ cell}
\rightarrow
2\text{ prediction slots}
$$

This allows the same cell to represent two different objects.

For example:

    Cell
        │
        ├── Anchor 1 → Object A
        │
        └── Anchor 2 → Object B

This is one of the main reasons Anchor Boxes are useful.

---

## 4. Anchor Boxes and Different Object Shapes

Anchor boxes are also useful because objects can have very different aspect ratios.

For example:

    Wide object

    ┌──────────────────────┐
    │                      │
    └──────────────────────┘


    Tall object

        ┌──────┐
        │      │
        │      │
        │      │
        └──────┘

Suppose we define:

$$
a_1=\text{wide anchor}
$$

and:

$$
a_2=\text{tall anchor}
$$

A wide object can be represented more naturally using $a_1$, while a tall object can be represented using $a_2$.

This gives:

$$
\boxed{
\text{Anchor Boxes}
\rightarrow
\text{Multiple Prediction Slots}
+
\text{Reference Geometries}
}
$$

The model can then refine each anchor to match the actual object.

---

## 5. What Does One Anchor Predict?

Suppose there are $C=3$ classes.

For one anchor, a prediction can contain:

$$
[
p_c,
b_x,
b_y,
b_w,
b_h,
c_1,c_2,c_3
]
$$

The number of values is:

$$
5+C
$$

so:

$$
5+3=8
$$

values per anchor.

If a cell has $B=2$ anchors, then that cell has:

$$
2\times8=16
$$

values.

Therefore, with an $S\times S$ grid, the output can have shape:

$$
\boxed{
S\times S\times B(5+C)
}
$$

For example:

$$
S=3,\quad B=2,\quad C=3
$$

gives:

$$
3\times3\times16
$$

---

## 6. How to Interpret the Output Tensor

The value:

$$
16
$$

in:

$$
3\times3\times16
$$

does **not** mean 16 classes.

Instead:

$$
16
=
2\text{ anchors}
\times
8\text{ values per anchor}
$$

At one cell:

$$
[
\underbrace{
p_c^{(1)},b_x^{(1)},b_y^{(1)},b_w^{(1)},b_h^{(1)},c_1^{(1)},c_2^{(1)},c_3^{(1)}
}_{\text{Anchor 1}}
\mid
\underbrace{
p_c^{(2)},b_x^{(2)},b_y^{(2)},b_w^{(2)},b_h^{(2)},c_1^{(2)},c_2^{(2)},c_3^{(2)}
}_{\text{Anchor 2}}
]
$$

So:

$$
\boxed{
\text{Anchor Dimension}
=
\text{Number of Prediction Slots}
}
$$

and:

$$
\boxed{
\text{Class Dimension}
=
\text{Number of Classes}
}
$$

These are completely different concepts.

---

## 7. Anchor Assignment

Now suppose we have a ground-truth bounding box:

$$
b_{\text{gt}}
$$

and several anchor shapes:

$$
a_1,a_2,\ldots,a_B
$$

We want to determine which anchor is most suitable for that object.

A common geometric criterion is IoU.

For example:

$$
IoU(b_{\text{gt}},a_1)=0.30
$$

while:

$$
IoU(b_{\text{gt}},a_2)=0.80
$$

Then anchor 2 is a better geometric match.

Conceptually:

$$
\boxed{
\text{Ground Truth Box}
\rightarrow
\text{Compare with Anchors}
\rightarrow
\text{Choose Best-Matching Anchor}
}
$$

If the object center is in cell $(i,j)$ and anchor 2 is selected, the object can be assigned to:

$$
(i,j,\text{anchor 2})
$$

Thus the object is identified by:

$$
\boxed{
\text{Cell Location}
+
\text{Anchor}
}
$$

rather than only by the cell location.

---

## 8. Example: Two Objects in One Cell

Suppose a single cell contains:

- one wide object;
- one tall object.

We have:

$$
a_1=\text{wide anchor}
$$

and:

$$
a_2=\text{tall anchor}
$$

Suppose:

$$
IoU(\text{Object A},a_1)
>
IoU(\text{Object A},a_2)
$$

and:

$$
IoU(\text{Object B},a_2)
>
IoU(\text{Object B},a_1)
$$

Then we can assign:

$$
\text{Object A}
\rightarrow
a_1
$$

and:

$$
\text{Object B}
\rightarrow
a_2
$$

So the same cell can represent:

    Cell
       │
       ├── Anchor 1 → Object A
       │
       └── Anchor 2 → Object B

This is the main conceptual advantage over having only one prediction per cell.

---

## 9. Anchor Boxes and Bounding Box Regression

An anchor is not the final bounding box.

Instead, the network learns how to adjust the anchor:

$$
\text{Anchor}
\rightarrow
(\Delta x,\Delta y,\Delta w,\Delta h)
\rightarrow
\text{Predicted Box}
$$

At a conceptual level:

$$
\text{Reference Shape}
+
\text{Learned Offsets}
=
\text{Final Box}
$$

Therefore, Anchor Boxes make the regression problem easier to organize:

> Rather than predicting every possible box shape from scratch, the model starts from a set of reference geometries and learns how to refine them.

---

## 10. Connection to IoU

IoU appears in two different places around Anchor Boxes.

### 10.1 Anchor Assignment

We may use IoU to determine:

> Which anchor is the best geometric match for this ground-truth box?

$$
\text{Ground Truth}
\rightarrow
\text{IoU with Anchors}
\rightarrow
\text{Best Anchor}
$$

### 10.2 Non-Max Suppression

After prediction, IoU is also used for NMS:

$$
\text{Candidate Boxes}
\rightarrow
\text{IoU}
\rightarrow
\text{NMS}
$$

These are different purposes.

Therefore:

$$
\boxed{
\text{IoU for Anchor Assignment}
\neq
\text{IoU for NMS}
}
$$

The same metric is used, but for different decisions.

---

## 11. Forward Propagation

Suppose:

$$
h=f(x;\theta)
$$

is the shared CNN representation.

The detection head produces:

$$
\hat Y
\in
\mathbb{R}^{S\times S\times B(5+C)}
$$

For example:

$$
S=3,\quad B=2,\quad C=3
$$

gives:

$$
\hat Y
\in
\mathbb{R}^{3\times3\times16}
$$

The conceptual flow is:

```mermaid
flowchart TD
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Spatial Feature Map"]
    C --> D["Detection Head"]

    D --> E["Grid Cell"]
    E --> F["Anchor 1 Prediction"]
    E --> G["Anchor 2 Prediction"]
```

Each anchor produces its own:

$$
[p_c,b_x,b_y,b_w,b_h,c_1,\ldots,c_C]
$$

prediction.

---

## 12. Why Anchor Boxes Are Useful

Anchor Boxes solve an important limitation of simple grid-based detection.

Without anchors:

$$
1\text{ Cell}
\rightarrow
1\text{ Prediction}
$$

With $B$ anchors:

$$
1\text{ Cell}
\rightarrow
B\text{ Prediction Slots}
$$

Therefore:

$$
\boxed{
\text{Anchor Boxes}
\rightarrow
\text{Multiple Objects per Cell}
}
$$

and they also provide:

$$
\boxed{
\text{Useful Reference Geometries}
}
$$

for objects with different aspect ratios.

The important distinction is:

> **An anchor is not an object and is not a final bounding box. It is a reference geometry and prediction slot that the network can refine into a final bounding box.**

---

## 13. Anchor Boxes in the Overall Detection Pipeline

The concepts learned so far now fit together naturally:

```mermaid
flowchart LR
    A["Input Image"] --> B["CNN"]
    B --> C["Spatial Grid"]

    C --> D["Anchor-based Predictions"]
    D --> E["Bounding Box Regression"]
    E --> F["Candidate Boxes"]
    F --> G["Confidence Filtering"]
    G --> H["Class-wise NMS"]
    H --> I["Final Detections"]
```

Their roles are:

$$
\text{Grid}
\rightarrow
\text{Where is the prediction made?}
$$

$$
\text{Anchor}
\rightarrow
\text{Which prediction slot and reference geometry?}
$$

$$
(b_x,b_y,b_w,b_h)
\rightarrow
\text{How should the anchor be refined?}
$$

$$
\text{Class}
\rightarrow
\text{What object is it?}
$$

$$
\text{NMS}
\rightarrow
\text{Which candidate predictions are redundant?}
$$

---

## 14. Core Mental Model

The most useful way to remember Anchor Boxes is:

$$
\boxed{
\text{Grid Cell}
+
\text{Anchor}
+
\text{Bounding Box Regression}
\rightarrow
\text{Object Detection}
}
$$

For $S\times S$ grid cells, $B$ anchors per cell, and $C$ classes:

$$
\boxed{
\text{Output Shape}
=
S\times S\times B(5+C)
}
$$

For example:

$$
S=3,\quad B=2,\quad C=3
$$

gives:

$$
3\times3\times16
$$

where:

$$
16
=
2\times(5+3)
$$

and **not** 16 classes.

The overall progression is:

$$
\text{Grid}
\rightarrow
\text{Anchor Boxes}
\rightarrow
\text{Bounding Box Regression}
\rightarrow
\text{IoU}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}
$$

The key idea to retain is:

> **Anchor Boxes provide multiple prediction slots and reference box shapes for each grid cell, allowing the detector to represent multiple objects in the same cell and handle different object geometries more effectively.**

# YOLO Algorithm

**YOLO (You Only Look Once)** is an important step in Object Detection because it brings together most of the concepts we have studied so far.

The core idea is:

$$
\boxed{
\text{One Image}
\rightarrow
\text{One CNN Forward Pass}
\rightarrow
\text{Many Candidate Detections}
\rightarrow
\text{Post-Processing}
\rightarrow
\text{Final Detections}
}
$$

YOLO is not just "a CNN for object detection". It is a way of organizing the entire detection problem into a **single-stage, spatial prediction problem**.

However, YOLO is a **family of detectors**, so different versions use different:

- output representations;
- anchor mechanisms;
- assignment strategies;
- box parameterizations;
- loss functions.

Therefore, we first study the common YOLO idea, then examine the evolution of the loss and training mechanisms.

---

## 1. From Object Localization to YOLO

Object Localization asks:

$$
\text{What?}
+
\text{Where?}
$$

Object Detection adds:

$$
\text{How Many?}
$$

A detector therefore needs to produce:

$$
\{
\text{Object}_1,
\text{Object}_2,
\ldots,
\text{Object}_N
\}
$$

where $N$ is not known in advance.

The progression we have studied is:

$$
\text{Object Localization}
$$

$$
\downarrow
$$

$$
\text{Sliding Window}
$$

$$
\downarrow
$$

$$
\text{Convolutional Sliding Window}
$$

$$
\downarrow
$$

$$
\text{Bounding Box Prediction}
$$

$$
\downarrow
$$

$$
\text{IoU}
$$

$$
\downarrow
$$

$$
\text{NMS}
$$

$$
\downarrow
$$

$$
\text{Anchor Boxes}
$$

$$
\downarrow
$$

$$
\boxed{\text{YOLO}}
$$

The important idea is that these are not isolated concepts. Each one solves a problem introduced by the previous stage.

---

## 2. Core Idea of YOLO

YOLO divides the image into spatial regions and predicts objects from those regions.

Conceptually:

```mermaid
flowchart LR
    A["Input Image"] --> B["CNN"]
    B --> C["Spatial Predictions"]
    C --> D["Candidate Objects"]
    D --> E["Post-Processing"]
    E --> F["Final Detections"]
```

The fundamental difference from naive sliding windows is:

$$
\boxed{
\text{Full Image}
\rightarrow
\text{One CNN Forward Pass}
\rightarrow
\text{Many Predictions}
}
$$

instead of:

$$
\text{Window}_1\rightarrow CNN
$$

$$
\text{Window}_2\rightarrow CNN
$$

$$
\text{Window}_3\rightarrow CNN
$$

and so on.

---

## 3. Grid Representation

Suppose the image is divided into:

$$
S\times S
$$

grid cells.

In the classical formulation from the course, an example is:

$$
S=19
$$

so the image contains:

$$
19\times19=361
$$

spatial cells.

The exact grid size is architecture-dependent, but the conceptual structure is:

```text
┌────┬────┬────┬────┐
│    │    │    │    │
├────┼────┼────┼────┤
│    │    │    │    │
├────┼────┼────┼────┤
│    │    │    │    │
├────┼────┼────┼────┤
│    │    │    │    │
└────┴────┴────┴────┘
```

The grid provides the spatial structure for the detector.

---

## 4. Which Grid Cell Is Responsible for an Object?

A key rule in the classical YOLO formulation is:

> **The grid cell containing the center of an object is responsible for predicting that object.**

Suppose:

```text
┌──────────┬──────────┬──────────┐
│          │          │          │
│          │    ●     │          │
│          │          │          │
└──────────┴──────────┴──────────┘
```

If the object center is inside the middle cell, then:

$$
\text{Middle Cell}
\rightarrow
\text{Object Prediction}
$$

Thus:

$$
\boxed{
\text{Object Center}
\rightarrow
\text{Responsible Grid Cell}
}
$$

This is the **coarse spatial assignment**.

The bounding-box parameters then provide the fine geometric information.

---

## 5. What Does One YOLO Prediction Contain?

For a problem with $C$ classes, one prediction can conceptually contain:

$$
[
p_c,
b_x,
b_y,
b_w,
b_h,
c_1,\ldots,c_C
]
$$

where:

### Objectness

$$
p_c
$$

answers:

> **Is there an object associated with this prediction?**

### Bounding Box

$$
b_x,b_y,b_w,b_h
$$

answers:

> **Where is the object and how large is it?**

### Classification

$$
c_1,\ldots,c_C
$$

answers:

> **What class is the object?**

Therefore:

$$
\boxed{
\text{Prediction}
=
\text{Objectness}
+
\text{Geometry}
+
\text{Semantics}
}
$$

---

## 6. Output Tensor

Suppose each grid cell produces $K$ values.

The network output is a spatial tensor:

$$
\hat{Y}
\in
\mathbb{R}^{S\times S\times K}
$$

For example:

$$
3\times3\times8
$$

means:

$$
3\times3
$$

spatial locations, each with an 8-dimensional prediction.

It does **not** mean 8 classes.

The useful mental model is:

$$
\boxed{
H\times W
=
\text{Where}
}
$$

and:

$$
\boxed{
K
=
\text{What is predicted at each location}
}
$$

---

## 7. Anchor-Based YOLO

In classical anchor-based YOLO formulations, each grid cell may contain:

$$
B
$$

anchor boxes.

Each anchor produces:

$$
5+C
$$

values.

Therefore:

$$
\boxed{
\hat{Y}
\in
\mathbb{R}^{S\times S\times B(5+C)}
}
$$

For example:

$$
S=3,\quad B=2,\quad C=3
$$

gives:

$$
3\times3\times2(5+3)
=
3\times3\times16
$$

At one cell:

$$
16
=
2\text{ anchors}\times8\text{ values}
$$

Thus 16 is not 16 classes.

---

## 8. Role of Anchor Boxes

An anchor is a reference geometry and a prediction slot.

Conceptually:

$$
\text{Anchor}
+
\text{Predicted Offsets}
\rightarrow
\text{Refined Bounding Box}
$$

Anchor boxes provide:

- multiple prediction slots;
- reference shapes for different object geometries.

Thus:

$$
\boxed{
\text{Grid Cell}
+
\text{Anchor}
\rightarrow
\text{Prediction Slot}
}
$$

This is especially useful when multiple objects or different aspect ratios need to be represented.

---

## 9. Anchor Assignment During Training

This point must be separated from inference.

Suppose the Ground Truth box is:

$$
b_{gt}
$$

and the object center lies in cell:

$$
(i,j)
$$

That cell contains anchors:

$$
a_1,a_2,\ldots,a_B
$$

In an anchor-based training setup, we determine which anchor is the best match for the Ground Truth.

One possible criterion is IoU:

$$
IoU(b_{gt},a_k)
$$

For example:

$$
IoU(b_{gt},a_1)=0.30
$$

$$
IoU(b_{gt},a_2)=0.80
$$

Then anchor 2 is the better match:

$$
(i,j,a_2)
\rightarrow
\text{Responsible Prediction Slot}
$$

Conceptually:

$$
\boxed{
\text{Ground Truth}
\rightarrow
\text{Cell Assignment}
+
\text{Anchor Assignment}
\rightarrow
\text{Training Target}
}
$$

This is a **training-time operation**.

During inference, there is no Ground Truth, so we do not compute:

$$
IoU(\text{prediction},\text{GT})
$$

to choose an anchor.

---

## 10. Forward Propagation

The CNN backbone computes a shared feature representation:

$$
h=f(x;\theta)
$$

The detection head produces:

$$
\hat{Y}=g(h;\phi)
$$

For a classical anchor-based formulation:

$$
\hat{Y}
\in
\mathbb{R}^{S\times S\times B(5+C)}
$$

Conceptually:

```mermaid
flowchart LR
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Feature Maps"]
    C --> D["Detection Head"]
    D --> E["Prediction Tensor"]
```

A single forward pass therefore produces predictions across all grid cells and anchors.

---

## 11. Bounding Box Decoding

The network output is not necessarily the final image-space bounding box directly.

Especially in anchor-based detectors, the network often predicts transformed parameters or offsets:

$$
\hat t_x,\hat t_y,\hat t_w,\hat t_h
$$

These are decoded using the relevant grid location and anchor to obtain:

$$
\hat b_x,\hat b_y,\hat b_w,\hat b_h
$$

Conceptually:

$$
\boxed{
\text{Anchor}
+
\text{Predicted Parameters}
\rightarrow
\text{Decoded Bounding Box}
}
$$

The exact decoding equations depend on the YOLO version.

This is important because:

> **The raw network output space and the final bounding-box coordinate space are not necessarily the same.**

---

## 12. Confidence Filtering

After the forward pass, the model may produce many weak predictions.

Suppose:

$$
p_c=0.02,0.05,0.1,\ldots
$$

Most of these are background-like predictions.

We can apply a confidence threshold:

$$
p_c<\tau_{\text{conf}}
\rightarrow
\text{Discard}
$$

For example:

$$
\tau_{\text{conf}}=0.6
$$

means low-confidence predictions are removed before more expensive post-processing.

The purpose is:

$$
\boxed{
\text{Remove Weak Candidates}
}
$$

---

## 13. Class Prediction

For a surviving candidate, suppose:

$$
(c_1,c_2,c_3)
=
(0.1,0.8,0.1)
$$

Then the predicted class is:

$$
\arg\max_i c_i
=
2
$$

The candidate therefore has:

$$
\text{Class}=2
$$

The complete candidate now consists conceptually of:

$$
[
\text{confidence},
\text{box},
\text{class}
]
$$

---

## 14. Non-Max Suppression in YOLO

Even after confidence filtering, several predictions may still describe the same object.

Suppose:

$$
B_1,B_2,B_3
$$

are all candidate boxes for one car.

NMS performs:

1. Select the highest-confidence box.
2. Keep it.
3. Compute IoU with the remaining boxes.
4. Suppress boxes with:

$$
IoU\ge\tau_{\text{NMS}}
$$

5. Repeat on the remaining candidates.

Conceptually:

$$
\boxed{
\text{Pick}
\rightarrow
\text{Keep}
\rightarrow
\text{Suppress Duplicates}
\rightarrow
\text{Repeat}
}
$$

NMS is normally applied **per class**:

$$
\text{NMS(Car)}
$$

separately from:

$$
\text{NMS(Person)}
$$

This prevents a highly overlapping car and person prediction from automatically suppressing one another.

---

## 15. Complete Inference Pipeline

The complete YOLO-style inference process can be viewed as:

```mermaid
flowchart TD
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Detection Head"]
    C --> D["Raw Prediction Tensor"]
    D --> E["Decode Predictions"]
    E --> F["Confidence Filtering"]
    F --> G["Class-wise NMS"]
    G --> H["Final Detections"]
```

Mathematically:

$$
\boxed{
x
\rightarrow
\hat{Y}
\rightarrow
\text{Decode}
\rightarrow
\text{Filter}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Boxes}
}
$$

---

# 16. YOLO Training Pipeline

Inference and training must be separated carefully.

During training, we have:

$$
\text{Image}
+
\text{Ground Truth}
$$

The pipeline is:

```mermaid
flowchart TD
    A["Image + Ground Truth"] --> B["CNN"]
    B --> C["Predictions"]

    A --> D["Target Assignment"]
    D --> E["Training Targets"]

    C --> F["Loss"]
    E --> F

    F --> G["Backpropagation"]
    G --> H["Parameter Update"]
```

Conceptually:

$$
x
\rightarrow
\hat{Y}
$$

and:

$$
GT
\rightarrow
Y
$$

then:

$$
L(\hat{Y},Y)
$$

followed by:

$$
L
\rightarrow
\frac{\partial L}{\partial\hat{Y}}
\rightarrow
\frac{\partial L}{\partial\theta}
$$

---

## 17. YOLOv1 Loss

YOLOv1 used a relatively simple squared-error-based objective.

Conceptually:

$$
L
=
\lambda_{\text{coord}}L_{\text{coord}}
+
L_{\text{obj}}
+
\lambda_{\text{noobj}}L_{\text{noobj}}
+
L_{\text{class}}
$$

The original YOLO formulation used squared error for localization and confidence/classification terms, with special weighting for localization and background confidence. It also used square-rooted width and height terms in the localization objective. ([arXiv](https://arxiv.org/abs/1506.02640))

### 17.1 Localization Loss

Conceptually:

$$
L_{\text{coord}}
=
(x-\hat x)^2
+
(y-\hat y)^2
+
(\sqrt w-\sqrt{\hat w})^2
+
(\sqrt h-\sqrt{\hat h})^2
$$

### 17.2 Object Confidence

For a positive object prediction:

$$
L_{\text{obj}}
\approx
(1-\hat p_c)^2
$$

### 17.3 No-Object Confidence

For background-like predictions:

$$
L_{\text{noobj}}
\approx
(0-\hat p_c)^2
$$

with reduced weight:

$$
\lambda_{\text{noobj}}
$$

### 17.4 Classification

Conceptually:

$$
L_{\text{class}}
=
\sum_i(c_i-\hat c_i)^2
$$

The important idea is that YOLOv1 used a **single structured loss combining localization, confidence, and classification**.

---

## 18. Why Did YOLO Loss Evolve?

YOLOv1's direct coordinate regression is simple, but it has limitations.

### Coordinate Loss Does Not Directly Optimize IoU

The evaluation of detection geometry naturally depends on:

$$
IoU
$$

while the original training objective uses coordinate-wise squared errors.

Therefore:

$$
\text{Training Geometry Objective}
\neq
\text{IoU Geometry}
$$

### Background Dominates

Dense detection creates many negative locations.

Without careful weighting:

$$
L_{\text{background}}
\gg
L_{\text{positive}}
$$

This can make optimization inefficient.

### Bounding Box Geometry Is Coupled

A bounding box is not just four independent scalar values.

Its quality depends on:

- overlap;
- center position;
- width;
- height;
- aspect ratio.

This motivates IoU-aware box losses.

---

# 19. IoU-Based Bounding Box Loss

A natural idea is to make the training objective more directly reflect box overlap:

$$
L_{\text{IoU}}
=
1-IoU
$$

This is conceptually more aligned with detection geometry than plain MSE.

However, when boxes do not overlap:

$$
IoU=0
$$

and plain IoU loss provides limited geometric information.

This motivates improved IoU-based losses.

---

## 20. GIoU, DIoU, and CIoU

### 20.1. GIoU

**Generalized IoU** extends IoU to provide a useful signal even when the boxes do not overlap.

Conceptually:

$$
L_{\text{GIoU}}
=
1-GIoU
$$

### 20.2. DIoU

**Distance IoU** adds the distance between the centers of the boxes.

Conceptually:

$$
L_{\text{DIoU}}
=
1-IoU
+
\text{Center Distance Penalty}
$$

This means the loss cares about:

$$
\boxed{
\text{Overlap}
+
\text{Center Alignment}
}
$$

### 20.3. CIoU

**Complete IoU** further considers aspect-ratio consistency:

$$
L_{\text{CIoU}}
=
1-IoU
+
\text{Center Distance Penalty}
+
\text{Aspect Ratio Penalty}
$$

The DIoU/CIoU work explicitly incorporates overlap, center distance, and aspect-ratio information into bounding-box regression. ([arXiv](https://arxiv.org/abs/1911.08287))

YOLOv4 was an important milestone that incorporated CIoU-based box regression into a YOLO detector. ([arXiv](https://arxiv.org/abs/2004.10934))

---

# 21. YOLOv5-Era Loss

A common YOLOv5-style loss decomposition is:

$$
\boxed{
L
=
\lambda_{\text{box}}L_{\text{box}}
+
\lambda_{\text{obj}}L_{\text{obj}}
+
\lambda_{\text{cls}}L_{\text{cls}}
}
$$

A representative formulation uses:

$$
L_{\text{box}}
=
L_{\text{CIoU}}
$$

and BCE-style losses for objectness and classification. Ultralytics' YOLOv5 documentation describes CIoU for box regression and BCE for objectness/classification, with balancing across detection outputs. ([GitHub](https://github.com/ultralytics/ultralytics/blob/main/docs/en/yolov5/tutorials/architecture-description.md))

### Objectness

$$
L_{\text{obj}}
=
BCE(\hat p_c,p_c)
$$

### Classification

$$
L_{\text{cls}}
=
BCE(\hat c,c)
$$

The exact weighting and implementation details can vary.

---

# 22. Why BCE Is More Natural for Objectness

Objectness is essentially a binary prediction:

$$
y\in\{0,1\}
$$

The Binary Cross-Entropy loss is:

$$
L_{\text{BCE}}
=
-
[
y\log p
+
(1-y)\log(1-p)
]
$$

This gives a natural probabilistic interpretation.

Thus:

$$
\boxed{
\text{Objectness}
\rightarrow
\text{Binary Classification}
\rightarrow
\text{BCE-style Loss}
}
$$

This is more natural than using squared error purely as a probability regression objective.

In implementation, logits are often passed to a numerically stable BCE-with-logits formulation.

---

# 23. Focal Loss

Dense detection creates a large number of easy negative predictions.

Suppose:

$$
10000
$$

candidate locations exist but only:

$$
20
$$

are positive.

Then easy negatives can dominate the loss.

Focal Loss reduces the contribution of easy examples:

$$
FL(p_t)
=
-\alpha(1-p_t)^\gamma\log(p_t)
$$

The intuition is:

$$
\text{Easy Example}
\rightarrow
\text{Low Loss Weight}
$$

while:

$$
\text{Hard Example}
\rightarrow
\text{Higher Relative Importance}
$$

Focal-style weighting can be applied to BCE-based objectness or classification losses depending on the implementation.

---

# 24. Distribution Focal Loss

Modern detectors can go beyond predicting a single scalar coordinate.

Instead of directly predicting:

$$
x=23.7
$$

the model can predict a distribution over discrete bins:

$$
P(x=0),P(x=1),\ldots,P(x=n)
$$

The final continuous coordinate can then be recovered from the learned distribution.

This idea comes from **Generalized Focal Loss**, which proposed distributional representations for localization rather than treating localization as a single point estimate. ([arXiv](https://arxiv.org/abs/2006.04388))

This is the conceptual basis of **Distribution Focal Loss (DFL)** used in later detector implementations.

---

# 25. Modern Ultralytics Detection Loss

In current Ultralytics detection code, the bounding-box loss combines:

$$
L_{\text{CIoU}}
$$

with:

$$
L_{\text{DFL}}
$$

through `BboxLoss`. The detection loss is conceptually decomposed into:

$$
\boxed{
L
=
L_{\text{box}}
+
L_{\text{cls}}
+
L_{\text{dfl}}
}
$$

where:

$$
L_{\text{box}}
\approx
L_{\text{CIoU}}
$$

and:

$$
L_{\text{dfl}}
=
\text{Distribution Focal Loss}
$$

The exact implementation also depends on the specific model and training configuration. Current Ultralytics loss code exposes `BboxLoss`, `DFLoss`, `FocalLoss`, and `VarifocalLoss` components. ([Ultralytics Docs](https://docs.ultralytics.com/reference/utils/loss), [GitHub](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/utils/loss.py))

---

# 26. Modern YOLO Is Not One Fixed Loss

This is extremely important.

It would be incorrect to say:

> "Modern YOLO always uses CIoU + BCE + DFL."

Different generations and implementations use different formulations.

A useful historical view is:

$$
\boxed{
\text{YOLOv1}
\rightarrow
\text{Squared-Error-Style Loss}
}
$$

$$
\boxed{
\text{YOLOv5 Era}
\rightarrow
\text{CIoU + BCE-Based Losses}
}
$$

$$
\boxed{
\text{Later Detectors}
\rightarrow
\text{IoU-Based Regression + Distributional Localization + Modern Classification/Quality Losses}
}
$$

Therefore:

> **"YOLO loss" is a family of design choices, not one immutable formula.**

---

# 27. Anchor-Based vs. Anchor-Free YOLO

Another important evolution is the move away from explicit anchors.

### Classical Anchor-Based YOLO

Conceptually:

$$
\text{Cell}
+
\text{Anchor}
\rightarrow
\text{Prediction}
$$

with:

$$
S\times S\times B(5+C)
$$

### Modern Anchor-Free Detectors

Instead of predefined anchor boxes, predictions can originate from feature-map locations or anchor points:

$$
\text{Feature Location}
\rightarrow
\text{Positive Assignment}
\rightarrow
\text{Box Prediction}
$$

Modern Ultralytics detector code uses anchor points together with task-aligned assignment rather than relying on predefined anchor boxes in the old anchor-based sense. ([GitHub](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/utils/loss.py))

Therefore:

$$
\boxed{
\text{Anchor Boxes}
\text{ are important historically}
}
$$

but:

$$
\boxed{
\text{Modern YOLO is not necessarily anchor-based}
}
$$

---

# 28. Modern Target Assignment

The assignment mechanism has also evolved.

Classical anchor-based mental model:

$$
\text{GT}
\rightarrow
\text{Cell}
\rightarrow
\text{Best Anchor}
$$

Modern anchor-free assignment may instead use:

$$
\text{GT}
\rightarrow
\text{Candidate Locations}
\rightarrow
\text{Assignment Strategy}
\rightarrow
\text{Positive Samples}
$$

For example, modern Ultralytics detection uses task-aligned assignment to determine which predictions are responsible for each ground-truth object. ([GitHub](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/utils/loss.py))

This is another reason why the training pipeline of modern YOLO should not be reduced to:

> "Compute IoU between every anchor and GT and choose the highest one."

That description is appropriate for a simplified anchor-based explanation, but not for every modern YOLO implementation.

---

# 29. Backward Propagation in YOLO

For a classical multi-component loss:

$$
L
=
\lambda_{\text{box}}L_{\text{box}}
+
\lambda_{\text{obj}}L_{\text{obj}}
+
\lambda_{\text{cls}}L_{\text{cls}}
$$

the gradient is:

$$
\frac{\partial L}{\partial\theta}
=
\lambda_{\text{box}}
\frac{\partial L_{\text{box}}}{\partial\theta}
+
\lambda_{\text{obj}}
\frac{\partial L_{\text{obj}}}{\partial\theta}
+
\lambda_{\text{cls}}
\frac{\partial L_{\text{cls}}}{\partial\theta}
$$

For a modern formulation:

$$
L
=
\lambda_{\text{box}}L_{\text{CIoU}}
+
\lambda_{\text{cls}}L_{\text{cls}}
+
\lambda_{\text{dfl}}L_{\text{dfl}}
$$

then:

$$
\frac{\partial L}{\partial\theta}
=
\lambda_{\text{box}}
\frac{\partial L_{\text{CIoU}}}{\partial\theta}
+
\lambda_{\text{cls}}
\frac{\partial L_{\text{cls}}}{\partial\theta}
+
\lambda_{\text{dfl}}
\frac{\partial L_{\text{dfl}}}{\partial\theta}
$$

This is directly connected to the Multi-Task Learning intuition we discussed earlier.

The shared backbone receives gradient contributions from multiple objectives.

---

# 30. Forward and Backward Computational Graph

A conceptual training graph looks like:

```mermaid
flowchart TD
    A["Input Image"] --> B["Shared CNN Backbone"]
    B --> C["Detection Head"]
    C --> D["Raw Predictions"]

    D --> E["Box Prediction"]
    D --> F["Classification Prediction"]
    D --> G["Objectness / Quality Prediction"]

    E --> H["Box Loss"]
    F --> I["Classification Loss"]
    G --> J["Objectness / Quality Loss"]

    H --> K["Total Loss"]
    I --> K
    J --> K

    K --> L["Backward"]
    L --> M["Shared CNN Parameters"]
```

Forward:

$$
x
\rightarrow
h
\rightarrow
\hat Y
\rightarrow
L
$$

Backward:

$$
L
\rightarrow
\frac{\partial L}{\partial\hat Y}
\rightarrow
\frac{\partial L}{\partial h}
\rightarrow
\frac{\partial L}{\partial\theta}
$$

This is the same computational principle as the neural networks you studied earlier.

What changes is the **structured detection output and its loss**.

---

# 31. Detailed Numerical Example

Suppose:

- grid: $3\times3$;
- 2 anchors;
- 3 classes.

Then:

$$
\hat Y
\in
\mathbb R^{3\times3\times16}
$$

Suppose one Ground Truth object is:

$$
b_{gt}
=
(0.5,0.5,0.4,0.3)
$$

with class 2.

Assume its responsible prediction slot produces:

$$
\hat p_c=0.90
$$

and:

$$
\hat b
=
(0.55,0.48,0.45,0.28)
$$

with:

$$
\hat c=(0.1,0.8,0.1)
$$

---

## 31.1. Objectness

Ground Truth:

$$
p_c=1
$$

Prediction:

$$
\hat p_c=0.9
$$

With BCE:

$$
L_{\text{obj}}
=
-\log(0.9)
$$

---

## 31.2. Classification

Ground Truth:

$$
c=(0,1,0)
$$

Prediction:

$$
\hat c=(0.1,0.8,0.1)
$$

The classification loss compares these distributions using the chosen classification objective.

---

## 31.3. Bounding Box

Ground Truth:

$$
b=(0.5,0.5,0.4,0.3)
$$

Prediction:

$$
\hat b=(0.55,0.48,0.45,0.28)
$$

The box loss is not necessarily MSE in modern YOLO.

A modern implementation may use:

$$
L_{\text{box}}
=
L_{\text{CIoU}}
$$

and:

$$
L_{\text{dfl}}
$$

for distributional localization.

---

## 31.4. Total Loss

Conceptually:

$$
L
=
\lambda_{\text{box}}L_{\text{box}}
+
\lambda_{\text{cls}}L_{\text{cls}}
+
\lambda_{\text{dfl}}L_{\text{dfl}}
$$

or, in a formulation containing objectness:

$$
L
=
\lambda_{\text{box}}L_{\text{box}}
+
\lambda_{\text{obj}}L_{\text{obj}}
+
\lambda_{\text{cls}}L_{\text{cls}}
$$

---

# 32. What Happens During Backward?

Suppose the box prediction is poor.

Then:

$$
\frac{\partial L_{\text{box}}}{\partial\hat b}
$$

creates a gradient signal telling the model how to change the box representation.

If classification is wrong:

$$
\frac{\partial L_{\text{cls}}}{\partial\hat c}
$$

provides another gradient.

If objectness is wrong:

$$
\frac{\partial L_{\text{obj}}}{\partial\hat p_c}
$$

provides another gradient.

All these signals ultimately flow into the shared feature extractor:

$$
\text{Box Loss}
$$

$$
+
$$

$$
\text{Classification Loss}
$$

$$
+
$$

$$
\text{Objectness / Quality Loss}
$$

↓

$$
\text{Shared Backbone}
$$

Thus the backbone learns representations useful for multiple detection objectives simultaneously.

---

# 33. Why YOLO Is Called a One-Stage Detector

A **one-stage detector** directly predicts detections from the image through one detector network.

Conceptually:

$$
\text{Image}
\rightarrow
\text{Detector}
\rightarrow
\text{Boxes + Classes}
$$

This contrasts with traditional two-stage pipelines where:

$$
\text{Image}
\rightarrow
\text{Region Proposals}
\rightarrow
\text{RoI Processing}
\rightarrow
\text{Final Detections}
$$

YOLO's philosophy is to avoid a separate region-proposal stage.

This contributes to its speed and simplicity.

---

# 34. Why Is YOLO Efficient?

The central computational advantage is:

$$
\boxed{
\text{One Global CNN Forward Pass}
}
$$

instead of:

$$
\boxed{
\text{Many Independent Region Evaluations}
}
$$

The network uses convolutional weight sharing to produce dense spatial predictions.

Therefore:

$$
\text{Shared Feature Extraction}
\rightarrow
\text{Many Predictions}
$$

This is the same computational insight we studied in Convolutional Implementation of Sliding Windows, now turned into a complete detection algorithm.

---

# 35. Complete Training vs. Inference Mental Model

This is the distinction you should remember most carefully.

### Training

$$
\boxed{
\text{Image + GT}
\rightarrow
\text{CNN}
\rightarrow
\text{Predictions}
\rightarrow
\text{Assignment}
\rightarrow
\text{Loss}
\rightarrow
\text{Backward}
\rightarrow
\text{Update}
}
$$

Ground Truth is used to determine which predictions should learn each object.

### Inference

$$
\boxed{
\text{Image}
\rightarrow
\text{CNN}
\rightarrow
\text{Predictions}
\rightarrow
\text{Decode}
\rightarrow
\text{Confidence Filtering}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}
}
$$

There is no Ground Truth and therefore no training assignment or loss.

---

# 36. Evolution of YOLO

The YOLO family has evolved substantially.

A useful high-level picture is:

$$
\text{YOLOv1}
$$

↓

> Simple grid + direct regression

↓

$$
\text{YOLOv2}
$$

↓

> Anchor boxes + improved box parameterization

↓

$$
\text{YOLOv3}
$$

↓

> Multi-scale detection

↓

$$
\text{YOLOv4 / YOLOv5-era designs}
$$

↓

> Stronger backbones, improved training, CIoU/BCE-style losses, better engineering

↓

$$
\text{Modern YOLO}
$$

↓

> Anchor-free designs, advanced assignment, distributional localization, improved quality/classification objectives

The important point is not to memorize every version here.

The important idea is:

> **The core one-stage, dense-detection philosophy remains, while the representation and optimization machinery evolves.**

---

# 37. The Most Important Distinctions

Before moving further, keep these distinctions very clear.

### Grid vs. Anchor

$$
\text{Grid}
\rightarrow
\text{Spatial Location}
$$

$$
\text{Anchor}
\rightarrow
\text{Prediction Slot / Reference Geometry}
$$

### Anchor Assignment vs. NMS

Training:

$$
\text{GT}
\rightarrow
\text{Assignment}
$$

Inference:

$$
\text{Predictions}
\rightarrow
\text{NMS}
$$

### Confidence Threshold vs. NMS Threshold

$$
\tau_{\text{conf}}
\rightarrow
\text{Remove weak predictions}
$$

$$
\tau_{\text{NMS}}
\rightarrow
\text{Remove redundant predictions}
$$

### Box Loss vs. IoU Evaluation

Training:

$$
L_{\text{box}}
$$

optimizes parameters.

Evaluation:

$$
IoU
$$

measures geometric agreement.

The loss may use IoU-derived terms, but IoU itself is primarily a geometric metric.

---

# 38. The Full YOLO Mental Model

The entire algorithm can now be summarized as:

$$
\boxed{
\text{Input Image}
}
$$

↓

$$
\boxed{
\text{Shared CNN Feature Extraction}
}
$$

↓

$$
\boxed{
\text{Dense Spatial Predictions}
}
$$

↓

$$
\boxed{
\text{Objectness + Bounding Box + Class}
}
$$

↓

$$
\boxed{
\text{Decode Predictions}
}
$$

↓

$$
\boxed{
\text{Confidence Filtering}
}
$$

↓

$$
\boxed{
\text{Class-wise NMS}
}
$$

↓

$$
\boxed{
\text{Final Detections}
}
$$

For training:

$$
\boxed{
\text{Prediction}
+
\text{Assigned Ground Truth}
\rightarrow
\text{Loss}
\rightarrow
\text{Backpropagation}
}
$$

For modern implementations, the loss may conceptually contain:

$$
\boxed{
L_{\text{box}}
+
L_{\text{classification/quality}}
+
L_{\text{distributional localization}}
}
$$

with concrete formulas depending on the YOLO generation and implementation.

The most important takeaway is:

> **YOLO turns Object Detection into a dense prediction problem: one CNN processes the whole image and produces many structured predictions. Training determines which prediction locations should learn which objects and optimizes box, classification, and quality objectives. Inference decodes those predictions, removes weak candidates, and uses class-wise NMS to convert many candidates into a clean set of final detections.**

This is the foundation for going deeper into **YOLO's exact output parameterization, target assignment, loss derivation, multi-scale detection, and the differences between classical anchor-based YOLO and modern anchor-free YOLO**.

# Evolution of YOLO: From YOLOv1 to YOLO26

**YOLO (You Only Look Once)** is not a single model that has simply been upgraded 26 times. It is a **family of object detectors developed by different research groups and organizations** over time.

The most useful way to study YOLO evolution is not to memorize every version number, but to ask:

> **What limitation did this generation try to solve, and what algorithmic change was introduced to solve it?**

The overall evolution can be summarized as:

$$
\boxed{
\text{Fast Detection}
\rightarrow
\text{Better Localization}
\rightarrow
\text{Multi-Scale Detection}
\rightarrow
\text{Better Training}
\rightarrow
\text{Anchor-Free}
\rightarrow
\text{Better Assignment}
\rightarrow
\text{End-to-End Detection}
}
$$

---

## 1. YOLOv1 — One-Stage Object Detection

YOLOv1 introduced the core YOLO idea:

> **Treat object detection as a single regression problem over the entire image.**

Instead of:

$$
\text{Region Proposal}
\rightarrow
\text{Classification}
\rightarrow
\text{Localization}
$$

YOLO performs:

$$
\text{Image}
\rightarrow
\text{One CNN}
\rightarrow
\text{Bounding Boxes + Classes}
$$

The image is divided into an $S\times S$ grid, and each cell predicts object information.

### Main contribution

$$
\boxed{
\text{One-stage, end-to-end detection}
}
$$

### Problems it solved

- High computational cost of region-based approaches;
- complicated multi-stage detection pipelines;
- slow inference.

### Remaining problems

- Localization accuracy was relatively weak;
- small objects were difficult;
- multiple objects associated with one cell were difficult;
- the box regression formulation was relatively simple;
- recall was lower than some region-based detectors.

Mental model:

$$
\boxed{
\text{YOLOv1}
=
\text{Fast but relatively coarse detection}
}
$$

---

## 2. YOLOv2 / YOLO9000 — Better Localization and Generalization

YOLOv2 focused on improving the weaknesses of YOLOv1, especially localization and recall.

### Major changes

#### Batch Normalization

Improved training stability and model performance.

#### Higher Resolution Training

Reduced the mismatch between classification pretraining and detection.

#### Anchor Boxes

YOLOv1:

$$
\text{Cell}
\rightarrow
\text{Bounding Box}
$$

YOLOv2:

$$
\text{Cell}
+
\text{Anchor}
\rightarrow
\text{Bounding Box}
$$

Anchor boxes provided better geometric priors for different object shapes.

#### Dimension Clustering

Anchor shapes could be derived from the training data rather than selected entirely by hand.

#### Multi-Scale Training

Training at different input resolutions improved robustness.

### YOLO9000

YOLO9000 extended the system to large numbers of classes through joint detection and classification training.

Mental model:

$$
\boxed{
\text{YOLOv2}
=
\text{Better Box Representation}
+
\text{Better Training}
+
\text{Better Recall}
}
$$

---

## 3. YOLOv3 — Multi-Scale Object Detection

YOLOv3 focused strongly on a major weakness:

> **Small objects were difficult to detect.**

The main idea was **multi-scale prediction**.

Instead of predicting only at one feature-map resolution, YOLOv3 predicts at multiple scales:

$$
\text{Large Feature Map}
\rightarrow
\text{Small Objects}
$$

$$
\text{Medium Feature Map}
\rightarrow
\text{Medium Objects}
$$

$$
\text{Small Feature Map}
\rightarrow
\text{Large Objects}
$$

YOLOv3 also introduced the stronger:

$$
\boxed{\text{Darknet-53}}
$$

backbone with residual connections.

### Main improvements

- multi-scale prediction;
- stronger feature extractor;
- improved classification formulation;
- better small-object detection.

Mental model:

$$
\boxed{
\text{YOLOv3}
=
\text{Better Representation Across Scales}
}
$$

---

## 4. YOLOv4 — Optimize the Entire Pipeline

YOLOv4 focused less on one single architectural idea and more on:

> **How can we combine effective architecture and training techniques to improve the speed–accuracy trade-off?**

Important techniques included:

$$
\text{CSP-style architectures}
$$

$$
\text{Mosaic Augmentation}
$$

$$
\text{Mish}
$$

$$
\text{DropBlock}
$$

$$
\text{CIoU Loss}
$$

### Why Mosaic mattered

Mosaic augmentation combines multiple images into one training image.

This helps the model learn:

- multiple objects;
- different scales;
- richer contexts;
- crowded scenes.

### Why CIoU mattered

Instead of treating box coordinates independently, CIoU considers:

$$
\text{Overlap}
+
\text{Center Distance}
+
\text{Aspect Ratio}
$$

Mental model:

$$
\boxed{
\text{YOLOv4}
=
\text{Architecture}
+
\text{Loss}
+
\text{Augmentation}
+
\text{Training Optimization}
}
$$

---

## 5. YOLOv5 — Practical Engineering and Deployment

YOLOv5 belongs to the **Ultralytics** branch rather than being a direct continuation of the original YOLO research lineage.

The major contribution was not a completely new detection paradigm, but a highly practical ecosystem:

- PyTorch implementation;
- easy training;
- model scaling;
- strong augmentation pipeline;
- deployment support;
- engineering-focused optimizations.

Important architectural components included CSP-style structures and SPPF.

The key shift was:

$$
\boxed{
\text{Research Detector}
\rightarrow
\text{Practical Engineering Platform}
}
$$

YOLOv5 helped make YOLO extremely accessible for real-world AI development.

---

## 6. YOLOX — Anchor-Free Detection

YOLOX was an important research branch because it challenged a fundamental assumption:

> **Do we really need predefined anchor boxes?**

The answer was: not necessarily.

YOLOX introduced an anchor-free design together with:

$$
\text{Decoupled Head}
$$

and:

$$
\text{SimOTA}
$$

for label assignment.

The representation becomes conceptually:

$$
\text{Feature Location}
\rightarrow
\text{Object Prediction}
$$

instead of:

$$
\text{Feature Location}
+
\text{Predefined Anchor}
\rightarrow
\text{Object Prediction}
$$

Main idea:

$$
\boxed{
\text{Remove Anchor Complexity}
}
$$

---

## 7. YOLOv6 — Industrial Efficiency

YOLOv6 was developed with strong emphasis on practical deployment and industrial applications.

The focus included:

- efficient backbone design;
- re-parameterization;
- decoupled heads;
- quantization;
- hardware-aware optimization;
- deployment efficiency.

The underlying goal was:

$$
\boxed{
\text{Accuracy}
+
\text{Latency}
+
\text{Deployment Efficiency}
}
$$

rather than purely optimizing benchmark accuracy.

Mental model:

> **YOLOv6 makes YOLO more hardware- and production-oriented.**

---

## 8. YOLOv7 — Better Training Without Proportional Inference Cost

YOLOv7 introduced the concept of:

$$
\boxed{
\text{Trainable Bag-of-Freebies}
}
$$

The idea is:

> Make the training process more sophisticated while keeping inference efficient.

Important ideas included:

$$
\text{E-ELAN}
$$

$$
\text{Model Scaling}
$$

$$
\text{Auxiliary Training Heads}
$$

and:

$$
\text{Re-parameterization}
$$

A useful mental model is:

$$
\boxed{
\text{Training Complexity}
\neq
\text{Inference Complexity}
}
$$

We can accept more complexity during training if it produces a better and still-fast inference model.

---

## 9. YOLOv8 — Modern Anchor-Free Ultralytics YOLO

YOLOv8 is an important milestone in the Ultralytics lineage.

It moved toward:

$$
\boxed{
\text{Anchor-Free Detection}
}
$$

and used:

$$
\boxed{
\text{Decoupled Detection Head}
}
$$

together with modern box regression based on distributional localization.

Important architectural ideas include:

$$
\text{C2f}
$$

and:

$$
\text{DFL-based localization}
$$

The main conceptual shift is:

$$
\text{Anchor-Based}
\rightarrow
\text{Anchor-Free}
$$

This simplifies prediction representation and removes the need to carefully design predefined anchor shapes.

Mental model:

$$
\boxed{
\text{YOLOv8}
=
\text{Modernized Anchor-Free YOLO}
}
$$

---

## 10. YOLOv9 — Better Information and Gradient Flow

YOLOv9 focused on a deeper problem:

> **How efficiently does information and gradient flow through the network?**

It introduced:

$$
\boxed{
\text{Programmable Gradient Information (PGI)}
}
$$

and:

$$
\boxed{
\text{GELAN}
}
$$

The motivation was that transformations through deep networks can create information bottlenecks.

Thus YOLOv9 focused on:

$$
\boxed{
\text{Better Information Flow During Training}
}
$$

rather than simply making the CNN bigger.

This is especially interesting from the optimization perspective.

---

## 11. YOLOv10 — NMS-Free Detection

YOLOv10 addressed a problem directly connected to what we just studied:

> **NMS adds post-processing latency and breaks the clean end-to-end structure of the detector.**

Traditional YOLO inference:

$$
\text{CNN}
\rightarrow
\text{Many Boxes}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}
$$

YOLOv10 investigated how to remove NMS using:

$$
\boxed{
\text{Consistent Dual Assignments}
}
$$

The goal was:

$$
\boxed{
\text{CNN}
\rightarrow
\text{Final Detections}
}
$$

without requiring NMS as the mandatory final selection step.

This is an important shift:

$$
\boxed{
\text{Post-Processing Detector}
\rightarrow
\text{More End-to-End Detector}
}
$$

---

## 12. YOLO11 — Further Ultralytics Optimization

YOLO11 continued the Ultralytics direction of improving the accuracy–efficiency trade-off.

Important architectural changes included:

$$
\text{C3k2}
$$

and:

$$
\text{C2PSA}
$$

together with improvements to the backbone and detection head.

Its main focus was not a completely new object-detection paradigm, but:

$$
\boxed{
\text{Better Accuracy}
+
\text{Better Efficiency}
+
\text{Practical Deployment}
}
$$

Mental model:

> **YOLO11 refines the modern Ultralytics detector rather than reinventing YOLO.**

---

## 13. YOLO12 — Attention-Centric YOLO

YOLO12 explored a different direction:

> CNNs are highly efficient, but attention mechanisms can model long-range dependencies more naturally.

The challenge is:

$$
\text{Attention}
\rightarrow
\text{Strong Global Modeling}
$$

but:

$$
\text{Attention}
\rightarrow
\text{Potentially Expensive}
$$

YOLO12 therefore introduced attention-oriented designs while maintaining a real-time objective.

Important ideas include:

$$
\text{Area Attention}
$$

and:

$$
\text{R-ELAN}
$$

The conceptual shift is:

$$
\boxed{
\text{CNN-centric YOLO}
\rightarrow
\text{Attention-enhanced YOLO}
}
$$

---

## 14. YOLO13 — Higher-Order Global Correlations

YOLO13 went deeper into the global-representation problem.

Rather than only modeling local CNN features or pairwise attention relationships, it explored:

$$
\boxed{
\text{Higher-Order Global Correlations}
}
$$

A major component was:

$$
\text{HyperACE}
$$

with:

$$
\text{FullPAD}
$$

for correlation-enhanced propagation.

The conceptual evolution is:

$$
\text{Local CNN Features}
$$

↓

$$
\text{Attention / Pairwise Relations}
$$

↓

$$
\boxed{
\text{Higher-Order Global Relations}
}
$$

This represents a shift from simply making convolutional features stronger toward modeling richer relationships among visual information.

---

## 15. YOLO26 — Native End-to-End and Deployment Simplification

YOLO26 is the latest **Ultralytics YOLO generation** as of 2026.

Its major focus is directly connected to several limitations we have already studied:

- NMS overhead;
- localization complexity;
- deployment complexity;
- small-target assignment;
- training efficiency.

### 15.1 NMS-Free

YOLO26 is designed around native end-to-end inference:

$$
\boxed{
\text{No Mandatory NMS}
}
$$

Instead of:

$$
\text{Predictions}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}
$$

the model is trained to produce final predictions more directly.

This continues the direction introduced by YOLOv10.

### 15.2 DFL-Free

Modern YOLO generations introduced distributional localization:

$$
\text{DFL}
$$

YOLO26 simplifies this by moving toward:

$$
\boxed{
\text{DFL-Free Regression}
}
$$

This reduces head and deployment complexity.

### 15.3 Progressive Loss

YOLO26 introduces:

$$
\boxed{
\text{ProgLoss}
}
$$

to progressively align training with the final detection objective.

### 15.4 Small-Target-Aware Assignment

YOLO26 introduces:

$$
\boxed{
\text{STAL}
}
$$

to improve positive assignment for small objects.

### 15.5 MuSGD

YOLO26 also introduces:

$$
\boxed{
\text{MuSGD}
}
$$

to improve training behavior through a hybrid optimization strategy.

Mental model:

$$
\boxed{
\text{YOLO26}
=
\text{End-to-End}
+
\text{NMS-Free}
+
\text{DFL-Free}
+
\text{Better Assignment}
+
\text{Deployment Simplicity}
}
$$

---

## 16. Version-by-Version Summary

| Version | Main Limitation | Main Improvement | Key Idea |
| --- | --- | --- | --- |
| **YOLOv1** | Detection was complex and slow | One-stage unified detector | **One CNN for detection** |
| **YOLOv2** | Localization and box representation | Anchor boxes, dimension clustering, BN, multi-scale training | **Better box representation** |
| **YOLOv3** | Small objects | Multi-scale prediction, Darknet-53 | **Detect at multiple scales** |
| **YOLOv4** | Speed–accuracy trade-off | Architecture + augmentation + CIoU + training improvements | **Optimize the whole pipeline** |
| **YOLOv5** | Practical training/deployment | PyTorch ecosystem and engineering improvements | **Production-friendly YOLO** |
| **YOLOX** | Anchor complexity | Anchor-free + decoupled head + SimOTA | **Remove anchor dependence** |
| **YOLOv6** | Industrial deployment | Efficient architecture and re-parameterization | **Hardware-aware YOLO** |
| **YOLOv7** | Accuracy without extra inference cost | Trainable bag-of-freebies, E-ELAN | **Better training efficiency** |
| **YOLOv8** | Old anchor-based representation | Anchor-free + decoupled head + DFL | **Modern Ultralytics design** |
| **YOLOv9** | Information/gradient bottleneck | PGI + GELAN | **Better information flow** |
| **YOLOv10** | NMS latency | End-to-end NMS-free design | **Remove post-processing dependency** |
| **YOLO11** | Further accuracy–efficiency optimization | C3k2 + C2PSA and architecture refinements | **Efficient modern YOLO** |
| **YOLO12** | Limited global modeling | Attention-centric architecture | **Attention for real-time detection** |
| **YOLO13** | Limited higher-order correlations | HyperACE + FullPAD | **Higher-order global modeling** |
| **YOLO26** | NMS, DFL, assignment and deployment complexity | NMS-free, DFL-free, ProgLoss, STAL, MuSGD | **Native end-to-end YOLO** |

---

## 17. The Evolution as Problem → Solution

This is the most useful way to remember the entire history.

### YOLOv1

> Detection is too slow and complicated.

$$
\rightarrow
\boxed{
\text{One-Stage Detection}
}
$$

### YOLOv2

> Bounding-box representation is weak.

$$
\rightarrow
\boxed{
\text{Anchor Boxes}
}
$$

### YOLOv3

> Small objects are difficult.

$$
\rightarrow
\boxed{
\text{Multi-Scale Detection}
}
$$

### YOLOv4

> Need better accuracy without losing real-time performance.

$$
\rightarrow
\boxed{
\text{Architecture + Training + Augmentation Optimization}
}
$$

### YOLOv5

> Need a practical ecosystem.

$$
\rightarrow
\boxed{
\text{Engineering + Deployment}
}
$$

### YOLOX / YOLOv6

> Anchor systems and architecture can be simplified and optimized.

$$
\rightarrow
\boxed{
\text{Anchor-Free + Efficient Heads}
}
$$

### YOLOv7

> Can training be improved without increasing inference cost?

$$
\rightarrow
\boxed{
\text{Trainable Bag-of-Freebies}
}
$$

### YOLOv8

> Modernize YOLO's prediction representation.

$$
\rightarrow
\boxed{
\text{Anchor-Free + Decoupled Head + DFL}
}
$$

### YOLOv9

> Information and gradient flow are limiting learning.

$$
\rightarrow
\boxed{
\text{PGI + GELAN}
}
$$

### YOLOv10

> NMS introduces inference overhead.

$$
\rightarrow
\boxed{
\text{NMS-Free End-to-End Detection}
}
$$

### YOLO11

> Continue improving accuracy and efficiency in a practical ecosystem.

$$
\rightarrow
\boxed{
\text{Architecture Refinement}
}
$$

### YOLO12

> CNNs have limited global interaction.

$$
\rightarrow
\boxed{
\text{Attention-Centric Design}
}
$$

### YOLO13

> Pairwise global modeling is still limited.

$$
\rightarrow
\boxed{
\text{Higher-Order Global Correlation}
}
$$

### YOLO26

> NMS, DFL, assignment and deployment can still be simplified.

$$
\rightarrow
\boxed{
\text{NMS-Free}
+
\text{DFL-Free}
+
\text{Better Assignment}
+
\text{Deployment Simplification}
}
$$

---

## 18. Five Major Eras of YOLO

Instead of remembering every version independently, the entire history can be grouped into five broad eras.

### Era 1 — Make Detection Fast

$$
\text{YOLOv1}
$$

Core idea:

$$
\boxed{
\text{One-stage detection}
}
$$

### Era 2 — Make Localization Better

$$
\text{YOLOv2}
\rightarrow
\text{YOLOv4}
$$

Main ideas:

$$
\text{Anchors}
+
\text{Multi-Scale}
+
\text{Better Loss}
+
\text{Better Training}
$$

### Era 3 — Make YOLO Practical and Efficient

$$
\text{YOLOv5}
\rightarrow
\text{YOLOv8}
$$

Main ideas:

$$
\text{Engineering}
+
\text{Anchor-Free}
+
\text{Efficient Heads}
+
\text{Better Localization}
$$

### Era 4 — Improve the Learning Process

$$
\text{YOLOv9}
$$

Main focus:

$$
\boxed{
\text{Information Flow + Gradient Quality}
}
$$

### Era 5 — Move Toward True End-to-End Detection

$$
\text{YOLOv10}
\rightarrow
\text{YOLO26}
$$

Main ideas:

$$
\text{NMS-Free}
+
\text{Better Assignment}
+
\text{Simpler Localization}
+
\text{Better Global Modeling}
+
\text{Deployment Efficiency}
$$

---

## 19. One Very Important Historical Caveat

It is incorrect to think:

$$
\text{YOLOv1}
\rightarrow
\text{YOLOv2}
\rightarrow
\text{YOLOv3}
\rightarrow
\cdots
\rightarrow
\text{YOLO26}
$$

as one uninterrupted implementation lineage.

There are multiple research branches.

For example:

$$
\text{YOLOv1-v4}
$$

belong to the original research lineage.

$$
\text{YOLOv5}
$$

belongs to Ultralytics.

$$
\text{YOLOv6}
$$

was developed by Meituan.

$$
\text{YOLOv7/v9}
$$

come from another research lineage.

$$
\text{YOLOv10}
$$

is another research line.

$$
\text{YOLO11/YOLO26}
$$

belong to the modern Ultralytics family.

There are also independent research papers using names such as YOLO12, YOLO13, and other YOLO-numbered variants.

Therefore, the best way to learn YOLO history is:

$$
\boxed{
\text{Study the Ideas}
}
$$

rather than:

$$
\boxed{
\text{Memorize the Version Numbers}
}
$$

---

## 20. The YOLO Evolution in One Diagram

```mermaid
flowchart LR
    A["YOLOv1<br/>One-Stage"] --> B["YOLOv2<br/>Anchors + Better Localization"]
    B --> C["YOLOv3<br/>Multi-Scale"]
    C --> D["YOLOv4<br/>Training + Augmentation + CIoU"]
    D --> E["YOLOv5<br/>Practical Engineering"]
    E --> F["YOLOX / YOLOv6<br/>Anchor-Free + Efficiency"]
    F --> G["YOLOv7<br/>Better Training"]
    G --> H["YOLOv8<br/>Anchor-Free + DFL"]
    H --> I["YOLOv9<br/>PGI + GELAN"]
    I --> J["YOLOv10<br/>NMS-Free"]
    J --> K["YOLO11<br/>Architecture Refinement"]
    K --> L["YOLO12<br/>Attention"]
    L --> M["YOLO13<br/>Higher-Order Correlation"]
    M --> N["YOLO26<br/>End-to-End + DFL-Free"]
```

This diagram is a **conceptual evolution of ideas**, not a strict author-to-author lineage.

---

## 21. The Most Important Ideas to Remember

You do not need to memorize 26 version numbers.

The key ideas are:

$$
\boxed{
1.\ \text{One-Stage Detection}
}
$$

$$
\boxed{
2.\ \text{Anchor Boxes}
}
$$

$$
\boxed{
3.\ \text{Multi-Scale Detection}
}
$$

$$
\boxed{
4.\ \text{IoU-Based Box Losses}
}
$$

$$
\boxed{
5.\ \text{Better Data Augmentation}
}
$$

$$
\boxed{
6.\ \text{Anchor-Free Detection}
}
$$

$$
\boxed{
7.\ \text{Better Label Assignment}
}
$$

$$
\boxed{
8.\ \text{Better Information and Gradient Flow}
}
$$

$$
\boxed{
9.\ \text{NMS-Free End-to-End Detection}
}
$$

$$
\boxed{
10.\ \text{Global Modeling + Deployment Efficiency}
}
$$

The overall evolution is:

$$
\boxed{
\text{Fast}
\rightarrow
\text{Accurate}
\rightarrow
\text{Multi-Scale}
\rightarrow
\text{Efficient}
\rightarrow
\text{Anchor-Free}
\rightarrow
\text{Better Training}
\rightarrow
\text{NMS-Free}
\rightarrow
\text{End-to-End}
}
$$

The important mental model is:

> **Every YOLO generation tries to remove one or more bottlenecks in the previous generation—first computational bottlenecks, then localization weaknesses, then scale variation, then training inefficiency, then anchor and assignment complexity, and finally post-processing and deployment overhead.**

As of 2026, **YOLO26 is the latest Ultralytics generation**, while the broader YOLO ecosystem also contains independent research branches, so “the latest YOLO” should always be interpreted with its specific lineage and implementation in mind.

# Region Proposals

**Region Proposals** are a core concept in **two-stage object detection**. They provide candidate regions that are likely to contain objects before the detector performs more detailed classification and bounding-box refinement.

The key idea is:

$$
\boxed{
\text{Find likely object regions first}
\rightarrow
\text{Classify and refine them afterward}
}
$$

This creates the foundation of:

$$
\boxed{
\text{Two-Stage Object Detection}
}
$$

and leads naturally to the evolution:

$$
\text{Selective Search}
\rightarrow
\text{R-CNN}
\rightarrow
\text{Fast R-CNN}
\rightarrow
\text{Faster R-CNN}
$$

---

## 1. What Are Region Proposals?

A **region proposal** is a candidate region in an image that is considered likely to contain an object.

For example:

    ┌────────────────────────────────┐
    │                                │
    │   ┌──────────────┐             │
    │   │              │             │
    │   │  Proposal 1  │             │
    │   │              │             │
    │   └──────────────┘             │
    │                                │
    │              ┌──────────────┐  │
    │              │              │  │
    │              │  Proposal 2 │  │
    │              │              │  │
    │              └──────────────┘  │
    │                                │
    └────────────────────────────────┘

A proposal is **not yet a final detection**.

It only means:

> **"This region may contain an object."**

Therefore:

$$
\boxed{
\text{Region Proposal}
=
\text{Candidate Region}
}
$$

while:

$$
\boxed{
\text{Final Detection}
=
\text{Class}
+
\text{Refined Bounding Box}
}
$$

The distinction is fundamental.

---

## 2. Why Do We Need Region Proposals?

Consider naive sliding-window detection.

We could enumerate a huge number of windows:

$$
W_1,W_2,\ldots,W_N
$$

and process them.

But most regions are background.

For example:

    ┌──────────────────────────────┐
    │                              │
    │       background             │
    │                              │
    │             ┌───────┐        │
    │             │ object│        │
    │             └───────┘        │
    │                              │
    └──────────────────────────────┘

If there are:

$$
10000
$$

possible candidate regions but only:

$$
200
$$

regions that are likely to contain objects, it is much more efficient to focus the expensive detection stage on those 200 candidates.

Thus:

$$
\boxed{
\text{Huge Search Space}
\rightarrow
\text{Smaller Candidate Set}
}
$$

This is the main motivation behind region proposals.

---

## 3. Two-Stage Object Detection

Region proposals create a two-stage pipeline:

```mermaid
flowchart LR
    A["Input Image"] --> B["Feature Extraction"]
    B --> C["Region Proposals"]
    C --> D["RoI Features"]
    D --> E["Classification + Box Refinement"]
    E --> F["Final Detections"]
```

### Stage 1: Region Proposal

> **Where might objects be?**

$$
\text{Feature Map}
\rightarrow
\text{Candidate Regions}
$$

### Stage 2: Detection

> **What is in each region, and what is its precise bounding box?**

$$
\text{Candidate Region}
\rightarrow
\text{Class}
+
\text{Refined Box}
$$

Therefore:

$$
\boxed{
\text{Two-Stage Detector}
=
\text{Proposal Stage}
+
\text{Detection Stage}
}
$$

---

## 4. Region Proposal vs. Sliding Window

Both approaches attempt to search for objects across an image, but they do it differently.

### Sliding Window

Generate a large number of fixed or predefined windows:

$$
W_1,W_2,\ldots,W_N
$$

Conceptually:

$$
\boxed{
\text{Enumerate Many Regions}
}
$$

### Region Proposal

Use a mechanism to identify regions that are more likely to contain objects:

$$
\boxed{
\text{Select Likely Regions}
}
$$

Therefore:

$$
\text{Sliding Window}
\rightarrow
\text{Search Broadly}
$$

while:

$$
\text{Region Proposal}
\rightarrow
\text{Reduce the Search Space}
$$

This is why region proposals are useful for two-stage detectors.

---

## 5. Region Proposal and Bounding Boxes

A proposal is usually represented using bounding-box coordinates such as:

$$
(x_{\min},y_{\min},x_{\max},y_{\max})
$$

or equivalently:

$$
(b_x,b_y,b_w,b_h)
$$

But its meaning is different from a final bounding-box prediction.

### Proposal

> A region that may contain an object.

### Final Bounding Box

> A refined region that the detector believes accurately encloses the object.

Therefore:

$$
\boxed{
\text{Proposal}
\rightarrow
\text{Candidate}
}
$$

then:

$$
\boxed{
\text{Detection Head}
\rightarrow
\text{Refined Box + Class}
}
$$

A proposal may overlap the Ground Truth reasonably well without being perfectly aligned.

For example:

$$
IoU=0.7
$$

may indicate that the proposal is already a good candidate, but the second stage can further refine the box.

---

## 6. Selective Search

One of the important early region-proposal techniques was:

$$
\boxed{
\text{Selective Search}
}
$$

Selective Search is a **class-independent** proposal method.

Its intuition is:

> Regions that look visually related may belong to the same object.

It uses image properties such as:

- color similarity;
- texture;
- spatial proximity;
- region size.

Conceptually:

```mermaid
flowchart LR
    A["Small Image Regions"] --> B["Measure Similarity"]
    B --> C["Merge Similar Regions"]
    C --> D["Larger Candidate Regions"]
    D --> E["Region Proposals"]
```

Instead of enumerating every possible box, Selective Search generates a more manageable set of candidate regions.

---

## 7. Limitation of Selective Search

Although Selective Search reduces the search space, it is still a separate computational procedure.

A traditional R-CNN pipeline looked conceptually like:

$$
\text{Image}
\rightarrow
\text{Selective Search}
\rightarrow
\text{Thousands of Proposals}
$$

Then each proposal could be processed separately:

$$
\text{Proposal}_1\rightarrow CNN
$$

$$
\text{Proposal}_2\rightarrow CNN
$$

$$
\text{Proposal}_3\rightarrow CNN
$$

and so on.

This creates a serious computational bottleneck.

Therefore:

$$
\boxed{
\text{Proposal Generation}
}
$$

and:

$$
\boxed{
\text{Repeated CNN Evaluation}
}
$$

were both major problems in the early pipeline.

---

## 8. R-CNN

R-CNN (**Regions with CNN features**) combined:

$$
\text{Region Proposals}
+
\text{CNN Features}
$$

The pipeline was:

```mermaid
flowchart LR
    A["Input Image"] --> B["Selective Search"]
    B --> C["Region Proposals"]
    C --> D["Crop / Warp Each Proposal"]
    D --> E["CNN"]
    E --> F["Feature Vector"]
    F --> G["Classifier + Box Regression"]
    G --> H["Final Detection"]
```

The major problem is:

$$
N\text{ proposals}
\rightarrow
N\text{ CNN forward passes}
$$

If there are thousands of proposals, this becomes extremely expensive.

Thus R-CNN demonstrated that region proposals were useful, but its computational structure was inefficient.

---

## 9. Fast R-CNN

Fast R-CNN solves an important part of the R-CNN bottleneck using a concept very closely related to **Convolutional Implementation of Sliding Windows**.

Instead of:

$$
\text{Proposal}_1\rightarrow CNN
$$

$$
\text{Proposal}_2\rightarrow CNN
$$

$$
\text{Proposal}_3\rightarrow CNN
$$

we compute the image features once:

$$
\boxed{
\text{Full Image}
\rightarrow
\text{CNN}
\rightarrow
\text{Shared Feature Map}
}
$$

Then the proposals are mapped onto that shared feature map.

The pipeline becomes:

```mermaid
flowchart LR
    A["Full Image"] --> B["CNN"]
    B --> C["Shared Feature Map"]
    C --> D["Region Proposals"]
    D --> E["RoI Pooling"]
    E --> F["Fully Connected Layers"]
    F --> G["Classification + Box Regression"]
```

The major improvement is:

$$
\boxed{
\text{CNN Once}
+
\text{Proposal-Specific Processing}
}
$$

instead of:

$$
\boxed{
\text{CNN per Proposal}
}
$$

This is conceptually very similar to the computation-sharing idea you studied earlier.

---

## 10. RoI Pooling

Fast R-CNN needs to extract features corresponding to each proposal from the shared feature map.

That is the role of:

$$
\boxed{
\text{RoI Pooling}
}
$$

Suppose the shared feature map is:

    ┌──────────────────────────┐
    │                          │
    │      ┌──────────────┐    │
    │      │      RoI     │    │
    │      │              │    │
    │      └──────────────┘    │
    │                          │
    └──────────────────────────┘

RoI Pooling converts the variable-size proposal region into a fixed-size feature representation.

Conceptually:

$$
\text{Feature Map}
+
\text{Proposal}
\rightarrow
\text{Fixed-Size RoI Feature}
$$

This allows subsequent fully connected layers to process proposals consistently.

---

## 11. Faster R-CNN

Fast R-CNN still depended on an external proposal algorithm such as Selective Search.

The remaining bottleneck was:

$$
\boxed{
\text{Region Proposal Generation}
}
$$

Faster R-CNN solves this by introducing:

$$
\boxed{
\text{Region Proposal Network (RPN)}
}
$$

The proposal generator itself becomes a neural network.

The pipeline becomes:

```mermaid
flowchart TD
    A["Input Image"] --> B["CNN Backbone"]
    B --> C["Shared Feature Map"]

    C --> D["Region Proposal Network"]
    D --> E["Region Proposals"]

    C --> F["RoI Feature Extraction"]
    E --> F

    F --> G["Detection Head"]
    G --> H["Classification + Refined Bounding Boxes"]
```

This is a major conceptual improvement:

> **Region proposal generation becomes learned rather than being a completely separate hand-crafted algorithm.**

---

## 12. Region Proposal Network

The RPN works directly on the shared feature map.

At each spatial location, it can consider candidate box geometries using anchors.

Conceptually:

$$
\text{Feature Location}
\rightarrow
\text{Anchors}
\rightarrow
\text{Objectness + Box Refinement}
$$

For an anchor:

$$
a_k
$$

the RPN predicts something like:

$$
p_{\text{object}}
$$

and:

$$
\Delta x,\Delta y,\Delta w,\Delta h
$$

Therefore:

$$
\boxed{
\text{Anchor}
\rightarrow
\text{RPN}
\rightarrow
\text{Proposal}
}
$$

The resulting proposals are then passed to the second-stage detector.

---

## 13. Anchor Boxes in RPN

This creates a direct connection to the Anchor Boxes concept.

At one feature-map location:

$$
a_1,a_2,\ldots,a_B
$$

may represent different sizes and aspect ratios.

The RPN evaluates and refines them.

Conceptually:

$$
\text{Anchor}
\rightarrow
\text{Objectness}
+
\text{Box Refinement}
$$

then:

$$
\text{RPN Predictions}
\rightarrow
\text{Region Proposals}
$$

Thus Anchor Boxes are not exclusive to YOLO.

They also appear naturally in two-stage detectors such as Faster R-CNN.

---

## 14. Region Proposal and Receptive Field

This connects directly to the CNN concepts we have studied.

A CNN produces a feature map.

Each spatial feature has a receptive field in the original image:

$$
\text{Input Region}
\rightarrow
\text{Receptive Field}
\rightarrow
\text{Feature}
$$

The RPN then operates on these features:

$$
\text{Feature}
\rightarrow
\text{Proposal Prediction}
$$

Therefore:

$$
\boxed{
\text{Receptive Field}
\rightarrow
\text{Visual Evidence}
\rightarrow
\text{Region Proposal}
}
$$

So Region Proposal is not independent of CNN mechanics. It is another way of using shared convolutional features to locate candidate objects.

---

## 15. Region Proposal vs. Final Detection

This distinction should remain very clear.

A proposal may be:

> "There might be an object here."

A final detection says:

> "This is a car, and this is its refined bounding box."

Therefore:

$$
\boxed{
\text{Proposal}
\neq
\text{Final Detection}
}
$$

The process is:

$$
\text{Proposal}
\rightarrow
\text{RoI Feature}
\rightarrow
\text{Classification}
+
\text{Box Refinement}
\rightarrow
\text{Final Detection}
$$

This is the fundamental reason two-stage detectors need two stages.

---

## 16. Why Can Two-Stage Detection Be Accurate?

The conceptual advantage is specialization.

### Stage 1

$$
\text{Where should I look?}
$$

The model finds candidate regions.

### Stage 2

$$
\text{What exactly is here?}
$$

The model examines those candidate regions more carefully.

Therefore:

$$
\boxed{
\text{Stage 1}
=
\text{Candidate Generation}
}
$$

$$
\boxed{
\text{Stage 2}
=
\text{Detailed Detection}
}
$$

This can produce strong localization and classification performance, at the cost of a more complicated pipeline.

---

## 17. Region Proposal vs. YOLO

This is the most important connection to the previous section.

### Two-Stage Detector

```text
Image
  ↓
CNN
  ↓
Region Proposals
  ↓
RoI Features
  ↓
Classification + Box Refinement
  ↓
Final Detections
```

Conceptually:

$$
\boxed{
\text{Proposal First}
\rightarrow
\text{Detection Second}
}
$$

### YOLO

```text
Image
  ↓
CNN
  ↓
Dense Predictions
  ↓
Decode / Filter / NMS
  ↓
Final Detections
```

Conceptually:

$$
\boxed{
\text{Direct Dense Detection}
}
$$

Therefore:

$$
\boxed{
\text{Two-Stage}
=
\text{Proposal First}
}
$$

while:

$$
\boxed{
\text{One-Stage}
=
\text{Detection Directly}
}
$$

---

## 18. Why Does YOLO Not Need a Separate Proposal Stage?

YOLO treats object detection itself as a dense prediction problem.

Instead of asking:

> "Which regions should I process?"

YOLO effectively predicts at many spatial locations at once:

$$
\text{Image}
\rightarrow
\text{Shared CNN}
\rightarrow
\text{Dense Detection Predictions}
$$

The detector simultaneously learns:

$$
\text{Objectness}
+
\text{Bounding Box}
+
\text{Class}
$$

Therefore, there is no separate explicit proposal stage like:

$$
\text{RPN}
\rightarrow
\text{RoI Processing}
$$

This is one of the core distinctions between YOLO-style one-stage detection and Faster R-CNN-style two-stage detection.

---

## 19. Evolution of Region Proposal Methods

The history can be seen as a progression:

$$
\boxed{
\text{Sliding Window}
}
$$

> Enumerate many possible regions.

↓

$$
\boxed{
\text{Selective Search}
}
$$

> Generate a smaller set of likely regions using image structure.

↓

$$
\boxed{
\text{R-CNN}
}
$$

> Apply CNN features to those proposals.

↓

$$
\boxed{
\text{Fast R-CNN}
}
$$

> Compute the CNN feature map once and share it across proposals.

↓

$$
\boxed{
\text{Faster R-CNN}
}
$$

> Learn region proposals using an RPN.

This represents a very important evolution:

$$
\boxed{
\text{Hand-Crafted Proposal Generation}
\rightarrow
\text{Learned Proposal Generation}
}
$$

---

## 20. The Connection to Convolutional Sliding Windows

There is a very useful relationship between the concepts we have studied.

### Convolutional Sliding Window asks:

> How can we evaluate many spatial locations efficiently?

$$
\text{Many Locations}
\rightarrow
\text{Shared Computation}
$$

### Region Proposals ask:

> How can we avoid spending detailed computation on obviously irrelevant regions?

$$
\text{Many Possible Regions}
\rightarrow
\text{Fewer Candidate Regions}
$$

Therefore, they optimize different aspects of the detection problem:

$$
\boxed{
\text{Convolutional Sliding Window}
\rightarrow
\text{Computation Sharing}
}
$$

and:

$$
\boxed{
\text{Region Proposal}
\rightarrow
\text{Search-Space Reduction}
}
$$

---

## 21. Complete Two-Stage Detection Mental Model

The complete idea is:

$$
\boxed{
\text{Image}
\rightarrow
\text{Backbone}
\rightarrow
\text{Feature Map}
\rightarrow
\text{Region Proposals}
\rightarrow
\text{RoI Features}
\rightarrow
\text{Classification + Box Refinement}
\rightarrow
\text{Final Detections}
}
$$

And the historical progression is:

$$
\boxed{
\text{Sliding Window}
\rightarrow
\text{Selective Search}
\rightarrow
\text{R-CNN}
\rightarrow
\text{Fast R-CNN}
\rightarrow
\text{Faster R-CNN}
}
$$

The fundamental distinction from YOLO is:

$$
\boxed{
\text{Two-Stage}
=
\text{Proposal First, Detection Second}
}
$$

while:

$$
\boxed{
\text{One-Stage}
=
\text{Direct Dense Detection}
}
$$

The key mental model to retain is:

> **A Region Proposal is not the final prediction. It is an intermediate candidate region used to reduce the search space before the detector performs more detailed classification and bounding-box refinement.** This idea eventually evolves from hand-crafted methods such as Selective Search into learned proposal generation through the Region Proposal Network in Faster R-CNN.

# Semantic Segmentation with U-Net

**Semantic Segmentation** is a computer vision task in which the model predicts a semantic class for **every pixel** in an image.

Unlike image classification, which predicts one class for the entire image, or object detection, which predicts object-level bounding boxes, semantic segmentation produces a **pixel-level prediction map**.

U-Net is a classic architecture designed specifically for this type of problem.

Its core architecture is:

$$
\boxed{
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder}
}
$$

with an essential mechanism:

$$
\boxed{
\text{Skip Connections}
}
$$

The central idea is:

> **Use deep features to understand what is present in the image, while preserving high-resolution spatial information to determine exactly where each class appears.**

---

## 1. What Is Semantic Segmentation?

Suppose the input image has shape:

$$
H\times W\times3
$$

and there are $C$ semantic classes.

The model produces:

$$
\hat Y\in\mathbb R^{H\times W\times C}
$$

At each pixel $(i,j)$, the network predicts a vector:

$$
\hat y_{ij}
=
[
p_{ij}^{(1)},
p_{ij}^{(2)},
\ldots,
p_{ij}^{(C)}
]
$$

where:

$$
p_{ij}^{(c)}
$$

is the predicted probability that pixel $(i,j)$ belongs to class $c$.

The predicted class is:

$$
\hat c_{ij}
=
\arg\max_c p_{ij}^{(c)}
$$

Therefore:

$$
\boxed{
\text{One Pixel}
\rightarrow
\text{One Semantic Class}
}
$$

The final output is therefore a **segmentation map** with the same spatial resolution as the original image.

---

## 2. Semantic Segmentation vs. Other Computer Vision Tasks

### 2.1. Image Classification

Classification answers:

> What is present in the image?

$$
\text{Image}
\rightarrow
\text{Class}
$$

For example:

$$
\text{Image}
\rightarrow
\text{Cat}
$$

Only one global prediction is needed.

### 2.2. Object Detection

Detection answers:

> What objects are present, and where are they?

$$
\text{Image}
\rightarrow
\{
\text{Class},\text{Bounding Box}
\}
$$

The prediction is object-level.

### 2.3. Semantic Segmentation

Segmentation answers:

> Which class does each pixel belong to?

$$
\text{Image}
\rightarrow
\text{Class for Every Pixel}
$$

Therefore:

$$
\boxed{
\text{Classification}
\rightarrow
\text{Image-level}
}
$$

$$
\boxed{
\text{Object Detection}
\rightarrow
\text{Object-level}
}
$$

$$
\boxed{
\text{Semantic Segmentation}
\rightarrow
\text{Pixel-level}
}
$$

---

## 3. Why Is Semantic Segmentation Difficult?

The difficulty comes from a fundamental trade-off inside CNNs.

Deep CNN layers are good at learning semantic information:

$$
\text{Edges}
\rightarrow
\text{Textures}
\rightarrow
\text{Parts}
\rightarrow
\text{Objects}
$$

As the network becomes deeper, spatial resolution usually decreases:

$$
H\times W
\rightarrow
\frac H2\times\frac W2
\rightarrow
\frac H4\times\frac W4
\rightarrow
\frac H8\times\frac W8
$$

This provides larger receptive fields and stronger semantic understanding.

However, downsampling also removes fine spatial details.

For example, a deep feature may know:

> "There is a person in this region."

but may not preserve enough information to determine the exact boundary of the person's arm, hair, or body.

Therefore segmentation needs both:

$$
\boxed{
\text{Strong Semantic Representation}
}
$$

and:

$$
\boxed{
\text{Fine Spatial Information}
}
$$

This is the fundamental problem that U-Net addresses.

---

## 4. Overall U-Net Architecture

U-Net has an encoder-decoder structure.

```mermaid
flowchart LR
    A["Input Image"] --> B["Encoder"]
    B --> C["Bottleneck"]
    C --> D["Decoder"]
    D --> E["Segmentation Map"]

    B -. "Skip Connections" .-> D
```

The architecture can be divided into three major parts:

### Encoder

Extracts increasingly abstract features while reducing spatial resolution.

### Bottleneck

Contains the deepest semantic representation.

### Decoder

Gradually increases spatial resolution and reconstructs the pixel-level prediction.

The distinctive component is:

$$
\boxed{
\text{Skip Connections}
}
$$

which connect encoder features to decoder features at corresponding resolutions.

---

## 5. Encoder

The encoder is essentially a CNN feature extractor.

Suppose the input is:

$$
256\times256\times3
$$

A simplified encoder might produce:

$$
256\times256
\rightarrow
128\times128
\rightarrow
64\times64
\rightarrow
32\times32
\rightarrow
16\times16
$$

At the same time, the number of channels typically increases:

$$
3
\rightarrow
64
\rightarrow
128
\rightarrow
256
\rightarrow
512
$$

The general pattern is:

$$
\text{Spatial Resolution}\downarrow
$$

while:

$$
\text{Feature Channels}\uparrow
$$

This allows the network to learn increasingly abstract representations.

Conceptually:

$$
\boxed{
\text{Encoder}
=
\text{Semantic Feature Extraction}
}
$$

---

## 6. Why Does the Encoder Downsample?

Downsampling is not simply a disadvantage.

It provides important benefits:

### Larger Receptive Field

A deeper neuron can see a larger region of the original input.

Therefore:

$$
\text{Depth + Downsampling}
\rightarrow
\text{Larger Context}
$$

### Stronger Semantic Representation

The network moves from low-level patterns:

$$
\text{Edges}
$$

to increasingly abstract concepts:

$$
\text{Textures}
\rightarrow
\text{Parts}
\rightarrow
\text{Objects}
$$

Therefore:

$$
\boxed{
\text{Downsampling}
\rightarrow
\text{Better Semantic Understanding}
}
$$

But the cost is:

$$
\boxed{
\text{Downsampling}
\rightarrow
\text{Loss of Spatial Precision}
}
$$

U-Net is designed to recover this lost precision.

---

## 7. Bottleneck

The bottleneck is the deepest representation of the network.

For example:

$$
16\times16\times512
$$

Compared with the original image, the spatial resolution is much smaller, but the representation contains highly abstract semantic information.

The receptive field is large, allowing the network to capture broad contextual information.

Conceptually:

$$
\boxed{
\text{Bottleneck}
=
\text{High-Level Semantic Representation}
}
$$

However, the bottleneck alone is not sufficient for precise segmentation because its spatial resolution is relatively low.

---

## 8. Decoder

The decoder performs the reverse spatial transformation.

For example:

$$
16\times16
\rightarrow
32\times32
\rightarrow
64\times64
\rightarrow
128\times128
\rightarrow
256\times256
$$

Its goal is to reconstruct a high-resolution prediction map.

Conceptually:

$$
\boxed{
\text{Decoder}
=
\text{Spatial Resolution Reconstruction}
}
$$

But simply upsampling the bottleneck is not enough.

The decoder cannot perfectly reconstruct information that was already discarded during encoder downsampling.

This is why U-Net uses skip connections.

---

## 9. Skip Connections

This is the most important component of U-Net.

At each encoder level, the corresponding feature map is forwarded directly to the decoder.

For example:

```mermaid
flowchart LR
    A["Encoder 64 × 64"] -. "Skip" .-> D["Decoder 64 × 64"]
    B["Encoder 32 × 32"] -. "Skip" .-> C["Decoder 32 × 32"]

    A --> B
    B --> E["Bottleneck"]
    E --> C
    C --> D
```

The two sources of information have different roles.

### Encoder Feature

Contains relatively high-resolution spatial information.

It helps preserve:

- edges;
- boundaries;
- local structures;
- fine spatial details.

### Decoder Feature

Contains information reconstructed from deeper semantic representations.

It helps answer:

> What object or semantic structure is present?

Therefore:

$$
\boxed{
\text{Decoder Feature}
+
\text{Encoder Feature}
}
$$

combines:

$$
\boxed{
\text{Semantic Information}
+
\text{Spatial Information}
}
$$

This is the central insight of U-Net.

---

## 10. Why Skip Connections Are Necessary

Suppose the encoder transforms:

$$
256\times256
\rightarrow
16\times16
$$

and the decoder later reconstructs:

$$
16\times16
\rightarrow
256\times256
$$

Upsampling can increase spatial resolution, but it does not magically recover every detail lost during the downsampling process.

For example, once a fine boundary has been compressed away, simply enlarging the feature map does not reconstruct the original boundary exactly.

U-Net therefore stores high-resolution features from the encoder:

$$
64\times64
$$

$$
32\times32
$$

and so on.

The decoder can then access those features directly.

Thus:

$$
\boxed{
\text{Skip Connection}
=
\text{Preserve Fine Spatial Information}
}
$$

---

## 11. Concatenation in U-Net

The original U-Net combines encoder and decoder features using **concatenation**.

Suppose the decoder feature is:

$$
D\in\mathbb R^{64\times64\times128}
$$

and the corresponding encoder feature is:

$$
E\in\mathbb R^{64\times64\times64}
$$

Concatenating along the channel dimension gives:

$$
\operatorname{Concat}(D,E)
\in
\mathbb R^{64\times64\times192}
$$

because:

$$
128+64=192
$$

Conceptually:

```text
Encoder Feature
64 × 64 × 64
        │
        │
        ▼
   Concatenate
        ▲
        │
Decoder Feature
64 × 64 × 128
        │
        ▼
64 × 64 × 192
```

The subsequent convolution layers learn how to combine these two sources of information.

---

## 12. Forward Propagation

Let:

$$
e_1,e_2,\ldots,e_n
$$

represent the encoder features and:

$$
z
$$

the bottleneck.

The decoder progressively reconstructs the spatial resolution.

A simplified decoder step is:

$$
u_1
=
\operatorname{Upsample}(z)
$$

Then combine the upsampled feature with the corresponding encoder feature:

$$
c_1
=
\operatorname{Concat}(u_1,e_n)
$$

and process it:

$$
d_1=f(c_1)
$$

The process continues:

$$
d_1
\rightarrow
\operatorname{Upsample}
\rightarrow
\operatorname{Concat}(e_{n-1})
\rightarrow
d_2
$$

and so on until the original spatial resolution is recovered.

Finally:

$$
\hat Y
=
\operatorname{Conv}_{1\times1}(d_{\text{final}})
$$

where:

$$
\hat Y\in\mathbb R^{H\times W\times C}
$$

The complete forward flow is:

$$
\boxed{
x
\rightarrow
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder + Skip Connections}
\rightarrow
\hat Y
}
$$

---

## 13. Output Representation

Suppose there are $C$ semantic classes.

The final output is:

$$
\hat Y\in\mathbb R^{H\times W\times C}
$$

For each pixel $(i,j)$:

$$
\hat y_{ij}
=
[
p_{ij}^{(1)},
p_{ij}^{(2)},
\ldots,
p_{ij}^{(C)}
]
$$

Therefore:

$$
\boxed{
H\times W
=
\text{Where}
}
$$

and:

$$
\boxed{
C
=
\text{What class is predicted at each pixel}
}
$$

This is conceptually similar to the spatial output tensor used in detection, but the prediction is now made at **pixel level** rather than object level.

---

## 14. Loss Function

Semantic segmentation is essentially a classification problem applied independently to every pixel.

For multi-class segmentation, a common loss is pixel-wise cross entropy:

$$
L
=
-\frac{1}{HW}
\sum_{i=1}^{H}
\sum_{j=1}^{W}
\log
p_{ij}^{(y_{ij})}
$$

where:

- $y_{ij}$ is the ground-truth class of pixel $(i,j)$;
- $p_{ij}^{(y_{ij})}$ is the predicted probability assigned to the correct class.

Thus:

$$
\boxed{
L
=
\text{Average Pixel-wise Classification Loss}
}
$$

The important difference from object detection is that the supervision exists at every pixel.

---

## 15. Backward Propagation

The forward process is:

$$
x
\rightarrow
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder}
\rightarrow
\hat Y
\rightarrow
L
$$

The backward process is:

$$
L
\rightarrow
\frac{\partial L}{\partial\hat Y}
\rightarrow
\text{Decoder}
\rightarrow
\text{Encoder}
$$

Because U-Net contains skip connections, the gradient can flow through multiple paths.

Conceptually:

```mermaid
flowchart BT
    A["Segmentation Loss"] --> B["Decoder"]
    B --> C["Bottleneck"]
    B --> D["Skip Connection"]
    C --> E["Deep Encoder"]
    D --> F["Shallow Encoder"]
```

Therefore, an encoder layer can receive gradient information through:

$$
\text{Main Encoder-Decoder Path}
$$

and:

$$
\text{Skip Connection Path}
$$

This gives skip connections another important role:

$$
\boxed{
\text{Better Gradient Flow}
}
$$

in addition to preserving spatial information during the forward pass.

---

## 16. Why U-Net Works

U-Net addresses the central CNN trade-off.

The encoder provides:

$$
\boxed{
\text{Deep Semantic Information}
}
$$

but loses some spatial precision.

The decoder provides:

$$
\boxed{
\text{High-Resolution Reconstruction}
}
$$

but needs enough spatial information to reconstruct precise boundaries.

Skip connections bridge the two:

$$
\boxed{
\text{Encoder Spatial Features}
+
\text{Decoder Semantic Features}
}
$$

Therefore:

$$
\boxed{
\text{U-Net}
=
\text{Semantic Understanding}
+
\text{Spatial Precision}
}
$$

This is the fundamental reason U-Net is effective for segmentation.

---

## 17. Semantic Segmentation vs. Instance Segmentation

Suppose an image contains two people.

Semantic segmentation assigns:

$$
\text{Person}
$$

to pixels belonging to both people.

It does not distinguish between:

$$
\text{Person 1}
$$

and:

$$
\text{Person 2}
$$

Therefore:

$$
\boxed{
\text{Semantic Segmentation}
=
\text{Class per Pixel}
}
$$

whereas:

$$
\boxed{
\text{Instance Segmentation}
=
\text{Instance Identity per Pixel}
}
$$

This distinction is important because U-Net is fundamentally designed around semantic segmentation.

---

## 18. U-Net vs. YOLO

The difference becomes very clear when comparing the two models.

### YOLO

YOLO produces object-level predictions:

$$
S\times S\times K
$$

A spatial prediction can contain:

$$
\text{Objectness}
+
\text{Bounding Box}
+
\text{Class}
$$

The question is:

> **Where are the objects?**

### U-Net

U-Net produces:

$$
H\times W\times C
$$

where every pixel gets a semantic class.

The question is:

> **What class does each pixel belong to?**

Therefore:

$$
\boxed{
\text{YOLO}
\rightarrow
\text{Object-Level Prediction}
}
$$

and:

$$
\boxed{
\text{U-Net}
\rightarrow
\text{Pixel-Level Prediction}
}
$$

---

## 19. A Complete U-Net Example

Suppose the task has three semantic classes:

$$
C=3
$$

such as:

$$
\{\text{Background},\text{Person},\text{Car}\}
$$

The input is:

$$
256\times256\times3
$$

The encoder progressively reduces the resolution:

$$
256\times256
\rightarrow
128\times128
\rightarrow
64\times64
\rightarrow
32\times32
\rightarrow
16\times16
$$

The bottleneck might be:

$$
16\times16\times512
$$

The decoder then reconstructs:

$$
16\times16
\rightarrow
32\times32
\rightarrow
64\times64
\rightarrow
128\times128
\rightarrow
256\times256
$$

At every stage, the decoder receives the corresponding encoder feature through a skip connection.

Finally:

$$
256\times256\times512
\rightarrow
1\times1\text{ Conv}
\rightarrow
256\times256\times3
$$

For one pixel:

$$
(i,j)
$$

the model predicts:

$$
[
p_{ij}^{(\text{background})},
p_{ij}^{(\text{person})},
p_{ij}^{(\text{car})}
]
$$

and:

$$
\hat c_{ij}
=
\arg\max_c p_{ij}^{(c)}
$$

The output is therefore a complete semantic segmentation mask.

---

## 20. Core Mental Model

The fundamental problem is:

$$
\text{Downsampling}
\rightarrow
\text{Strong Semantic Features}
$$

but:

$$
\text{Downsampling}
\rightarrow
\text{Loss of Spatial Detail}
$$

U-Net addresses this with:

$$
\boxed{
\text{Encoder}
\rightarrow
\text{Semantic Representation}
}
$$

$$
\boxed{
\text{Bottleneck}
\rightarrow
\text{High-Level Context}
}
$$

$$
\boxed{
\text{Decoder}
\rightarrow
\text{Recover Spatial Resolution}
}
$$

and:

$$
\boxed{
\text{Skip Connections}
\rightarrow
\text{Restore Fine Spatial Information}
}
$$

The complete flow is:

$$
\boxed{
\text{Image}
\rightarrow
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder + Skip Connections}
\rightarrow
\text{Pixel-wise Predictions}
}
$$

The most important insight is:

> **U-Net is not simply a CNN followed by upsampling. Its core idea is to combine deep semantic representations with high-resolution encoder features through skip connections, allowing the decoder to reconstruct precise pixel-level predictions.**

# Transpose Convolutions

**Transpose Convolution** is a learnable operation commonly used in decoder architectures to increase the spatial resolution of feature maps.

Its main purpose is:

$$
\boxed{
\text{Low-Resolution Feature Map}
\rightarrow
\text{Higher-Resolution Feature Map}
}
$$

For example:

$$
16\times16
\rightarrow
32\times32
$$

It is important to note:

$$
\boxed{
\text{Transpose Convolution}
\neq
\text{Inverse Convolution}
}
$$

The transpose refers to the transpose of the linear operator associated with convolution, not the mathematical inverse.

---

## 1. Why Do We Need Transpose Convolution?

In a CNN encoder, spatial resolution is often reduced:

$$
H\times W
\rightarrow
\frac H2\times\frac W2
\rightarrow
\frac H4\times\frac W4
$$

This allows the network to learn more abstract features and obtain larger receptive fields.

However, tasks such as semantic segmentation require a high-resolution output:

$$
\text{Low Resolution}
\rightarrow
\text{High Resolution}
$$

For example:

$$
16\times16
\rightarrow
32\times32
\rightarrow
64\times64
\rightarrow
128\times128
$$

Transpose convolution provides a **learnable way** to perform this spatial expansion.

---

## 2. Core Intuition

A normal convolution can be viewed as:

> A kernel looks at a region of the input and aggregates information into an output value.

Conceptually:

$$
\text{Many Input Values}
\rightarrow
\text{One Output Value}
$$

Transpose convolution can be understood from the opposite perspective:

> **Each input value creates a kernel-shaped contribution on the output.**

Conceptually:

$$
\text{One Input Value}
\rightarrow
\text{Kernel-Shaped Output Contribution}
$$

The contributions from all input values are then combined:

$$
\boxed{
\text{Input Contributions}
\rightarrow
\text{Overlap}
\rightarrow
\text{Sum}
\rightarrow
\text{Output}
}
$$

This is one of the most useful mental models for understanding transpose convolution.

---

## 3. Linear Algebra Interpretation

A convolution can be represented as a matrix operation:

$$
y=Kx
$$

The corresponding transpose operation is:

$$
x'=K^Ty
$$

This explains the name:

$$
\boxed{
\text{Transpose Convolution}
}
$$

However:

$$
K^T\neq K^{-1}
$$

Therefore transpose convolution does **not** reconstruct the original input exactly.

It is better understood as:

$$
\text{Transpose Convolution}
=
\text{Learnable Spatial Transformation}
$$

that can increase spatial resolution.

---

## 4. How Does Transpose Convolution Increase Spatial Resolution?

Consider an input of:

$$
2\times2
$$

and a transpose convolution with:

$$
k=3
$$

and:

$$
s=2
$$

Each input value contributes a $3\times3$ region to the output.

Because the stride is 2, neighboring contributions are shifted by two positions.

Since:

$$
k>s
$$

the contributions overlap.

The overlapping values are **summed**.

This overlap is fundamental to how transpose convolution constructs the output.

---

## 5. Output Size

For one spatial dimension, the output size is commonly expressed as:

$$
n_{\text{out}}
=
(n_{\text{in}}-1)s
-
2p
+
k
+
o_p
$$

where:

- $n_{\text{in}}$: input size;
- $k$: kernel size;
- $s$: stride;
- $p$: padding;
- $o_p$: output padding.

For example:

$$
n_{\text{in}}=16
$$

$$
k=4
$$

$$
s=2
$$

$$
p=1
$$

$$
o_p=0
$$

gives:

$$
n_{\text{out}}
=
(16-1)2-2+4
=
32
$$

Therefore:

$$
16\times16
\rightarrow
32\times32
$$

---

## 6. Role of Stride

Stride controls how far apart neighboring input contributions are placed on the output.

For example:

$$
s=2
$$

means neighboring input positions are mapped two output positions apart.

A larger stride generally produces greater spatial expansion.

Therefore:

$$
\boxed{
\text{Stride}
\rightarrow
\text{Controls Spatial Expansion}
}
$$

---

## 7. Role of Kernel Size

The kernel determines the spatial region contributed by each input activation.

For example:

$$
k=3
$$

means each input value produces a $3\times3$ contribution.

When:

$$
k>s
$$

neighboring contributions overlap.

When:

$$
k=s
$$

the contributions can be placed without overlap in the simplest configuration.

This difference becomes particularly important when analyzing checkerboard artifacts.

---

## 8. Padding and Output Padding

### Padding

Padding affects how the kernel interacts with boundaries and therefore affects the output spatial size.

### Output Padding

Output padding provides a small adjustment to the resulting spatial dimension.

It is not simply extra zero padding around the output.

For example:

$$
o_p=0
$$

and:

$$
o_p=1
$$

can produce output sizes that differ by one position.

---

## 9. Transpose Convolution in U-Net

In U-Net, the encoder reduces spatial resolution:

$$
256\times256
\rightarrow
128\times128
\rightarrow
64\times64
\rightarrow
32\times32
$$

while the decoder needs to reconstruct it:

$$
32\times32
\rightarrow
64\times64
\rightarrow
128\times128
\rightarrow
256\times256
$$

A transpose convolution can perform one of these upsampling stages:

$$
32\times32
\xrightarrow{\text{Transpose Conv}}
64\times64
$$

Then the decoder can combine the result with the corresponding encoder feature through a skip connection.

Conceptually:

$$
\boxed{
\text{Transpose Conv}
\rightarrow
\text{Increase Resolution}
}
$$

while:

$$
\boxed{
\text{Skip Connection}
\rightarrow
\text{Restore Spatial Information}
}
$$

These two operations have different roles.

---

## 10. Learnable vs. Fixed Upsampling

There are several ways to increase spatial resolution.

### Nearest Neighbor

$$
\text{Upsample}
\rightarrow
\text{Copy Values}
$$

No learnable parameters are introduced by the interpolation itself.

### Bilinear Interpolation

$$
\text{Upsample}
\rightarrow
\text{Interpolation}
$$

Again, the interpolation itself is fixed.

### Transpose Convolution

$$
\text{Input}
\rightarrow
\text{Learned Kernel}
\rightarrow
\text{Higher Resolution Feature}
$$

The kernel contains trainable parameters:

$$
W
$$

which are learned through gradient descent.

Therefore:

$$
\boxed{
\text{Transpose Conv}
=
\text{Learnable Upsampling}
}
$$

---

## 11. Checkerboard Artifacts

Transpose convolution can produce **checkerboard artifacts** when the contributions to different output pixels are not distributed uniformly.

This can happen when kernel size and stride create uneven overlap.

Conceptually:

```text
2 3 2 3
3 4 3 4
2 3 2 3
3 4 3 4
```

Different output positions receive different numbers of contributions.

This can lead to:

$$
\text{Uneven Coverage}
\rightarrow
\text{Uneven Activations}
\rightarrow
\text{Checkerboard Artifacts}
$$

This is one reason why some architectures use:

$$
\text{Upsampling}
+
\text{Normal Convolution}
$$

instead.

For example:

$$
16\times16
\xrightarrow{\text{Bilinear}}
32\times32
\xrightarrow{\text{Conv}}
32\times32
$$

instead of:

$$
16\times16
\xrightarrow{\text{Transpose Conv}}
32\times32
$$

Neither approach is universally better; the appropriate choice depends on the architecture and task.

---

## 12. Backward Propagation

Transpose convolution is a normal differentiable learnable layer.

Suppose:

$$
Y=f(X;W)
$$

Then backpropagation computes:

$$
\frac{\partial L}{\partial X}
$$

and:

$$
\frac{\partial L}{\partial W}
$$

The parameters are updated by:

$$
W^{new}
=
W^{old}
-
\eta
\frac{\partial L}{\partial W}
$$

Therefore transpose convolution participates normally in the computational graph.

Forward:

$$
X
\rightarrow
\text{Transpose Conv}
\rightarrow
Y
$$

Backward:

$$
L
\rightarrow
\frac{\partial L}{\partial Y}
\rightarrow
\frac{\partial L}{\partial X},
\frac{\partial L}{\partial W}
$$

---

## 13. Detailed Calculation Example

Consider:

$$
X=
\begin{bmatrix}
1&2\\
3&4
\end{bmatrix}
$$

with a $3\times3$ kernel:

$$
K=
\begin{bmatrix}
1&2&1\\
0&1&0\\
1&2&1
\end{bmatrix}
$$

and:

$$
s=2
$$

$$
p=0
$$

The input has size:

$$
2\times2
$$

The output size is:

$$
n_{\text{out}}
=
(2-1)2+3
=
5
$$

so:

$$
\boxed{
2\times2
\rightarrow
5\times5
}
$$

### Contribution of input value 1

The value $1$ creates:

$$
1K
=
\begin{bmatrix}
1&2&1\\
0&1&0\\
1&2&1
\end{bmatrix}
$$

and is placed starting at the top-left corner.

### Contribution of input value 2

The value $2$ creates:

$$
2K
=
\begin{bmatrix}
2&4&2\\
0&2&0\\
2&4&2
\end{bmatrix}
$$

Because:

$$
s=2
$$

this contribution starts two columns to the right.

### Contribution of input value 3

The value $3$ creates:

$$
3K
=
\begin{bmatrix}
3&6&3\\
0&3&0\\
3&6&3
\end{bmatrix}
$$

and starts two rows lower.

### Contribution of input value 4

The value $4$ creates:

$$
4K
=
\begin{bmatrix}
4&8&4\\
0&4&0\\
4&8&4
\end{bmatrix}
$$

and starts two rows lower and two columns to the right.

Combining all four contributions gives:

$$
Y=
\begin{bmatrix}
1&2&3&4&2\\
0&1&0&0&2\\
1&2&7&8&5\\
3&6&3&8&4\\
3&6&3&6&4
\end{bmatrix}
$$

The important part is not the final numerical matrix itself, but how it was created:

$$
\boxed{
\text{Each Input Value}
\rightarrow
\text{Scaled Kernel}
\rightarrow
\text{Place According to Stride}
\rightarrow
\text{Overlap}
\rightarrow
\text{Sum}
}
$$

Because:

$$
k=3>s=2
$$

the kernel contributions overlap.

For example, the center value $7$ comes from overlapping contributions:

$$
1\times1
+
2\times1
+
3\times1
+
4\times1
=
10
$$

If a position is covered by multiple contributions, those values are added together.

This overlap-and-sum mechanism is the most important computational intuition behind transpose convolution.

---

## 14. Transpose Convolution vs. Normal Convolution

A useful conceptual comparison is:

### Normal Convolution

$$
\text{Input Region}
\rightarrow
\text{Kernel}
\rightarrow
\text{One Output Value}
$$

The operation aggregates information.

### Transpose Convolution

$$
\text{One Input Value}
\rightarrow
\text{Kernel-Shaped Contribution}
$$

then:

$$
\text{Multiple Contributions}
\rightarrow
\text{Overlap + Sum}
$$

The important distinction is:

$$
\boxed{
\text{Convolution}
\rightarrow
\text{Aggregate Local Input}
}
$$

while:

$$
\boxed{
\text{Transpose Convolution}
\rightarrow
\text{Spread Contributions + Aggregate Overlaps}
}
$$

---

## 15. Core Mental Model

The most important idea is:

$$
\boxed{
\text{Transpose Convolution}
=
\text{Learnable Spatial Expansion}
}
$$

Its computational intuition is:

$$
\boxed{
\text{Each Input Activation}
\rightarrow
\text{Kernel-Shaped Contribution}
}
$$

then:

$$
\boxed{
\text{Overlapping Contributions}
\rightarrow
\text{Summation}
}
$$

It is **not**:

$$
\boxed{
\text{Convolution}^{-1}
}
$$

and:

$$
\boxed{
K^T\neq K^{-1}
}
$$

In U-Net:

$$
\text{Encoder}
\rightarrow
\text{Low-Resolution Semantic Features}
$$

then:

$$
\text{Transpose Convolution}
\rightarrow
\text{Higher-Resolution Features}
$$

and:

$$
\text{Skip Connection}
\rightarrow
\text{Fine Spatial Information}
$$

Thus a decoder block can be understood as:

$$
\boxed{
\text{Low-Resolution Feature}
\rightarrow
\text{Transpose Conv}
\rightarrow
\text{Skip Connection}
\rightarrow
\text{Feature Refinement}
}
$$

The key takeaway is:

> **Transpose convolution increases spatial resolution through a learnable kernel. Each input activation produces a kernel-shaped contribution on the output, and overlapping contributions are summed together.**

# U-Net Architecture Intuition

U-Net is an architecture designed for **semantic segmentation**, where the model predicts a semantic class for every pixel in an image.

The central challenge is a fundamental trade-off in CNNs:

$$
\boxed{
\text{Deep Features}
\rightarrow
\text{Strong Semantic Understanding}
}
$$

but:

$$
\boxed{
\text{Downsampling}
\rightarrow
\text{Loss of Spatial Detail}
}
$$

U-Net addresses this trade-off with:

$$
\boxed{
\text{Encoder}
+
\text{Decoder}
+
\text{Skip Connections}
}
$$

The core intuition is:

> **The encoder learns what is in the image, the decoder reconstructs where things are, and skip connections provide the decoder with the fine spatial information that was lost during downsampling.**

---

## 1. Encoder: Understanding "What"

The encoder works like a typical CNN.

It repeatedly extracts features and reduces spatial resolution:

$$
256\times256
\rightarrow
128\times128
\rightarrow
64\times64
\rightarrow
32\times32
$$

As the network goes deeper:

$$
\text{Spatial Resolution}\downarrow
$$

while:

$$
\text{Semantic Abstraction}\uparrow
$$

The model gradually learns:

$$
\text{Edges}
\rightarrow
\text{Textures}
\rightarrow
\text{Parts}
\rightarrow
\text{Objects}
$$

Therefore, a simple mental model is:

$$
\boxed{
\text{Encoder}
\rightarrow
\text{What is in the image?}
}
$$

---

## 2. The Problem with Downsampling

Downsampling is useful because it allows the network to learn more abstract features and obtain larger receptive fields.

However, it also removes fine spatial information.

For example:

$$
256\times256
\rightarrow
16\times16
$$

At the $16\times16$ representation, the network may understand:

> "There is a person in this region."

but it may no longer preserve enough detail to determine the person's exact boundaries at the pixel level.

Therefore:

$$
\boxed{
\text{Semantic Understanding}
\uparrow
}
$$

but:

$$
\boxed{
\text{Spatial Precision}
\downarrow
}
$$

This is the central problem that U-Net is designed to solve.

---

## 3. Decoder: Recovering "Where"

The decoder performs the opposite spatial process:

$$
16\times16
\rightarrow
32\times32
\rightarrow
64\times64
\rightarrow
128\times128
\rightarrow
256\times256
$$

Its goal is to transform the low-resolution semantic representation into a high-resolution segmentation prediction.

A useful mental model is:

$$
\boxed{
\text{Decoder}
\rightarrow
\text{Where exactly is it?}
}
$$

However, simply upsampling the bottleneck is not enough.

Upsampling can increase spatial resolution, but it cannot perfectly reconstruct details that were already lost during downsampling.

---

## 4. Skip Connections: The Core Idea of U-Net

U-Net solves this by passing encoder feature maps directly to the decoder through **skip connections**.

Conceptually:

```mermaid
flowchart LR
    A["Encoder"] --> B["Bottleneck"]
    B --> C["Decoder"]
    A -. "Skip Connections" .-> C
```

In practice, there are multiple skip connections at corresponding spatial resolutions.

The encoder therefore provides:

$$
\text{Fine Spatial Information}
$$

while the deeper representation provides:

$$
\text{High-Level Semantic Information}
$$

The decoder combines both:

$$
\boxed{
\text{Semantic Information}
+
\text{Spatial Information}
}
$$

---

## 5. Why Do Skip Connections Matter?

Suppose the encoder produces a feature map at:

$$
64\times64
$$

This feature still preserves relatively detailed spatial information.

The encoder then continues:

$$
64\times64
\rightarrow
32\times32
\rightarrow
16\times16
$$

Later, the decoder reconstructs:

$$
16\times16
\rightarrow
32\times32
\rightarrow
64\times64
$$

When the decoder reaches $64\times64$, it can receive the original high-resolution encoder feature through the skip connection.

Therefore, instead of reconstructing everything from the compressed $16\times16$ representation, the decoder gets access to the earlier spatial details directly.

The intuition is:

$$
\boxed{
\text{Deep Features}
\rightarrow
\text{Strong Semantics}
}
$$

$$
\boxed{
\text{Skip Features}
\rightarrow
\text{Fine Spatial Details}
}
$$

---

## 6. Semantic Information + Spatial Information

This is the most important intuition of U-Net.

Deep features are good at answering:

> "What is this?"

Shallow, high-resolution features are better at preserving:

> "Where are its boundaries?"

U-Net combines both:

$$
\boxed{
\text{Deep Semantic Features}
+
\text{High-Resolution Spatial Features}
\rightarrow
\text{Precise Segmentation}
}
$$

This is the fundamental reason the architecture works well for pixel-level prediction.

---

## 7. Multi-Scale Information

U-Net does not rely on only one spatial scale.

The encoder creates representations at multiple resolutions:

$$
256\times256
$$

$$
128\times128
$$

$$
64\times64
$$

$$
32\times32
$$

$$
16\times16
$$

Different scales provide different kinds of information.

### High-Resolution Features

Better preserve:

- edges;
- boundaries;
- small structures;
- local spatial details.

### Low-Resolution Features

Provide:

- larger receptive fields;
- broader context;
- stronger semantic understanding.

Therefore:

$$
\boxed{
\text{U-Net}
\rightarrow
\text{Multi-Scale Feature Fusion}
}
$$

---

## 8. Why Does U-Net Have a U Shape?

The U shape reflects the spatial transformation performed by the network.

```text
High Resolution
      │
      ▼
   Encoder
      │
      ▼
Low Resolution
      │
      ▼
  Bottleneck
      │
      ▼
High Resolution
      │
      ▼
   Decoder
```

The encoder compresses the image.

The bottleneck captures a deep semantic representation.

The decoder reconstructs the spatial resolution.

The skip connections connect corresponding levels of the encoder and decoder.

Thus the U shape reflects:

$$
\boxed{
\text{Compress}
\rightarrow
\text{Understand}
\rightarrow
\text{Reconstruct}
}
$$

---

## 9. U-Net vs. a Simple Encoder-Decoder

A simple encoder-decoder contains:

$$
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder}
$$

The decoder must rely primarily on the compressed representation.

U-Net adds:

$$
\boxed{
\text{Encoder Features}
\rightarrow
\text{Skip Connections}
\rightarrow
\text{Decoder}
}
$$

Therefore:

$$
\boxed{
\text{Simple Encoder-Decoder}
=
\text{Semantic Reconstruction}
}
$$

while:

$$
\boxed{
\text{U-Net}
=
\text{Semantic Reconstruction}
+
\text{Spatial Detail Recovery}
}
$$

---

## 10. Overall Architecture Intuition

The entire architecture can be summarized as:

```mermaid
flowchart LR
    A["Input Image"] --> B["Encoder"]
    B --> C["Bottleneck"]
    C --> D["Decoder"]
    D --> E["Segmentation Output"]

    B -. "Skip Connections" .-> D
```

The encoder progressively trades spatial resolution for semantic abstraction:

$$
\boxed{
\text{Resolution}\downarrow
\quad
\text{Semantic Abstraction}\uparrow
}
$$

The decoder reverses this:

$$
\boxed{
\text{Resolution}\uparrow
}
$$

The skip connections provide the decoder with information that would otherwise be difficult to reconstruct:

$$
\boxed{
\text{Fine Spatial Information}
\rightarrow
\text{Decoder}
}
$$

Therefore:

$$
\boxed{
\text{Deep Semantics}
+
\text{Fine Spatial Details}
\rightarrow
\text{Pixel-Level Prediction}
}
$$

The key mental model is:

> **The encoder learns what is present, the decoder reconstructs the spatial structure, and skip connections preserve the detailed spatial information needed for precise segmentation boundaries.**

# U-Net Architecture

U-Net is an **encoder-decoder architecture** designed for semantic segmentation, where the model predicts a semantic class for every pixel.

The core architecture is:

$$
\boxed{
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder}
}
$$

with **skip connections** connecting corresponding encoder and decoder levels.

The key idea is:

> **The encoder extracts increasingly semantic features, the decoder restores spatial resolution, and skip connections provide the decoder with high-resolution spatial information that would otherwise be lost during downsampling.**

---

## 1. Overall Architecture

A typical U-Net can be represented as:

```mermaid
flowchart LR
    A["Input Image"] --> B["Encoder Block 1"]
    B --> C["Encoder Block 2"]
    C --> D["Encoder Block 3"]
    D --> E["Bottleneck"]

    E --> F["Decoder Block 3"]
    F --> G["Decoder Block 2"]
    G --> H["Decoder Block 1"]
    H --> I["Segmentation Output"]

    B -. "Skip Connection" .-> H
    C -. "Skip Connection" .-> G
    D -. "Skip Connection" .-> F
```

U-Net contains four main components:

- **Encoder**: extracts features and reduces spatial resolution.
- **Bottleneck**: contains the deepest semantic representation.
- **Decoder**: progressively restores spatial resolution.
- **Skip Connections**: transfer encoder features to the corresponding decoder stages.

---

## 2. Encoder

The encoder behaves similarly to a conventional CNN.

It repeatedly performs feature extraction and downsampling:

$$
256\times256
\rightarrow
128\times128
\rightarrow
64\times64
\rightarrow
32\times32
$$

while the number of channels generally increases:

$$
C
\rightarrow
2C
\rightarrow
4C
\rightarrow
8C
$$

This creates a trade-off:

$$
\boxed{
\text{Spatial Resolution}\downarrow
\quad
\text{Semantic Abstraction}\uparrow
}
$$

Early layers tend to preserve fine spatial information, while deeper layers learn increasingly abstract semantic representations.

Conceptually:

$$
\text{Edges}
\rightarrow
\text{Textures}
\rightarrow
\text{Parts}
\rightarrow
\text{Semantic Concepts}
$$

---

## 3. Bottleneck

After the final encoder stage, the feature map reaches its lowest spatial resolution.

For example:

$$
16\times16\times512
$$

The bottleneck contains highly abstract semantic information and has a large receptive field.

It allows the network to understand the broader context of the image.

However, because its spatial resolution is low, it does not preserve all of the fine-grained details required for accurate pixel-level segmentation.

Therefore the decoder needs additional information from earlier encoder layers.

---

## 4. Decoder

The decoder gradually increases spatial resolution:

$$
16\times16
\rightarrow
32\times32
\rightarrow
64\times64
\rightarrow
128\times128
\rightarrow
256\times256
$$

Its purpose is to transform the compressed semantic representation into a high-resolution representation suitable for pixel-wise prediction.

A decoder stage generally performs:

$$
\boxed{
\text{Upsampling}
\rightarrow
\text{Feature Fusion}
\rightarrow
\text{Convolution}
}
$$

The upsampling can be implemented using a transpose convolution or another upsampling mechanism.

---

## 5. Skip Connections

Skip connections are the defining feature of U-Net.

At each decoder stage, the upsampled decoder feature is combined with the corresponding encoder feature from the same spatial resolution.

For example:

$$
E_3
\rightarrow
\text{Skip Connection}
\rightarrow
D_3
$$

where:

- $E_3$: encoder feature;
- $D_3$: corresponding decoder feature.

Conceptually:

```mermaid
flowchart LR
    A["Encoder Feature<br/>32 × 32"] -. "Skip" .-> D["Feature Fusion<br/>32 × 32"]
    B["Decoder Feature<br/>32 × 32"] --> D
```

The encoder feature contributes relatively fine spatial information, while the decoder feature contains information reconstructed from deeper semantic representations.

Therefore:

$$
\boxed{
\text{Encoder Spatial Information}
+
\text{Decoder Semantic Information}
}
$$

---

## 6. Why Do We Connect Corresponding Resolutions?

Suppose the decoder has been upsampled to:

$$
32\times32
$$

and the encoder previously produced:

$$
32\times32
$$

The two feature maps therefore have matching spatial dimensions and can be combined directly.

This allows the decoder to access the high-resolution spatial information that existed before further downsampling.

Thus:

$$
\boxed{
\text{Same Spatial Resolution}
\rightarrow
\text{Meaningful Feature Fusion}
}
$$

---

## 7. Concatenation

Original U-Net uses **concatenation** for the skip connection.

Suppose:

$$
E\in\mathbb R^{32\times32\times256}
$$

and:

$$
D\in\mathbb R^{32\times32\times128}
$$

The feature maps are concatenated along the channel dimension:

$$
F
=
\operatorname{Concat}(D,E)
$$

giving:

$$
F\in\mathbb R^{32\times32\times384}
$$

because:

$$
128+256=384
$$

The important point is:

> **Concatenation does not itself learn how to combine the information. It simply places the two sets of feature channels together.**

After concatenation, the resulting tensor contains both:

$$
\text{Decoder Features}
$$

and:

$$
\text{Encoder Features}
$$

---

## 8. Convolution After Concatenation

This is where actual **feature fusion** is learned.

After concatenation:

$$
F=
\operatorname{Concat}(D,E)
$$

we pass the result through convolutional layers:

$$
F
\xrightarrow{\text{Conv}}
F'
$$

For example:

$$
32\times32\times384
\xrightarrow{\text{Conv}}
32\times32\times256
$$

The convolution has learnable parameters:

$$
W
$$

and learns which combinations of encoder and decoder channels are useful.

Conceptually:

$$
\boxed{
\text{Concatenation}
=
\text{Put Information Together}
}
$$

while:

$$
\boxed{
\text{Convolution}
=
\text{Learn How to Combine the Information}
}
$$

For example, the network may learn a feature that combines:

$$
\text{Semantic Evidence}
+
\text{Boundary Information}
\rightarrow
\text{Object Boundary Feature}
$$

Therefore, the convolution after concatenation is essential for transforming the raw combination of features into a useful fused representation.

---

## 9. A Complete Decoder Block

A decoder stage can therefore be represented mathematically as:

$$
D_l^{up}
=
\operatorname{Upsample}(D_{l+1})
$$

then:

$$
F_l
=
\operatorname{Concat}(D_l^{up},E_l)
$$

and finally:

$$
D_l
=
f_l(F_l)
$$

where:

- $D_{l+1}$: decoder feature from the deeper level;
- $D_l^{up}$: upsampled decoder feature;
- $E_l$: corresponding encoder feature;
- $F_l$: concatenated feature;
- $f_l$: convolutional block.

Therefore:

$$
\boxed{
D_l
=
f_l
\left(
\operatorname{Concat}
\left(
\operatorname{Upsample}(D_{l+1}),
E_l
\right)
\right)
}
$$

This operation is repeated at each decoder level.

---

## 10. Complete Forward Architecture

Let the encoder produce:

$$
E_1,E_2,E_3,E_4
$$

and let:

$$
Z
$$

be the bottleneck.

The decoder progressively reconstructs the resolution:

$$
Z
\rightarrow
\operatorname{Upsample}
\rightarrow
\operatorname{Concat}(E_4)
\rightarrow
D_4
$$

then:

$$
D_4
\rightarrow
\operatorname{Upsample}
\rightarrow
\operatorname{Concat}(E_3)
\rightarrow
D_3
$$

then:

$$
D_3
\rightarrow
\operatorname{Upsample}
\rightarrow
\operatorname{Concat}(E_2)
\rightarrow
D_2
$$

and:

$$
D_2
\rightarrow
\operatorname{Upsample}
\rightarrow
\operatorname{Concat}(E_1)
\rightarrow
D_1
$$

Finally:

$$
D_1
\rightarrow
1\times1\text{ Conv}
\rightarrow
\hat Y
$$

where:

$$
\hat Y\in\mathbb R^{H\times W\times C}
$$

---

## 11. Final Segmentation Head

Suppose the final decoder representation is:

$$
D_{\text{final}}
\in
\mathbb R^{H\times W\times C'}
$$

A $1\times1$ convolution maps:

$$
C'
\rightarrow
C
$$

to produce:

$$
\hat Y
\in
\mathbb R^{H\times W\times C}
$$

At each pixel $(i,j)$:

$$
\hat y_{ij}
=
[
p_{ij}^{(1)},
p_{ij}^{(2)},
\ldots,
p_{ij}^{(C)}
]
$$

The predicted class is:

$$
\hat c_{ij}
=
\arg\max_c p_{ij}^{(c)}
$$

Thus:

$$
\boxed{
\text{One Pixel}
\rightarrow
\text{One Class Prediction}
}
$$

---

## 12. Backward Propagation

The segmentation loss is computed from the pixel-wise predictions.

The forward direction is:

$$
X
\rightarrow
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder}
\rightarrow
\hat Y
\rightarrow
L
$$

During backpropagation:

$$
L
\rightarrow
\text{Decoder}
\rightarrow
\text{Encoder}
$$

Because of skip connections, gradients can propagate through additional paths.

Conceptually:

```mermaid
flowchart BT
    A["Segmentation Loss"] --> B["Decoder"]
    B --> C["Bottleneck"]
    B --> D["Skip Connection"]
    D --> E["Earlier Encoder"]
    C --> F["Deeper Encoder"]
```

Therefore:

$$
\boxed{
\text{Skip Connection}
\rightarrow
\text{Additional Gradient Path}
}
$$

However, the primary architectural purpose of the skip connection is still to preserve and reuse high-resolution spatial information.

---

## 13. Semantic and Spatial Information Across the Network

Different levels of U-Net specialize in different types of information.

| Level | Spatial Resolution | Main Information |
| --- | ---: | --- |
| Early Encoder | High | Edges, local structures, fine details |
| Middle Encoder | Medium | Shapes and object parts |
| Bottleneck | Low | Semantic context and abstraction |
| Decoder | Increasing | Semantic reconstruction + spatial refinement |
| Final Output | Full | Pixel-wise class predictions |

The architecture therefore combines:

$$
\boxed{
\text{High-Level Semantics}
}
$$

with:

$$
\boxed{
\text{Fine-Grained Spatial Information}
}
$$

---

## 14. U-Net vs. Simple Encoder-Decoder

A simple encoder-decoder follows:

$$
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder}
$$

The decoder must rely mainly on the compressed representation.

U-Net adds:

$$
\boxed{
\text{Encoder Feature}
\rightarrow
\text{Skip Connection}
\rightarrow
\text{Decoder}
}
$$

Therefore:

$$
\boxed{
\text{Simple Encoder-Decoder}
=
\text{Semantic Reconstruction}
}
$$

while:

$$
\boxed{
\text{U-Net}
=
\text{Semantic Reconstruction}
+
\text{Spatial Detail Recovery}
}
$$

---

## 15. Complete U-Net Architecture

```mermaid
flowchart TD
    A["Input Image"] --> B["Encoder Block 1"]
    B --> C["Encoder Block 2"]
    C --> D["Encoder Block 3"]
    D --> E["Bottleneck"]

    E --> F["Upsample"]
    D --> G["Concat"]
    F --> G
    G --> H["Conv Block"]

    H --> I["Upsample"]
    C --> J["Concat"]
    I --> J
    J --> K["Conv Block"]

    K --> L["Upsample"]
    B --> M["Concat"]
    L --> M
    M --> N["Conv Block"]

    N --> O["1 × 1 Conv"]
    O --> P["Segmentation Map"]
```

The repeated decoder pattern is:

$$
\boxed{
\text{Upsample}
\rightarrow
\text{Concatenate Skip Feature}
\rightarrow
\text{Convolution}
\rightarrow
\text{Refined Feature}
}
$$

The convolution after concatenation is particularly important because it learns how to fuse the spatial information from the encoder with the semantic information carried by the decoder.

---

## 16. Core Mental Model

The architecture can be summarized through three roles.

### Encoder

$$
\boxed{
\text{Extract and Compress Features}
}
$$

$$
\text{Resolution}\downarrow
\qquad
\text{Semantic Abstraction}\uparrow
$$

### Decoder

$$
\boxed{
\text{Recover Spatial Resolution}
}
$$

### Skip Connection + Convolution

$$
\boxed{
\text{Encoder Spatial Information}
+
\text{Decoder Semantic Information}
\rightarrow
\text{Fused Representation}
}
$$

The complete process is:

$$
\boxed{
\text{Input}
\rightarrow
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Upsample}
\rightarrow
\text{Concat}
\rightarrow
\text{Conv Fusion}
\rightarrow
\text{Pixel-wise Output}
}
$$

The most important architectural insight is:

> **Concatenation brings the encoder and decoder features together, but the subsequent convolution learns how to combine those features into a useful representation for segmentation.**

Thus U-Net is fundamentally:

$$
\boxed{
\text{Deep Semantic Features}
+
\text{High-Resolution Spatial Features}
\rightarrow
\text{Precise Pixel-level Prediction}
}
$$